# Algebraic population geometry of SOFI explanations on Derm7pt+

This notebook trains a hard concept bottleneck model on Derm7pt+ with five seeds (Part 1). Part 2 studies the SOFI explanations of the correct decisions as two populations, melanoma and non-melanoma, in three experiments. The first computes every geometric and algebraic object of the paper for one trained model. The second repeats the analysis for every seed, since each seed gives another model with other explanations. The third compares one global ranking per population with the local SOFI ranking of each image.

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
# ============================================================
# GLOBAL SETTINGS SHARED BY BOTH PARTS
# ============================================================
# One model is trained per seed in Part 1. Part 2 studies the model with the largest test F1 in depth,
# then compares the explanations of all seeds. Every path of the notebook derives from the variables below.

SEEDS = [42, 64, 128, 256, 512]
BACKBONE_NAME = "efficientnet_b4"        # one entry of BACKBONE_LIST in Part 1
FILTER_STRATEGY = "asymmetric"           # "none" | "symmetric" | "asymmetric"
RUN_TRAINING = True                      # False reuses the checkpoints that an earlier run of Part 1 saved
SKIP_TRAINED_SEEDS = True                # True trains only the seeds without a saved checkpoint

PROJECT_DIR = "/content/drive/MyDrive/Notebooks/Derm7pt"
META_PATH = f"{PROJECT_DIR}/meta"                    # meta.csv with the split index files
IMAGES_PATH = f"{PROJECT_DIR}/images_cropped"        # dermoscopic images
MODELS_PATH = f"{PROJECT_DIR}/models"                # Part 1 checkpoints, logs, predictions, manifest
APG_OUTPUT_PATH = f"{PROJECT_DIR}/APG_outputs"       # Part 2 console log, csv tables, reports
LOCAL_IMAGES_PATH = "/content/derm7pt_images_local"  # scratch copy of the images on the local disk

# Part 1. Model training

The seeds, backbone, filtering strategy and paths come from the global settings cell above. With `SKIP_TRAINED_SEEDS = True`, a seed whose checkpoint already exists is reloaded instead of retrained, so adding seeds only trains the new ones. Set `RUN_TRAINING = False` to skip the whole cell.

In [3]:
# ============================================================
# CBM IMPLEMENTATION
# ============================================================

import os
import random
import warnings
import builtins

import numpy as np
import pandas as pd
from PIL import Image

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score,
    roc_auc_score, confusion_matrix, balanced_accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight

original_print = builtins.print

# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

# ============================================================
# CONFIG
# ============================================================

class Config:
    META_DIR   = META_PATH
    IMAGES_DIR = IMAGES_PATH
    OUTPUT_DIR = MODELS_PATH

    TUNABLE_LAYERS    = 75
    FILTER_STRATEGY   = FILTER_STRATEGY  # set in the global settings cell
    CONCEPT_WEIGHTING = True
    BATCH_SIZE        = 16
    EPOCHS            = 20
    IMG_SIZE          = 224
    LEARNING_RATE     = 5e-4
    ETA_MIN           = 1e-5
    LAMBDA1           = 1.0
    LAMBDA2           = 1.0

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    CONCEPT_COLUMNS = [
        "pigment_network", "streaks", "pigmentation",
        "regression_structures", "dots_and_globules",
        "blue_whitish_veil", "vascular_structures"
    ]
    TARGET_COLUMN = "diagnosis"

CFG = Config()
os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)

# ============================================================
# INCONSISTENCY FILTERING
# ============================================================

def _add_concept_key(df):
    """Assign a concept-profile key to every row."""
    df = df.copy()
    for c in CFG.CONCEPT_COLUMNS:
        df[c] = df[c].astype(str).str.strip().str.lower().fillna("unknown")
    df["_concept_key"] = df[CFG.CONCEPT_COLUMNS].apply(
        lambda row: "|".join(f"{c}={v}" for c, v in row.items()), axis=1
    )
    return df


def get_inconsistent_concept_keys(df):
    """
    Return the set of concept-profile keys that map to more than one
    diagnosis label, i.e. the keys that define BND_C(d).
    """
    df = _add_concept_key(df)
    groups = df.groupby("_concept_key")[CFG.TARGET_COLUMN].agg(set)
    return set(groups[groups.apply(lambda s: len(s) > 1)].index)


def filter_inconsistent_samples(df, inconsistent_keys, strategy):

    if strategy == "none":
        return df

    df = _add_concept_key(df)
    in_boundary = df["_concept_key"].isin(inconsistent_keys)

    if strategy == "symmetric":
        mask = ~in_boundary

    elif strategy == "asymmetric":
        is_melanoma = df[CFG.TARGET_COLUMN] == 1
        mask = (~in_boundary) | (in_boundary & is_melanoma)

    else:
        raise ValueError(
            f"Unknown filter strategy {strategy!r}. "
            "Choose 'none', 'symmetric', or 'asymmetric'."
        )

    return (
        df[mask]
        .drop(columns=["_concept_key"])
        .reset_index(drop=True)
    )

# ============================================================
# CONCEPT ENCODERS
# ============================================================

def build_global_concept_encoders(full_meta_df):
    concept_info, label_encoders = {}, {}
    for c in CFG.CONCEPT_COLUMNS:
        le = LabelEncoder()
        le.fit(full_meta_df[c].astype(str).fillna("unknown").values)
        label_encoders[c] = le
        concept_info[c]   = {"n_classes": len(le.classes_)}
    return concept_info, label_encoders

# ============================================================
# BACKBONE
# ============================================================

def get_backbone(name):
    if name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        features, feature_dim = m.features, 1280
    elif name == "efficientnet_b1":
        m = models.efficientnet_b1(weights=models.EfficientNet_B1_Weights.DEFAULT)
        features, feature_dim = m.features, 1280
    elif name == "efficientnet_b2":
        m = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
        features, feature_dim = m.features, 1408
    elif name == "efficientnet_b3":
        m = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
        features, feature_dim = m.features, 1536
    elif name == "efficientnet_b4":
        m = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.DEFAULT)
        features, feature_dim = m.features, 1792
    elif name == "efficientnet_b5":
        m = models.efficientnet_b5(weights=models.EfficientNet_B5_Weights.DEFAULT)
        features, feature_dim = m.features, 2048
    elif name == "efficientnet_b6":
        m = models.efficientnet_b6(weights=models.EfficientNet_B6_Weights.DEFAULT)
        features, feature_dim = m.features, 2304
    elif name == "efficientnet_b7":
        m = models.efficientnet_b7(weights=models.EfficientNet_B7_Weights.DEFAULT)
        features, feature_dim = m.features, 2560
    elif name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        features, feature_dim = m.features, 1024
    elif name == "densenet161":
        m = models.densenet161(weights=models.DenseNet161_Weights.DEFAULT)
        features, feature_dim = m.features, 2208
    elif name == "densenet169":
        m = models.densenet169(weights=models.DenseNet169_Weights.DEFAULT)
        features, feature_dim = m.features, 1664
    elif name == "densenet201":
        m = models.densenet201(weights=models.DenseNet201_Weights.DEFAULT)
        features, feature_dim = m.features, 1920
    elif name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        features, feature_dim = nn.Sequential(*list(m.children())[:-2]), 512
    elif name == "resnet34":
        m = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        features, feature_dim = nn.Sequential(*list(m.children())[:-2]), 512
    elif name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        features, feature_dim = nn.Sequential(*list(m.children())[:-2]), 2048
    elif name == "resnet101":
        m = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        features, feature_dim = nn.Sequential(*list(m.children())[:-2]), 2048
    elif name == "resnet152":
        m = models.resnet152(weights=models.ResNet152_Weights.DEFAULT)
        features, feature_dim = nn.Sequential(*list(m.children())[:-2]), 2048
    elif name == "wide_resnet50_2":
        m = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.DEFAULT)
        features, feature_dim = nn.Sequential(*list(m.children())[:-2]), 2048
    elif name == "wide_resnet101_2":
        m = models.wide_resnet101_2(weights=models.Wide_ResNet101_2_Weights.DEFAULT)
        features, feature_dim = nn.Sequential(*list(m.children())[:-2]), 2048
    else:
        raise ValueError(f"Unknown backbone: {name!r}")

    total        = len(features)
    freeze_up_to = int(round(total * (100 - CFG.TUNABLE_LAYERS) / 100))
    for i, layer in enumerate(features):
        for p in layer.parameters():
            p.requires_grad = (i >= freeze_up_to)

    original_print(
        f"  Backbone {name}: {total} layers | "
        f"frozen={freeze_up_to} | trainable={total - freeze_up_to} | "
        f"tunable={CFG.TUNABLE_LAYERS}%"
    )
    return features, feature_dim

# ============================================================
# DATASET
# ============================================================

class Derm7ptDataset(Dataset):
    def __init__(self, df, images_dir, concept_info, label_encoders, transform=None):
        self.df             = df.reset_index(drop=True)
        self.images_dir     = images_dir
        self.transform      = transform
        self.concept_info   = concept_info
        self.label_encoders = label_encoders

        self.image_paths = {}
        for root, _, files in os.walk(images_dir):
            for f in files:
                if f.lower().endswith((".jpg", ".jpeg", ".png")):
                    full = os.path.join(root, f)
                    rel  = os.path.relpath(full, images_dir).replace("\\", "/").lower()
                    self.image_paths[rel] = full

        for c in CFG.CONCEPT_COLUMNS:
            self.df[c] = label_encoders[c].transform(
                df[c].astype(str).fillna("unknown").values)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        rel_path = str(row["derm"]).replace("\\", "/").lower()
        if rel_path not in self.image_paths:
            raise FileNotFoundError(f"Image not found: {rel_path}")
        image = Image.open(self.image_paths[rel_path]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        concepts = torch.tensor(
            [int(row[c]) for c in CFG.CONCEPT_COLUMNS], dtype=torch.long
        )
        label = torch.tensor(int(row[CFG.TARGET_COLUMN]), dtype=torch.long)
        return image, concepts, label

# ============================================================
# TRANSFORMS
# ============================================================

def get_train_transforms():
    return transforms.Compose([
        transforms.RandomResizedCrop(CFG.IMG_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=90),
        transforms.RandomApply([
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=15)
        ], p=0.5),
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.RandomChoice([
            transforms.RandomAdjustSharpness(sharpness_factor=2.0),
            transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
            transforms.Lambda(lambda x: x),
        ]),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


def get_val_transforms():
    return transforms.Compose([
        transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

# ============================================================
# MODEL
# ============================================================

class ConceptBottleneckModel(nn.Module):
    def __init__(self, backbone_name, concept_info, num_classes=2):
        super().__init__()
        self.feature_extractor, self.feature_dim = get_backbone(backbone_name)
        self.pool          = nn.AdaptiveAvgPool2d(1)
        self.concept_info  = concept_info
        self.concept_heads = nn.ModuleDict({
            c: nn.Linear(self.feature_dim, concept_info[c]["n_classes"])
            for c in CFG.CONCEPT_COLUMNS
        })
        total_concept_dim = sum(
            concept_info[c]["n_classes"] for c in CFG.CONCEPT_COLUMNS
        )
        self.classifier = nn.Sequential(
            nn.Linear(total_concept_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        feats = self.feature_extractor(x)
        feats = self.pool(feats).view(feats.size(0), -1)
        concept_logits, one_hot_concepts = [], []
        for c in CFG.CONCEPT_COLUMNS:
            logits = self.concept_heads[c](feats)
            concept_logits.append(logits)
            hard = F.one_hot(
                torch.argmax(logits, dim=1),
                num_classes=self.concept_info[c]["n_classes"],
            ).float()
            one_hot_concepts.append(hard.detach())
        logits_y = self.classifier(torch.cat(one_hot_concepts, dim=1))
        return concept_logits, logits_y

# ============================================================
# LOSS
# ============================================================

class CBMLoss(nn.Module):
    def __init__(self, class_weights_label, concept_class_weights,
                 use_concept_weights=True):
        super().__init__()
        self.label_weight        = class_weights_label
        self.concept_weights     = concept_class_weights
        self.use_concept_weights = use_concept_weights

    def forward(self, concept_preds, concept_true, logits_y, labels):
        loss_c = torch.stack([
            F.cross_entropy(
                concept_preds[i],
                concept_true[:, i],
                weight=(self.concept_weights[c] if self.use_concept_weights else None),
            )
            for i, c in enumerate(CFG.CONCEPT_COLUMNS)
        ]).mean()

        loss_y = F.cross_entropy(
            logits_y, labels, weight=self.label_weight
        )

        return CFG.LAMBDA1 * loss_c + CFG.LAMBDA2 * loss_y

# ============================================================
# TRAIN / EVAL
# ============================================================

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for imgs, concepts, labels in loader:
        imgs     = imgs.to(CFG.DEVICE)
        concepts = concepts.to(CFG.DEVICE)
        labels   = labels.to(CFG.DEVICE)
        optimizer.zero_grad()
        concept_preds, logits_y = model(imgs)
        loss = criterion(concept_preds, concepts, logits_y, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    c_preds = [[] for _ in CFG.CONCEPT_COLUMNS]
    c_true  = [[] for _ in CFG.CONCEPT_COLUMNS]
    with torch.no_grad():
        for imgs, concepts, labels in loader:
            imgs     = imgs.to(CFG.DEVICE)
            concepts = concepts.to(CFG.DEVICE)
            labels   = labels.to(CFG.DEVICE)
            concept_logits, logits_y = model(imgs)
            all_preds.extend(logits_y.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            for i, lg in enumerate(concept_logits):
                c_preds[i].extend(lg.argmax(1).cpu().numpy())
                c_true[i].extend(concepts[:, i].cpu().numpy())
    acc   = accuracy_score(all_labels, all_preds)
    f1    = f1_score(all_labels, all_preds, average="macro")
    c_acc = np.mean([
        accuracy_score(c_true[i], c_preds[i])
        for i in range(len(CFG.CONCEPT_COLUMNS))])
    c_f1  = np.mean([
        f1_score(c_true[i], c_preds[i], average="macro", zero_division=0)
        for i in range(len(CFG.CONCEPT_COLUMNS))])
    return acc, f1, c_acc, c_f1


# ============================================================
# PREDICTION COLLECTION AND METRIC HELPERS
# ============================================================

def collect_predictions(model, loader):
    """
    Returns (y_true, y_pred, y_prob, c_true_list, c_pred_list).
    y_prob is the softmax probability for the melanoma class (class 1).
    """
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    c_preds = [[] for _ in CFG.CONCEPT_COLUMNS]
    c_true  = [[] for _ in CFG.CONCEPT_COLUMNS]
    with torch.no_grad():
        for imgs, concepts, labels in loader:
            imgs     = imgs.to(CFG.DEVICE)
            concepts = concepts.to(CFG.DEVICE)
            concept_logits, logits_y = model(imgs)
            probs = torch.softmax(logits_y, dim=1)[:, 1].cpu().numpy()
            preds = logits_y.argmax(1).cpu().numpy()
            y_true.extend(labels.numpy())
            y_pred.extend(preds)
            y_prob.extend(probs)
            for i, lg in enumerate(concept_logits):
                c_preds[i].extend(lg.argmax(1).cpu().numpy())
                c_true[i].extend(concepts[:, i].cpu().numpy())
    return (np.array(y_true), np.array(y_pred), np.array(y_prob),
            c_true, c_preds)


def compute_label_metrics(y_true, y_pred, y_prob):
    """
    Returns a dict with seven label-level metrics:
      label_acc, weighted_acc, label_f1,
      macro_sensitivity, mel_sensitivity,
      macro_specificity, auroc.
    """
    acc          = accuracy_score(y_true, y_pred)
    weighted_acc = balanced_accuracy_score(y_true, y_pred)
    f1           = f1_score(y_true, y_pred, average="macro", zero_division=0)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    sensitivities, specificities = [], []
    for i in range(2):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        sensitivities.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
        specificities.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)

    try:
        auroc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auroc = float("nan")

    return {
        "label_acc":          round(float(acc),                    4),
        "weighted_acc":       round(float(weighted_acc),            4),
        "label_f1":           round(float(f1),                     4),
        "macro_sensitivity":  round(float(np.mean(sensitivities)), 4),
        "mel_sensitivity":    round(float(sensitivities[1]),        4),
        "macro_specificity":  round(float(np.mean(specificities)), 4),
        "auroc":              round(float(auroc),                  4),
    }


def print_split_statistics(meta, inconsistent_keys):
    """
    Print melanoma / non-melanoma counts per split before and after
    filtering, plus cross-split concept-profile overlap.
    Called once before the backbone loop.
    """
    train_idx = pd.read_csv(os.path.join(CFG.META_DIR, "train_indexes.csv"))["indexes"].values
    valid_idx = pd.read_csv(os.path.join(CFG.META_DIR, "valid_indexes.csv"))["indexes"].values
    test_idx  = pd.read_csv(os.path.join(CFG.META_DIR, "test_indexes.csv"))["indexes"].values

    splits_raw = {
        "train": meta.iloc[train_idx].copy(),
        "valid": meta.iloc[valid_idx].copy(),
        "test" : meta.iloc[test_idx].copy(),
    }

    print("=" * 65)
    print("DATASET SPLIT STATISTICS")
    print("=" * 65)

    for strategy in ["symmetric", "asymmetric"]:
        print(f"\n--- Strategy: {strategy} ---")
        print(f"{'Split':<8} {'Total':>7} {'Melanoma':>10} {'Non-Mel':>9} "
              f"{'Boundary':>10} {'Profiles':>10}")
        print("-" * 65)

        def get_profiles(df):
            return set(
                _add_concept_key(
                    filter_inconsistent_samples(df, inconsistent_keys, strategy)
                )["_concept_key"].tolist()
            )

        profile_sets = {}
        for split_name, df_raw in splits_raw.items():
            df_filt    = filter_inconsistent_samples(df_raw, inconsistent_keys, strategy)
            n_total    = len(df_filt)
            n_mel      = int((df_filt[CFG.TARGET_COLUMN] == 1).sum())
            n_nonmel   = int((df_filt[CFG.TARGET_COLUMN] == 0).sum())
            df_keyed   = _add_concept_key(df_raw)
            n_boundary = int(df_keyed["_concept_key"].isin(inconsistent_keys).sum())
            profile_sets[split_name] = get_profiles(df_raw)
            n_profiles = len(profile_sets[split_name])
            print(f"{split_name:<8} {n_total:>7} {n_mel:>10} {n_nonmel:>9} "
                  f"{n_boundary:>10} {n_profiles:>10}")

        tr_p, va_p, te_p = (profile_sets["train"],
                             profile_sets["valid"],
                             profile_sets["test"])
        print(f"\n  Profile overlap ({strategy}):")
        print(f"    train ∩ valid : {len(tr_p & va_p):>4} profiles")
        print(f"    train ∩ test  : {len(tr_p & te_p):>4} profiles")
        print(f"    valid ∩ test  : {len(va_p & te_p):>4} profiles")
        print(f"    test-only     : {len(te_p - tr_p - va_p):>4} profiles "
              f"(unseen at training time)")

    print("\n" + "=" * 65)


print("CBM implementation ready.")
print(f"Device : {CFG.DEVICE}")

CBM implementation ready.
Device : cuda


In [4]:
# ============================================================
# MODEL TRAINING, ONE MODEL PER SEED
# ============================================================

import sys
import copy
import torch.optim as optim
from torch.utils.data import DataLoader

original_print = builtins.print

# ============================================================
# SEEDS AND BACKBONE LIST
# ============================================================

# SEEDS is the global parameter defined in the first cell of the notebook.

BACKBONE_LIST = [
    "efficientnet_b0",
    "efficientnet_b1",
    "efficientnet_b2",
    "efficientnet_b3",
    "efficientnet_b4",
    "efficientnet_b5",
    "efficientnet_b6",
    "efficientnet_b7",
    "densenet121",
    "densenet161",
    "densenet169",
    "densenet201",
    "resnet18",
    "resnet34",
    "resnet50",
    "resnet101",
    "resnet152",
    "wide_resnet50_2",
    "wide_resnet101_2",
]

# BACKBONE_NAME, the backbone trained in this run, is set in the global settings cell.
if BACKBONE_NAME not in BACKBONE_LIST:
    raise ValueError(f"BACKBONE_NAME must be one of {BACKBONE_LIST}")

# Label metric display order used throughout
METRIC_LABELS = [
    ("label_acc",        "Label Accuracy    "),
    ("weighted_acc",     "Weighted Accuracy "),
    ("label_f1",         "Macro F1          "),
    ("macro_sensitivity","Macro Sensitivity "),
    ("mel_sensitivity",  "Mel. Sensitivity  "),
    ("macro_specificity","Macro Specificity "),
    ("auroc",            "AUROC             "),
]

# ============================================================
# SINGLE BACKBONE + SINGLE SEED RUN
# ============================================================

def run_backbone(backbone_name, run_id, seed,
                 meta, inconsistent_keys, concept_info, label_encoders):
    """
    Train and evaluate one CBM for a given backbone and seed.
    Returns a dict with all per-split metrics for downstream aggregation.
    """
    set_seed(seed)

    tag = (
        f"run{run_id:03d}"
        f"_seed{seed}"
        f"_{backbone_name}"
        f"_tl{CFG.TUNABLE_LAYERS}pct"
        f"_bs{CFG.BATCH_SIZE}"
        f"_{CFG.FILTER_STRATEGY}"
    )

    log_path = os.path.join(CFG.OUTPUT_DIR, f"{tag}.txt")
    log_file = open(log_path, "w", encoding="utf-8")

    def pf(*args, **kwargs):
        original_print(*args, **kwargs)
        sys.stdout.flush()
        log_file.write(" ".join(str(a) for a in args) + kwargs.get("end", "\n"))
        log_file.flush()

    try:
        pf("=" * 70)
        pf(f"RUN {run_id:03d}  |  seed={seed}  |  {backbone_name}")
        pf("=" * 70)
        pf(f"  tunable_layers   : {CFG.TUNABLE_LAYERS}%")
        pf(f"  optimizer        : Adam")
        pf(f"  batch_size       : {CFG.BATCH_SIZE}")
        pf(f"  filter_strategy  : {CFG.FILTER_STRATEGY}")
        pf(f"  concept_weighting: {CFG.CONCEPT_WEIGHTING}")
        pf(f"  epochs           : {CFG.EPOCHS} (with early stopping)")
        pf()

        # ---- official splits ----
        train_idx = pd.read_csv(os.path.join(CFG.META_DIR, "train_indexes.csv"))["indexes"].values
        valid_idx = pd.read_csv(os.path.join(CFG.META_DIR, "valid_indexes.csv"))["indexes"].values
        test_idx  = pd.read_csv(os.path.join(CFG.META_DIR, "test_indexes.csv"))["indexes"].values

        train_df = meta.iloc[train_idx].copy()
        valid_df = meta.iloc[valid_idx].copy()
        test_df  = meta.iloc[test_idx].copy()

        # ---- inconsistency filtering ----
        tb, vb, eb = len(train_df), len(valid_df), len(test_df)
        train_df = filter_inconsistent_samples(train_df, inconsistent_keys, CFG.FILTER_STRATEGY)
        valid_df = filter_inconsistent_samples(valid_df, inconsistent_keys, CFG.FILTER_STRATEGY)
        test_df  = filter_inconsistent_samples(test_df,  inconsistent_keys, CFG.FILTER_STRATEGY)
        pf(f"Filtering ({CFG.FILTER_STRATEGY}): "
           f"train {tb}->{len(train_df)}, "
           f"val {vb}->{len(valid_df)}, "
           f"test {eb}->{len(test_df)}")

        for _sname, _sdf in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
            _nm  = int((_sdf[CFG.TARGET_COLUMN] == 1).sum())
            _nnm = int((_sdf[CFG.TARGET_COLUMN] == 0).sum())
            pf(f"  {_sname:<6}: total={len(_sdf):>4} | "
               f"melanoma={_nm:>3} | non-melanoma={_nnm:>3}")
        pf()

        # ---- class weights ----
        y_train = train_df[CFG.TARGET_COLUMN].values
        present = np.unique(y_train)
        w_arr   = compute_class_weight("balanced", classes=present, y=y_train)
        fw      = np.ones(2)
        for cls, w in zip(present, w_arr):
            fw[cls] = w
        label_weights = torch.tensor(fw, dtype=torch.float32).to(CFG.DEVICE)

        concept_class_weights = {}
        for c in CFG.CONCEPT_COLUMNS:
            le    = label_encoders[c]
            y_enc = le.transform(train_df[c].astype(str).fillna("unknown"))
            pres  = np.unique(y_enc)
            w_arr = compute_class_weight("balanced", classes=pres, y=y_enc)
            fw    = np.ones(len(le.classes_))
            for cls, w in zip(pres, w_arr):
                fw[cls] = w
            concept_class_weights[c] = torch.tensor(fw, dtype=torch.float32).to(CFG.DEVICE)

        # ---- data loaders ----
        g = torch.Generator()
        g.manual_seed(seed)

        train_loader = DataLoader(
            Derm7ptDataset(train_df, CFG.IMAGES_DIR, concept_info, label_encoders, get_train_transforms()),
            batch_size=CFG.BATCH_SIZE, shuffle=True,
            num_workers=0, generator=g, pin_memory=True)

        val_loader = DataLoader(
            Derm7ptDataset(valid_df, CFG.IMAGES_DIR, concept_info, label_encoders, get_val_transforms()),
            batch_size=CFG.BATCH_SIZE, num_workers=2, pin_memory=True)

        test_loader = DataLoader(
            Derm7ptDataset(test_df, CFG.IMAGES_DIR, concept_info, label_encoders, get_val_transforms()),
            batch_size=CFG.BATCH_SIZE, num_workers=2, pin_memory=True)

        # ---- model, loss, optimizer ----
        model = ConceptBottleneckModel(backbone_name, concept_info).to(CFG.DEVICE)
        total_p     = sum(p.numel() for p in model.parameters())
        trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
        pf(f"Params: total={total_p:,} | trainable={trainable_p:,} | frozen={total_p - trainable_p:,}")

        criterion = CBMLoss(label_weights, concept_class_weights, use_concept_weights=CFG.CONCEPT_WEIGHTING)
        optimizer = optim.Adam(model.parameters(), lr=CFG.LEARNING_RATE, weight_decay=1.0e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS, eta_min=CFG.ETA_MIN)

        # ---- early stopping + best model tracking ----
        best_val_f1      = -1.0
        best_epoch       = 0
        best_state       = None
        patience         = 10
        patience_counter = 0
        epoch_test_f1_scores = []

        pf(f"Early stopping patience: {patience} epochs (monitoring val macro F1)\n")

        for epoch in range(CFG.EPOCHS):
            train_loss = train_epoch(model, train_loader, optimizer, criterion)
            val_acc, val_f1, val_c_acc, val_c_f1 = evaluate(model, val_loader)
            _, test_f1_epoch, _, _ = evaluate(model, test_loader)
            _, train_f1_epoch, _, _ = evaluate(model, train_loader)
            scheduler.step()
            lr = scheduler.get_last_lr()[0]

            epoch_test_f1_scores.append({
                "epoch":    epoch + 1,
                "train_f1": round(train_f1_epoch, 4),
                "val_f1":   round(val_f1,         4),
                "test_f1":  round(test_f1_epoch,  4),
            })

            is_better = val_f1 > best_val_f1 + 1e-5

            pf(f"Epoch {epoch+1:>3}/{CFG.EPOCHS} | "
               f"LR: {lr:.2e} | Loss: {train_loss:.4f} | "
               f"TrainF1: {train_f1_epoch:.4f} | "
               f"LabelAcc: {val_acc:.4f} | LabelF1: {val_f1:.4f} | "
               f"ConceptAcc: {val_c_acc:.4f} | ConceptF1: {val_c_f1:.4f} | "
               f"TestF1: {test_f1_epoch:.4f}"
               + (" <- best" if is_better else ""))

            if is_better:
                best_val_f1      = val_f1
                best_epoch       = epoch + 1
                best_state       = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= patience:
                pf(f"Early stopping triggered after {epoch+1} epochs")
                break

        # ---- restore best model ----
        if best_state is not None:
            pf(f"\nRestoring best model from epoch {best_epoch} (val F1 = {best_val_f1:.4f})")
            model.load_state_dict(best_state)
        else:
            pf("Warning: No improvement during training — using final model")

        # ---- Extended metrics on all three splits ----
        split_raw_results = {}
        for _sname, _loader in [("train", train_loader),
                                 ("valid", val_loader),
                                 ("test",  test_loader)]:
            _yt, _yp, _yprob, _ct, _cp = collect_predictions(model, _loader)
            _lm = compute_label_metrics(_yt, _yp, _yprob)
            _c_acc = float(np.mean([accuracy_score(_ct[i], _cp[i])
                                    for i in range(len(CFG.CONCEPT_COLUMNS))]))
            _c_f1  = float(np.mean([f1_score(_ct[i], _cp[i],
                                    average="macro", zero_division=0)
                                    for i in range(len(CFG.CONCEPT_COLUMNS))]))
            _per_c_acc = {
                c: round(float(accuracy_score(_ct[i], _cp[i])), 4)
                for i, c in enumerate(CFG.CONCEPT_COLUMNS)
            }
            _per_c_f1 = {
                c: round(float(f1_score(_ct[i], _cp[i], average="macro", zero_division=0)), 4)
                for i, c in enumerate(CFG.CONCEPT_COLUMNS)
            }
            split_raw_results[_sname] = {
                "lm":        _lm,
                "c_acc":     round(_c_acc, 4),
                "c_f1":      round(_c_f1,  4),
                "per_c_acc": _per_c_acc,
                "per_c_f1":  _per_c_f1,
            }

        pf(f"\n{'='*70}")
        pf(f"EXTENDED METRICS AT BEST CHECKPOINT  |  seed={seed}  |  {backbone_name}")
        pf(f"Best epoch: {best_epoch}  (val F1={best_val_f1:.4f})")
        pf(f"{'='*70}")

        for _sname in ("train", "valid", "test"):
            _r = split_raw_results[_sname]
            pf(f"  [{_sname}]")
            for _key, _lbl in METRIC_LABELS:
                pf(f"    {_lbl}: {_r['lm'][_key]:.4f}")
            pf(f"    Concept Accuracy  : {_r['c_acc']:.4f}")
            pf(f"    Concept F1        : {_r['c_f1']:.4f}")

        pf(f"\n  Per-Concept Accuracy and F1 (test split):")
        pf(f"  {'Concept':<25} {'Acc':>8} {'Macro-F1':>10}")
        pf(f"  {'-'*25} {'-'*8} {'-'*10}")
        for _c in CFG.CONCEPT_COLUMNS:
            _cacc = split_raw_results["test"]["per_c_acc"][_c]
            _cf1  = split_raw_results["test"]["per_c_f1"][_c]
            pf(f"  {_c:<25} {_cacc:>8.4f} {_cf1:>10.4f}")

        # ---- Per-epoch F1 table ----
        pf(f"\n{'─'*70}")
        pf(f"PER-EPOCH MACRO F1  |  seed={seed}  |  {backbone_name}")
        pf(f"{'Epoch':>6} {'Train_F1':>10} {'Val_F1':>10} {'Test_F1':>10}")
        for _e in epoch_test_f1_scores:
            pf(f"{_e['epoch']:>6} {_e['train_f1']:>10.4f} "
               f"{_e['val_f1']:>10.4f} {_e['test_f1']:>10.4f}")
        pf()

        # ---- checkpoint ----
        torch.save(
            {
                "model_state_dict":      best_state if best_state is not None else model.state_dict(),
                "backbone":              backbone_name,
                "seed":                  seed,
                "concept_info":          concept_info,
                "label_encoders":        label_encoders,
                "best_epoch":            best_epoch,
                "best_val_f1":           best_val_f1,
                "epoch_test_f1_scores":  epoch_test_f1_scores,
                "config": {
                    "tunable_layers":    CFG.TUNABLE_LAYERS,
                    "filter_strategy":   CFG.FILTER_STRATEGY,
                    "concept_weighting": CFG.CONCEPT_WEIGHTING,
                    "batch_size":        CFG.BATCH_SIZE,
                    "epochs_actual":     epoch + 1,
                },
                "split_results": split_raw_results,
            },
            os.path.join(CFG.OUTPUT_DIR, f"{tag}_best.pth"),
        )

        return {
            "seed":          seed,
            "backbone":      backbone_name,
            "best_epoch":    best_epoch,
            "best_val_f1":   round(best_val_f1, 4),
            "splits":        split_raw_results,
        }

    finally:
        log_file.close()
        original_print(f"  Log saved -> {log_path}")


# ============================================================
# MULTI-SEED SUMMARY PRINTER
# ============================================================

def print_multiseed_summary(backbone_name, seed_results):
    """
    Aggregate metrics across all seed runs for one backbone and print
    mean +/- std for every label metric, concept metric, and
    per-concept metric to console.
    """
    n = len(seed_results)
    original_print()
    original_print("*" * 70)
    original_print(f"MULTI-SEED SUMMARY  |  {backbone_name}  ({n} seeds: {SEEDS})")
    original_print(f"Filter strategy: {CFG.FILTER_STRATEGY}")
    original_print("*" * 70)

    seeds_used  = [r["seed"]        for r in seed_results]
    best_epochs = [r["best_epoch"]  for r in seed_results]
    best_val_f1s= [r["best_val_f1"] for r in seed_results]
    original_print(f"  Seeds        : {seeds_used}")
    original_print(f"  Best epochs  : {best_epochs}")
    original_print(f"  Best val F1  : {[f'{v:.4f}' for v in best_val_f1s]}")
    original_print(f"  Val F1 mean  : {np.mean(best_val_f1s):.4f} +/- {np.std(best_val_f1s, ddof=1):.4f}")

    for split_name in ("train", "valid", "test"):
        original_print()
        original_print(f"  [{split_name.upper()}]")
        original_print(f"  {'Metric':<26} {'Mean':>8} {'Std':>8}")
        original_print(f"  {'-'*26} {'-'*8} {'-'*8}")

        # Label metrics
        for key, lbl in METRIC_LABELS:
            vals = [r["splits"][split_name]["lm"][key] for r in seed_results]
            original_print(
                f"  {lbl} {np.mean(vals):>8.4f} {np.std(vals, ddof=1):>8.4f}"
            )

        # Aggregate concept metrics
        c_accs = [r["splits"][split_name]["c_acc"] for r in seed_results]
        c_f1s  = [r["splits"][split_name]["c_f1"]  for r in seed_results]
        original_print(
            f"  {'Concept Accuracy  ':<26} {np.mean(c_accs):>8.4f} {np.std(c_accs, ddof=1):>8.4f}"
        )
        original_print(
            f"  {'Concept F1        ':<26} {np.mean(c_f1s):>8.4f} {np.std(c_f1s, ddof=1):>8.4f}"
        )

        # Per-concept accuracy and F1
        original_print()
        original_print(f"  Per-Concept Metrics ({split_name}):")
        original_print(f"  {'Concept':<25} {'Acc Mean':>10} {'Acc Std':>8} {'F1 Mean':>9} {'F1 Std':>8}")
        original_print(f"  {'-'*25} {'-'*10} {'-'*8} {'-'*9} {'-'*8}")
        for c in CFG.CONCEPT_COLUMNS:
            acc_vals = [r["splits"][split_name]["per_c_acc"][c] for r in seed_results]
            f1_vals  = [r["splits"][split_name]["per_c_f1"][c]  for r in seed_results]
            original_print(
                f"  {c:<25} {np.mean(acc_vals):>10.4f} {np.std(acc_vals, ddof=1):>8.4f}"
                f" {np.mean(f1_vals):>9.4f} {np.std(f1_vals, ddof=1):>8.4f}"
            )

    original_print()
    original_print("*" * 70)
    original_print()


# ============================================================
# MAIN RUN
# ============================================================

meta = pd.read_csv(os.path.join(CFG.META_DIR, "meta.csv"))
meta[CFG.TARGET_COLUMN] = (
    meta[CFG.TARGET_COLUMN].astype(str).str.lower()
    .apply(lambda x: 1 if "melanoma" in x else 0)
)
concept_info, label_encoders = build_global_concept_encoders(meta)

# Compute inconsistent concept-profile keys once on the full dataset.
inconsistent_keys = get_inconsistent_concept_keys(meta)

original_print(f"Device              : {CFG.DEVICE}")
original_print(f"Output dir          : {CFG.OUTPUT_DIR}")
original_print(f"Backbones           : {len(BACKBONE_LIST)}")
original_print(f"Selected backbone   : {BACKBONE_NAME}")
original_print(f"Seeds               : {SEEDS}")
original_print(f"Filter strategy     : {CFG.FILTER_STRATEGY}")
original_print(
    f"Settings            : tl={CFG.TUNABLE_LAYERS}% | "
    f"bs={CFG.BATCH_SIZE} | epochs={CFG.EPOCHS} (early stop) | "
    f"Adam lr={CFG.LEARNING_RATE}"
)
original_print(f"Inconsistent profiles: {len(inconsistent_keys)}\n")

# Dataset split statistics — printed once before training
print_split_statistics(meta, inconsistent_keys)

all_backbone_results = {}

if RUN_TRAINING:
    for run_id, backbone_name in enumerate(BACKBONE_LIST):
        if backbone_name != BACKBONE_NAME:
            continue
        original_print(f"\n{'='*70}")
        original_print(f"Backbone {run_id + 1}/{len(BACKBONE_LIST)}: {backbone_name}")
        original_print(f"Running {len(SEEDS)} seeds: {SEEDS}")
        original_print(f"{'='*70}")

        seed_results = []
        for seed_idx, seed in enumerate(SEEDS):
            original_print(f"\n  -- Seed {seed_idx + 1}/{len(SEEDS)}: {seed} --")
            _tag = (f"run{run_id:03d}_seed{seed}_{backbone_name}_tl{CFG.TUNABLE_LAYERS}pct"
                    f"_bs{CFG.BATCH_SIZE}_{CFG.FILTER_STRATEGY}")
            _ck = os.path.join(CFG.OUTPUT_DIR, f"{_tag}_best.pth")
            if SKIP_TRAINED_SEEDS and os.path.exists(_ck):
                # a seed trained by an earlier run is reloaded, so adding seeds does not retrain the old ones
                _c = torch.load(_ck, map_location="cpu", weights_only=False)
                original_print(f"  checkpoint found, seed {seed} is not retrained -> {_ck}")
                result = {"seed": seed, "backbone": backbone_name, "best_epoch": _c["best_epoch"],
                          "best_val_f1": round(_c["best_val_f1"], 4), "splits": _c["split_results"]}
            else:
                result = run_backbone(
                    backbone_name, run_id, seed,
                    meta, inconsistent_keys, concept_info, label_encoders
                )
            seed_results.append(result)

        # Aggregate across seeds and print summary for this backbone
        print_multiseed_summary(backbone_name, seed_results)
        all_backbone_results[backbone_name] = seed_results

    # ============================================================
    # GRAND SUMMARY ACROSS ALL BACKBONES
    # ============================================================

    original_print("\n" + "#" * 70)
    original_print("GRAND SUMMARY — ALL BACKBONES (test split, mean +/- std across seeds)")
    original_print(f"Filter strategy: {CFG.FILTER_STRATEGY}")
    original_print("#" * 70)

    col_w = 22
    header = f"{'Backbone':<{col_w}}"
    for _, lbl in METRIC_LABELS:
        header += f" {lbl.strip():>18}"
    header += f" {'Concept Acc':>13} {'Concept F1':>11}"
    original_print(header)
    original_print("-" * len(header))

    for backbone_name, seed_results in all_backbone_results.items():
        row = f"{backbone_name:<{col_w}}"
        for key, _ in METRIC_LABELS:
            vals = [r["splits"]["test"]["lm"][key] for r in seed_results]
            row += f"  {np.mean(vals):.4f}+/-{np.std(vals, ddof=1):.4f}"
        c_accs = [r["splits"]["test"]["c_acc"] for r in seed_results]
        c_f1s  = [r["splits"]["test"]["c_f1"]  for r in seed_results]
        row += f"  {np.mean(c_accs):.4f}+/-{np.std(c_accs, ddof=1):.4f}"
        row += f"  {np.mean(c_f1s):.4f}+/-{np.std(c_f1s, ddof=1):.4f}"
        original_print(row)

    original_print("#" * 70)
else:
    original_print("RUN_TRAINING is False. The checkpoints saved by an earlier run of this cell are reused as they are.")

Device              : cuda
Output dir          : /content/drive/MyDrive/Notebooks/Derm7pt/models
Backbones           : 19
Selected backbone   : efficientnet_b4
Seeds               : [42, 64, 128, 256, 512]
Filter strategy     : asymmetric
Settings            : tl=75% | bs=16 | epochs=20 (early stop) | Adam lr=0.0005
Inconsistent profiles: 50

DATASET SPLIT STATISTICS

--- Strategy: symmetric ---
Split      Total   Melanoma   Non-Mel   Boundary   Profiles
-----------------------------------------------------------------
train        298         42       256        115        133
valid        135         25       110         68         87
test         272         49       223        123        131

  Profile overlap (symmetric):
    train ∩ valid :   45 profiles
    train ∩ test  :   46 profiles
    valid ∩ test  :   35 profiles
    test-only     :   80 profiles (unseen at training time)

--- Strategy: asymmetric ---
Split      Total   Melanoma   Non-Mel   Boundary   Profiles
-----------

100%|██████████| 74.5M/74.5M [00:00<00:00, 163MB/s]


  Backbone efficientnet_b4: 9 layers | frozen=2 | trainable=7 | tunable=75%
Params: total=17,603,046 | trainable=17,597,508 | frozen=5,538
Early stopping patience: 10 epochs (monitoring val macro F1)

Epoch   1/20 | LR: 4.97e-04 | Loss: 2.0533 | TrainF1: 0.4633 | LabelAcc: 0.5673 | LabelF1: 0.5380 | ConceptAcc: 0.6149 | ConceptF1: 0.4066 | TestF1: 0.5361 <- best
Epoch   2/20 | LR: 4.88e-04 | Loss: 1.8825 | TrainF1: 0.6312 | LabelAcc: 0.7368 | LabelF1: 0.7163 | ConceptAcc: 0.6082 | ConceptF1: 0.4580 | TestF1: 0.7060 <- best
Epoch   3/20 | LR: 4.73e-04 | Loss: 1.7516 | TrainF1: 0.7305 | LabelAcc: 0.7076 | LabelF1: 0.6175 | ConceptAcc: 0.6241 | ConceptF1: 0.4730 | TestF1: 0.6955
Epoch   4/20 | LR: 4.53e-04 | Loss: 1.5749 | TrainF1: 0.7534 | LabelAcc: 0.7251 | LabelF1: 0.6575 | ConceptAcc: 0.6241 | ConceptF1: 0.4869 | TestF1: 0.7208
Epoch   5/20 | LR: 4.28e-04 | Loss: 1.5304 | TrainF1: 0.7878 | LabelAcc: 0.7661 | LabelF1: 0.7293 | ConceptAcc: 0.6358 | ConceptF1: 0.5163 | TestF1: 0.7920 <- 

## Correctly classified lesions and the manifest for Part 2

The best checkpoint of every seed is reloaded and applied to the filtered validation and test splits. The manifest records the backbone, filter, paths and checkpoints that Part 2 reads.

In [5]:
# ============================================================
# CORRECTLY CLASSIFIED VALIDATION AND TEST LESIONS
# ============================================================
# The best checkpoint of every seed is reloaded and applied to the filtered validation and test
# splits. One row per lesion records the prediction, so the experiments can keep the correctly
# classified lesions. A manifest describes the trained models for the experimental part.
# This cell runs after training and does not touch the random streams used above.

import json
from torch.utils.data import DataLoader


def patient_ids(df):
    """Patient identifier of every lesion: case_num when the metadata has it, else the image path."""
    return (df["case_num"] if "case_num" in df.columns else df["derm"]).astype(str).values


def filtered_splits(meta, inconsistent_keys):
    """The filtered splits exactly as run_backbone builds them."""
    frames = {}
    for name in ("train", "valid", "test"):
        idx = pd.read_csv(os.path.join(CFG.META_DIR, f"{name}_indexes.csv"))["indexes"].values
        frames[name] = filter_inconsistent_samples(meta.iloc[idx].copy(), inconsistent_keys, CFG.FILTER_STRATEGY)
    return frames


run_id = BACKBONE_LIST.index(BACKBONE_NAME)
frames = filtered_splits(meta, inconsistent_keys)
manifest = {
    "backbone": BACKBONE_NAME, "filter_strategy": CFG.FILTER_STRATEGY,
    "meta_dir": CFG.META_DIR, "images_dir": CFG.IMAGES_DIR, "models_dir": CFG.OUTPUT_DIR,
    "training": {"tunable_layers": CFG.TUNABLE_LAYERS, "concept_weighting": CFG.CONCEPT_WEIGHTING,
                 "batch_size": CFG.BATCH_SIZE, "epochs": CFG.EPOCHS, "img_size": CFG.IMG_SIZE,
                 "learning_rate": CFG.LEARNING_RATE, "eta_min": CFG.ETA_MIN,
                 "lambda1": CFG.LAMBDA1, "lambda2": CFG.LAMBDA2},
    "seeds": {},
}
summary = []
for seed in SEEDS:
    tag = (f"run{run_id:03d}_seed{seed}_{BACKBONE_NAME}_tl{CFG.TUNABLE_LAYERS}pct"
           f"_bs{CFG.BATCH_SIZE}_{CFG.FILTER_STRATEGY}")
    ckpt_path = os.path.join(CFG.OUTPUT_DIR, f"{tag}_best.pth")
    ckpt = torch.load(ckpt_path, map_location=CFG.DEVICE, weights_only=False)
    model = ConceptBottleneckModel(BACKBONE_NAME, concept_info).to(CFG.DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    rows = []
    for split in ("valid", "test"):
        df = frames[split]
        loader = DataLoader(Derm7ptDataset(df, CFG.IMAGES_DIR, concept_info, label_encoders, get_val_transforms()),
                            batch_size=CFG.BATCH_SIZE, num_workers=2, pin_memory=True)
        y_true, y_pred, y_prob, _, _ = collect_predictions(model, loader)
        logged_f1 = ckpt["split_results"][split]["lm"]["label_f1"]
        recomputed_f1 = compute_label_metrics(y_true, y_pred, y_prob)["label_f1"]
        rows.append(pd.DataFrame({
            "split": split, "position": np.arange(len(df)), "derm": df["derm"].astype(str).values,
            "patient_id": patient_ids(df), "y_true": y_true, "y_pred": y_pred,
            "p_melanoma": y_prob, "correct": (y_true == y_pred).astype(int)}))
        summary.append({"seed": seed, "split": split, "lesions": len(df),
                        "correct": int((y_true == y_pred).sum()),
                        "correct_melanoma": int(((y_true == 1) & (y_pred == 1)).sum()),
                        "correct_non_melanoma": int(((y_true == 0) & (y_pred == 0)).sum()),
                        "macro_f1_logged": logged_f1, "macro_f1_recomputed": recomputed_f1})
    pred_path = os.path.join(CFG.OUTPUT_DIR, f"{tag}_predictions.csv")
    pd.concat(rows, ignore_index=True).to_csv(pred_path, index=False)
    manifest["seeds"][str(seed)] = {"tag": tag, "checkpoint": ckpt_path, "predictions": pred_path,
                                    "best_epoch": ckpt["best_epoch"], "best_val_f1": ckpt["best_val_f1"]}
    del model
    torch.cuda.empty_cache()

manifest_path = os.path.join(CFG.OUTPUT_DIR, f"manifest_{BACKBONE_NAME}_{CFG.FILTER_STRATEGY}.json")
for path in (manifest_path, os.path.join(CFG.OUTPUT_DIR, "manifest_latest.json")):
    with open(path, "w") as fh:
        json.dump(manifest, fh, indent=1)

summary = pd.DataFrame(summary)
original_print(summary.to_string(index=False))
if not np.allclose(summary["macro_f1_logged"], summary["macro_f1_recomputed"], atol=1e-4):
    original_print("Warning: recomputed macro F1 differs from the value logged at training time.")
original_print(f"Manifest written to {manifest_path}")

  Backbone efficientnet_b4: 9 layers | frozen=2 | trainable=7 | tunable=75%
  Backbone efficientnet_b4: 9 layers | frozen=2 | trainable=7 | tunable=75%
  Backbone efficientnet_b4: 9 layers | frozen=2 | trainable=7 | tunable=75%
  Backbone efficientnet_b4: 9 layers | frozen=2 | trainable=7 | tunable=75%
  Backbone efficientnet_b4: 9 layers | frozen=2 | trainable=7 | tunable=75%
 seed split  lesions  correct  correct_melanoma  correct_non_melanoma  macro_f1_logged  macro_f1_recomputed
   42 valid      171      145                43                   102           0.8274               0.8274
   42  test      324      267                67                   200           0.7884               0.7884
   64 valid      171      142                39                   103           0.8028               0.8028
   64  test      324      281                68                   213           0.8340               0.8340
  128 valid      171      143                44                    99           

# Part 2. Population geometry of the explanations

Part 2 trains no model. It reloads the checkpoints of Part 1, explains every correctly classified lesion with SOFI on the deployed hard bottleneck with zero removal, and treats the melanomas and the non-melanomas as two separate populations. Each population explains the logit of its own class.

Every table is printed in full to the console, whose text is also mirrored to `console_log.txt` in the output folder. The csv files are written once, in the last cell, with one file per table type. They hold everything needed for later figures. Image-level tables and mean removal curves go to csv only, unless `PRINT_IMAGE_LEVEL = True`.

In [ ]:
# ============================================================
# PART 2 CONFIGURATION, CONSOLE LOG AND TABLE REGISTRY
# Backbone, filtering strategy, data paths and training hyperparameters come from the manifest that
# Part 1 writes. For every seed, the correctly classified validation lesions tune the two analysis
# choices (inverse temperature and tier threshold), which are stored in FROZEN_FILE and reused by every
# later run. The correctly classified test lesions carry every reported result.
# Every table is printed in full to the console, and the console is mirrored to console_log.txt.
# The csv files are written once, at the end of the notebook, one file per table type.
# ============================================================
import os
import sys
import json
import time
import copy
import math
import random
import warnings
import itertools
import builtins
import hashlib

import numpy as np
import pandas as pd
from PIL import Image

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import adjusted_rand_score
from scipy import stats as sps

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda v: f"{v:.4f}" if (v == 0 or not np.isfinite(v) or 1e-3 <= abs(v) < 1e7) else f"{v:.3e}")


class Config:
    # ---------------- inputs and outputs ----------------
    TRAINED_MODELS_DIR = MODELS_PATH
    MANIFEST_FILE = f"manifest_{BACKBONE_NAME}_{FILTER_STRATEGY}.json"
    LOCAL_IMAGES_DIR = LOCAL_IMAGES_PATH
    STAGE_IMAGES = True                      # False when the images already sit on a fast disk
    OUTPUT_ROOT = APG_OUTPUT_PATH

    # ---------------- populations ----------------
    # Each population explains the logit of its own class, s for melanoma and -s for non-melanoma.
    POPULATIONS = {"melanoma": 1, "non_melanoma": 0}
    POPULATION_LABEL = {"melanoma": "melanoma", "non_melanoma": "non-melanoma"}
    MIN_POPULATION_LESIONS = 10

    # ---------------- ranking laws and population report ----------------
    CBM_TYPE = "hard"                        # deployed hard concept bottleneck, zero removal
    BETA0_GRID = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 7.5, 10.0]
    TAU_GRID = [0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.14, 0.16, 0.18, 0.20, 0.25, 0.30]
    DELTA_GRID = [0.01, 0.05, 0.10]          # levels of the certificates in the sensitivity analysis
    TAU_BOOTSTRAP = 200
    TAU_TIE_DECIMALS = 2
    DELTA = 0.05
    N_BOOT = 1000
    N_PERM = 2000
    MAX_PCA_COMPONENTS = 5
    PCA_COMPONENTS_TO_PROFILE = 2            # principal exchange patterns profiled by concept states
    MIN_STATE_IMAGES = 5                     # smallest state group used in a state-conditioned statistic

    # ---------------- model studied in depth (first and third experiments) ----------------
    # The seed with the largest value of this test metric is studied in depth. It has the most correctly
    # classified test images, so the illustration explains the largest number of images.
    SELECTION_METRIC = "label_f1"
    SELECTION_SPLIT = "test"

    # ---------------- single global rankings (third experiment) ----------------
    # "test" builds each global ranking from the explanations of the test lesions it is then applied to,
    # which is the most favorable setting for a single ranking. "valid" builds it on the validation
    # lesions of the same class, then applies it to the test lesions.
    EXP3_FIT_SOURCES = ("test", "valid")

    # ---------------- console ----------------
    PRINT_IMAGE_LEVEL = False                # True also prints the image-level tables (long)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    CONCEPT_COLUMNS = ["pigment_network", "streaks", "pigmentation", "regression_structures",
                       "dots_and_globules", "blue_whitish_veil", "vascular_structures"]
    CONCEPT_ABBREV = ["PN", "ST", "PIG", "RS", "DG", "BWV", "VS"]
    CONCEPT_NAME = ["pigment network", "streaks", "pigmentation", "regression structures",
                    "dots and globules", "blue-whitish veil", "vascular structures"]
    CHECKLIST_MAJOR = ["pigment_network", "blue_whitish_veil", "vascular_structures"]
    CHECKLIST_MINOR = ["streaks", "pigmentation", "dots_and_globules", "regression_structures"]
    TARGET_COLUMN = "diagnosis"


CFG = Config()
with open(os.path.join(CFG.TRAINED_MODELS_DIR, CFG.MANIFEST_FILE)) as _fh:
    MANIFEST = json.load(_fh)
_missing = [s for s in SEEDS if str(s) not in MANIFEST["seeds"]]
if _missing:
    raise ValueError(f"Seeds {_missing} have no trained model in {CFG.MANIFEST_FILE}. Run Part 1 for them first.")

CFG.SEEDS = list(SEEDS)
CFG.BACKBONE = MANIFEST["backbone"]
CFG.FILTER_STRATEGY = MANIFEST["filter_strategy"]
CFG.META_DIR = MANIFEST["meta_dir"]
CFG.IMAGES_DIR = MANIFEST["images_dir"]
_tr = MANIFEST["training"]
CFG.PRETRAINED = False                   # checkpoints supply every weight
CFG.TUNABLE_LAYERS = _tr["tunable_layers"]
CFG.BATCH_SIZE = _tr["batch_size"]
CFG.IMG_SIZE = _tr["img_size"]

CFG.OUTPUT_DIR = os.path.join(CFG.OUTPUT_ROOT, f"casestudy_{CFG.CBM_TYPE}_cbm_{CFG.BACKBONE}_{CFG.FILTER_STRATEGY}")
CFG.CACHE_DIR = os.path.join(CFG.OUTPUT_DIR, "cache")
CFG.CSV_DIR = os.path.join(CFG.OUTPUT_DIR, "csv")
for _d in (CFG.OUTPUT_DIR, CFG.CACHE_DIR, CFG.CSV_DIR):
    os.makedirs(_d, exist_ok=True)


def model_stamp(seed):
    """Short fingerprint of the checkpoint of one seed (path, size and modification time)."""
    ck = MANIFEST["seeds"][str(seed)]["checkpoint"]
    st = os.stat(ck)
    return hashlib.md5(f"{ck}|{st.st_size}|{int(st.st_mtime)}".encode()).hexdigest()[:10]


CFG.FROZEN_FILE = os.path.join(CFG.OUTPUT_DIR, "frozen_choices.json")   # keys carry the seed and its checkpoint stamp
CFG.ACTIVE_IMAGES_DIR = CFG.IMAGES_DIR
N_ATTR = len(CFG.CONCEPT_COLUMNS)
AB = CFG.CONCEPT_ABBREV
NAME = CFG.CONCEPT_NAME

# ---------------- console mirror and table registry ----------------
_LOG_PATH = os.path.join(CFG.OUTPUT_DIR, "console_log.txt")
with open(_LOG_PATH, "w", encoding="utf-8") as _fh:
    _fh.write(f"Console log started {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
TABLES = {}


def say(*args):
    """Print to the console and append the same text to console_log.txt."""
    msg = " ".join(str(a) for a in args)
    builtins.print(msg)
    sys.stdout.flush()
    with open(_LOG_PATH, "a", encoding="utf-8") as fh:
        fh.write(msg + "\n")


def banner(title, char="="):
    say("")
    say(char * 100)
    say(title)
    say(char * 100)


def show(df, title, key=None, echo=True):
    """Print a whole table to the console. With a key, the table also joins the csv file of that key."""
    if not isinstance(df, pd.DataFrame):
        df = pd.DataFrame(df)
    if key is not None:
        TABLES.setdefault(key, []).append(df)
    if echo:
        say("")
        say(f"--- {title} ({len(df)} rows)")
        say(df.to_string(index=False) if len(df) else "(empty table)")
    return df


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed(CFG.SEEDS[0])
if torch.cuda.is_available():
    _p = torch.cuda.get_device_properties(0)
    say(f"GPU {_p.name} with {_p.total_memory / 2**30:.1f} GB")
else:
    say("No GPU found. Feature extraction runs on the CPU, which is slow but correct.")
say(f"Backbone {CFG.BACKBONE} | filter {CFG.FILTER_STRATEGY} | seeds {CFG.SEEDS} | "
    f"populations {list(CFG.POPULATIONS)} | outputs in {CFG.OUTPUT_DIR}")
with open(os.path.join(CFG.OUTPUT_DIR, "00_configuration.json"), "w") as _fh:
    json.dump({"manifest": MANIFEST, **{k: str(v) for k, v in vars(Config).items() if k.isupper()}}, _fh, indent=1)

## Concept bottleneck model

The definitions match Part 1 and serve only to reload its checkpoints. The image encoder runs once per lesion, after which every removal game is evaluated on the concept heads and the diagnosis head.

In [7]:
# ============================================================
# CONCEPT BOTTLENECK MODEL (RELOAD ONLY)
# Definitions matching Part 1. They reload its checkpoints, then run the image encoder once per lesion.
# Part 2 trains no model.
# ============================================================
import shutil


def stage_images_locally():
    """Copy the images to a fast local disk once, since remote reads dominate extraction time."""
    if os.path.isdir(CFG.LOCAL_IMAGES_DIR) and any(os.scandir(CFG.LOCAL_IMAGES_DIR)):
        return CFG.LOCAL_IMAGES_DIR
    t0 = time.time()
    shutil.copytree(CFG.IMAGES_DIR, CFG.LOCAL_IMAGES_DIR, dirs_exist_ok=True)
    say(f"Images copied to local disk in {time.time() - t0:.0f} s")
    return CFG.LOCAL_IMAGES_DIR


def resolve_images_dir():
    missing = [p for p in (CFG.META_DIR, CFG.IMAGES_DIR) if not os.path.isdir(p)]
    if missing:
        raise FileNotFoundError(f"Set META_PATH and IMAGES_PATH to the Derm7pt folders. Not found: {missing}.")
    if not os.path.exists(os.path.join(CFG.META_DIR, "meta.csv")):
        raise FileNotFoundError(f"meta.csv is missing in {CFG.META_DIR}.")
    return stage_images_locally() if CFG.STAGE_IMAGES else CFG.IMAGES_DIR


# ---------------- inconsistency filtering, identical to Part 1 ----------------

def _add_concept_key(df):
    df = df.copy()
    for c in CFG.CONCEPT_COLUMNS:
        df[c] = df[c].astype(str).str.strip().str.lower().fillna("unknown")
    df["_concept_key"] = df[CFG.CONCEPT_COLUMNS].apply(lambda row: "|".join(f"{c}={v}" for c, v in row.items()), axis=1)
    return df


def get_inconsistent_concept_keys(df):
    df = _add_concept_key(df)
    groups = df.groupby("_concept_key")[CFG.TARGET_COLUMN].agg(set)
    return set(groups[groups.apply(lambda s: len(s) > 1)].index)


def filter_inconsistent_samples(df, inconsistent_keys, strategy):
    if strategy == "none":
        return df
    df = _add_concept_key(df)
    in_boundary = df["_concept_key"].isin(inconsistent_keys)
    if strategy == "symmetric":
        mask = ~in_boundary
    elif strategy == "asymmetric":
        mask = (~in_boundary) | (in_boundary & (df[CFG.TARGET_COLUMN] == 1))
    else:
        raise ValueError(strategy)
    return df[mask].drop(columns=["_concept_key"]).reset_index(drop=True)


def build_global_concept_encoders(full_meta_df):
    concept_info, label_encoders = {}, {}
    for c in CFG.CONCEPT_COLUMNS:
        le = LabelEncoder()
        le.fit(full_meta_df[c].astype(str).fillna("unknown").values)
        label_encoders[c] = le
        concept_info[c] = {"n_classes": len(le.classes_), "states": [str(s) for s in le.classes_]}
    return concept_info, label_encoders


def patient_ids(df):
    return (df["case_num"] if "case_num" in df.columns else df["derm"]).astype(str).values


def load_data():
    """Binary diagnosis, official splits, inconsistent keys from the full dataset, the filter of Part 1."""
    meta = pd.read_csv(os.path.join(CFG.META_DIR, "meta.csv"))
    meta[CFG.TARGET_COLUMN] = meta[CFG.TARGET_COLUMN].astype(str).str.lower().apply(lambda x: 1 if "melanoma" in x else 0)
    concept_info, label_encoders = build_global_concept_encoders(meta)
    keys = get_inconsistent_concept_keys(meta)
    idx = {s: pd.read_csv(os.path.join(CFG.META_DIR, f"{s}_indexes.csv"))["indexes"].values
           for s in ("train", "valid", "test")}
    splits = {s: filter_inconsistent_samples(meta.iloc[idx[s]].copy(), keys, CFG.FILTER_STRATEGY) for s in idx}
    return meta, splits, concept_info, label_encoders, keys


# ---------------- backbone, dataset, model ----------------

def _tv(name, wname):
    return lambda: getattr(models, name)(weights=getattr(models, wname).DEFAULT if CFG.PRETRAINED else None)


_BACKBONES = {
    "efficientnet_b0": (_tv("efficientnet_b0", "EfficientNet_B0_Weights"), "features", 1280),
    "efficientnet_b1": (_tv("efficientnet_b1", "EfficientNet_B1_Weights"), "features", 1280),
    "efficientnet_b2": (_tv("efficientnet_b2", "EfficientNet_B2_Weights"), "features", 1408),
    "efficientnet_b3": (_tv("efficientnet_b3", "EfficientNet_B3_Weights"), "features", 1536),
    "efficientnet_b4": (_tv("efficientnet_b4", "EfficientNet_B4_Weights"), "features", 1792),
    "efficientnet_b5": (_tv("efficientnet_b5", "EfficientNet_B5_Weights"), "features", 2048),
    "efficientnet_b6": (_tv("efficientnet_b6", "EfficientNet_B6_Weights"), "features", 2304),
    "efficientnet_b7": (_tv("efficientnet_b7", "EfficientNet_B7_Weights"), "features", 2560),
    "densenet121": (_tv("densenet121", "DenseNet121_Weights"), "features", 1024),
    "densenet161": (_tv("densenet161", "DenseNet161_Weights"), "features", 2208),
    "densenet169": (_tv("densenet169", "DenseNet169_Weights"), "features", 1664),
    "densenet201": (_tv("densenet201", "DenseNet201_Weights"), "features", 1920),
    "resnet18": (_tv("resnet18", "ResNet18_Weights"), "children", 512),
    "resnet34": (_tv("resnet34", "ResNet34_Weights"), "children", 512),
    "resnet50": (_tv("resnet50", "ResNet50_Weights"), "children", 2048),
    "resnet101": (_tv("resnet101", "ResNet101_Weights"), "children", 2048),
    "resnet152": (_tv("resnet152", "ResNet152_Weights"), "children", 2048),
    "wide_resnet50_2": (_tv("wide_resnet50_2", "Wide_ResNet50_2_Weights"), "children", 2048),
    "wide_resnet101_2": (_tv("wide_resnet101_2", "Wide_ResNet101_2_Weights"), "children", 2048),
}


def get_backbone(name):
    ctor, kind, dim = _BACKBONES[name]
    m = ctor()
    features = m.features if kind == "features" else nn.Sequential(*list(m.children())[:-2])
    return features, dim


_IMAGE_PATHS = {}


def _image_index(images_dir):
    if images_dir not in _IMAGE_PATHS:
        d = {}
        for root, _, files in os.walk(images_dir):
            for f in files:
                if f.lower().endswith((".jpg", ".jpeg", ".png")):
                    full = os.path.join(root, f)
                    d[os.path.relpath(full, images_dir).replace("\\", "/").lower()] = full
        _IMAGE_PATHS[images_dir] = d
    return _IMAGE_PATHS[images_dir]


class Derm7ptDataset(Dataset):
    def __init__(self, df, images_dir, concept_info, label_encoders, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.image_paths = _image_index(images_dir)
        for c in CFG.CONCEPT_COLUMNS:
            self.df[c] = label_encoders[c].transform(df[c].astype(str).fillna("unknown").values)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        rel_path = str(row["derm"]).replace("\\", "/").lower()
        if rel_path not in self.image_paths:
            raise FileNotFoundError(f"Image not found: {rel_path}")
        image = Image.open(self.image_paths[rel_path]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        concepts = torch.tensor([int(row[c]) for c in CFG.CONCEPT_COLUMNS], dtype=torch.long)
        label = torch.tensor(int(row[CFG.TARGET_COLUMN]), dtype=torch.long)
        return image, concepts, label


def get_val_transforms():
    return transforms.Compose([transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)), transforms.ToTensor(),
                               transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])


class ConceptBottleneckModel(nn.Module):
    """Same modules and forward pass as Part 1, with the encoder exposed so it runs once per lesion."""

    def __init__(self, backbone_name, concept_info, cbm_type="hard", num_classes=2):
        super().__init__()
        self.cbm_type = cbm_type
        self.feature_extractor, self.feature_dim = get_backbone(backbone_name)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.concept_info = concept_info
        self.concept_heads = nn.ModuleDict({c: nn.Linear(self.feature_dim, concept_info[c]["n_classes"])
                                            for c in CFG.CONCEPT_COLUMNS})
        total_concept_dim = sum(concept_info[c]["n_classes"] for c in CFG.CONCEPT_COLUMNS)
        self.classifier = nn.Sequential(nn.Linear(total_concept_dim, 128), nn.LayerNorm(128), nn.ReLU(),
                                        nn.Dropout(0.3), nn.Linear(128, num_classes))

    def encode(self, x):
        f = self.feature_extractor(x)
        return self.pool(f).view(f.size(0), -1)

    def concept_logits_from_features(self, z):
        return [self.concept_heads[c](z) for c in CFG.CONCEPT_COLUMNS]

    def bottleneck(self, concept_logits):
        out = []
        for c, lg in zip(CFG.CONCEPT_COLUMNS, concept_logits):
            if self.cbm_type == "hard":
                out.append(F.one_hot(torch.argmax(lg, dim=1), num_classes=self.concept_info[c]["n_classes"]).float())
            else:
                out.append(torch.softmax(lg, dim=1))
        return torch.cat([o.detach() for o in out], dim=1)

    def forward(self, x):
        concept_logits = self.concept_logits_from_features(self.encode(x))
        return concept_logits, self.classifier(self.bottleneck(concept_logits))


def extract_features(model, splits, label_encoders, concept_info):
    """Run the image encoder once per lesion, keeping pooled features, labels and patient identifiers."""
    out = {}
    model.eval()
    for s in ("train", "valid", "test"):
        df = splits[s]
        ds = Derm7ptDataset(df, CFG.ACTIVE_IMAGES_DIR, concept_info, label_encoders, get_val_transforms())
        Z, C, Y = [], [], []
        with torch.no_grad():
            for x, c, y in DataLoader(ds, batch_size=64, num_workers=2, pin_memory=True):
                Z.append(model.encode(x.to(CFG.DEVICE)).double().cpu().numpy())
                C.append(c.numpy())
                Y.append(y.numpy())
        out[s] = {"z": np.concatenate(Z), "concepts": np.concatenate(C), "y": np.concatenate(Y),
                  "derm": df["derm"].astype(str).values, "case": patient_ids(df)}
    return out

## Ranking geometry library

Exact computations on the permutohedron of the seven concepts (5040 rankings). The library covers the SOFI score in Weyl coordinates (Lemma 1), the moment polytope with its Borda consensus (Theorem 3), root walls with the majority tournament (Theorem 4), exchange coordinates (Theorem 6), ordered tiers as faces with their Lie closure (Theorem 7), the finite-sample rule (Proposition 8) and the certified partial order (Proposition 9).

In [8]:
# ============================================================
# RANKING GEOMETRY LIBRARY
# Ranking laws on the permutohedron (Theorem 3), root walls and majority tournaments (Theorem 4),
# exchange coordinates (Theorem 6), ordered tiers as faces (Theorem 7), the finite-sample rule
# (Proposition 8) and the certified partial order (Proposition 9). Everything is exact for n >= 2.
# Subsets of C = {0,...,n-1} are encoded as bitmasks (bit j <=> attribute j).
# ============================================================
import itertools
import math
import numpy as np
from scipy.special import logsumexp, comb
from scipy.linalg import expm

LOG_ZERO = -np.inf


class Lattice:
    """Boolean lattice of subsets of C with the index tables used by every recursion."""

    def __init__(self, n):
        self.n = int(n)
        self.N = 1 << self.n
        idx = np.arange(self.N)
        self.idx = idx
        self.mem = ((idx[:, None] >> np.arange(self.n)) & 1).astype(bool)      # (N, n)
        self.size = self.mem.sum(1)
        self.full = self.N - 1
        self.comp = self.full ^ idx
        bits = (1 << np.arange(self.n))
        self.rm = np.where(self.mem, idx[:, None] ^ bits[None, :], -1)          # S \ {j}
        self.add = np.where(~self.mem, idx[:, None] | bits[None, :], -1)        # S U {j}
        self.layers = [np.where(self.size == k)[0] for k in range(self.n + 1)]
        self.interior = (self.size > 0) & (self.size < self.n)


_LATTICES = {}


def lattice(n):
    if n not in _LATTICES:
        _LATTICES[n] = Lattice(n)
    return _LATTICES[n]


def weyl_rho(n):
    """Weyl vector of sl_n, rho_k = (n+1)/2 - k for positions k = 1..n."""
    return (n + 1) / 2.0 - np.arange(1, n + 1, dtype=float)


def rho_norm2(n):
    return n * (n * n - 1) / 12.0


# ------------------------------------------------------------
# Removal games and the ranking score
# ------------------------------------------------------------

def eta_from_u(u, lat):
    """eta_x(S) = u_x(C minus S) - u_x(S); u has shape (L, 2^n)."""
    u = np.atleast_2d(u)
    return u[:, lat.comp] - u


def deg_from_eta_perm(eta, perms, lat):
    """Score of each ordering as deg(sigma) = sum_k eta(S_k), summed over the prefixes of sigma."""
    n = lat.n
    bits = (1 << perms)                            # (P, n)
    pref = np.cumsum(bits, axis=1)[:, :n - 1]      # prefixes S_1..S_{n-1}
    eta = np.atleast_2d(eta)
    return eta[:, pref].sum(-1)                    # (L, P)


def morf_lerf_deg(u, perm, lat):
    """Trapezoid areas of the two removal curves for one ordering, with unit steps."""
    n = lat.n
    pref = [0]
    for j in perm:
        pref.append(pref[-1] | (1 << int(j)))
    morf = sum(0.5 * (u[pref[k]] + u[pref[k + 1]]) for k in range(n))
    # LeRF removes least relevant first, the removed set after k steps is C minus S_{n-k}
    lerf_sets = [lat.full ^ pref[n - k] for k in range(n + 1)]
    lerf = sum(0.5 * (u[lerf_sets[k]] + u[lerf_sets[k + 1]]) for k in range(n))
    return lerf - morf


_PERMS = {}


def all_perms(n):
    if n not in _PERMS:
        if n > 9:
            raise ValueError("Enumeration of S_n only for n <= 9.")
        _PERMS[n] = np.array(list(itertools.permutations(range(n))), dtype=np.int64)
    return _PERMS[n]


def vertex(perm):
    """sigma . rho : attribute j receives rho at its position."""
    n = len(perm)
    rho = weyl_rho(n)
    v = np.empty(n)
    v[np.asarray(perm)] = rho
    return v


def perm_matrix(perm):
    """P_sigma(j,k) = 1 iff sigma(k) = j."""
    n = len(perm)
    P = np.zeros((n, n))
    P[np.asarray(perm), np.arange(n)] = 1.0
    return P


# ------------------------------------------------------------
# Forward and backward recursions over the subset lattice
# ------------------------------------------------------------

def gibbs_sofi_dp(u, beta, lat=None, want_pairs=True, want_chain=True, chunk=None):
    """
    Exact Gibbs-SOFI quantities for a batch of games u (L, 2^n) at inverse temperature beta
    (scalar or (L,) array). Returns dict with logZ, P (L,n,n) [attribute, position],
    pair (L,n,n) with pair[a,b] = mu(a before b), chain (L,2^n) chain-passage probabilities,
    Edeg, entropy H. Log-sum-exp recursions (numerically stable).
    """
    u = np.atleast_2d(np.asarray(u, dtype=float))
    L = u.shape[0]
    n = int(round(math.log2(u.shape[1])))
    lat = lat or lattice(n)
    beta = np.broadcast_to(np.asarray(beta, dtype=float), (L,)).copy()
    if chunk is not None and L > chunk:
        parts = [gibbs_sofi_dp(u[i:i + chunk], beta[i:i + chunk], lat, want_pairs, want_chain)
                 for i in range(0, L, chunk)]
        return {k: np.concatenate([p[k] for p in parts], 0) for k in parts[0]}
    eta = eta_from_u(u, lat)
    logw = beta[:, None] * eta
    logw[:, 0] = 0.0
    logw[:, lat.full] = 0.0
    logF = np.full((L, lat.N), LOG_ZERO)
    logF[:, 0] = 0.0
    for k in range(1, n + 1):
        S = lat.layers[k]
        prev = lat.rm[S]                                     # (m, n), -1 where j not in S
        vals = np.where(prev[None] >= 0, logF[:, np.maximum(prev, 0)], LOG_ZERO)
        logF[:, S] = logw[:, S] + logsumexp(vals, axis=-1)
    logB = np.full((L, lat.N), LOG_ZERO)
    logB[:, lat.full] = 0.0
    for k in range(n - 1, -1, -1):
        S = lat.layers[k]
        nxt = lat.add[S]
        vals = np.where(nxt[None] >= 0,
                        logw[:, np.maximum(nxt, 0)] + logB[:, np.maximum(nxt, 0)], LOG_ZERO)
        logB[:, S] = logsumexp(vals, axis=-1)
    logZ = logB[:, 0]
    out = {"logZ": logZ, "logZ_forward": logF[:, lat.full]}
    # q(S, j) = F(S) w(S+j) B(S+j) / Z  for j not in S
    addc = np.maximum(lat.add, 0)
    logq = np.where(lat.add[None] >= 0,
                    logF[:, :, None] + logw[:, addc] + logB[:, addc] - logZ[:, None, None],
                    LOG_ZERO)
    q = np.exp(logq)                                          # (L, N, n)
    P = np.zeros((L, n, n))
    for k in range(n):
        P[:, :, k] = q[:, lat.layers[k], :].sum(1)
    out["P"] = P
    if want_pairs:
        out["pair"] = np.einsum("sa,lsb->lab", lat.mem.astype(float), q)
    chain = np.exp(logF + logB - logZ[:, None])
    if want_chain:
        out["chain"] = chain
    inter = lat.interior
    Edeg = (chain[:, inter] * eta[:, inter]).sum(1)
    out["Edeg"] = Edeg
    out["H"] = logZ - beta * Edeg
    return out


def deg_extremes_dp(u, lat=None, tol_rel=1e-9):
    """
    Max-sum and min-sum recursions with counts of optimal chains (the beta to infinity limit).
    Returns dmax, dmin, one argmax ordering (lowest attribute index tie-break), the number of
    maximizing orderings, and the hard-limit law (uniform on all maximizers): Pinf, pairinf.
    """
    u = np.atleast_2d(np.asarray(u, dtype=float))
    L = u.shape[0]
    n = int(round(math.log2(u.shape[1])))
    lat = lat or lattice(n)
    eta = eta_from_u(u, lat)
    et = eta.copy()
    et[:, 0] = 0.0
    et[:, lat.full] = 0.0
    scale = 1.0 + np.abs(et).max(1)
    tol = tol_rel * scale
    res = {}
    for mode in ("max", "min"):
        sgn = 1.0 if mode == "max" else -1.0
        e = sgn * et
        G = np.full((L, lat.N), -np.inf)
        G[:, 0] = 0.0
        cF = np.zeros((L, lat.N))
        cF[:, 0] = 1.0
        for k in range(1, n + 1):
            S = lat.layers[k]
            prev = lat.rm[S]
            vals = np.where(prev[None] >= 0, G[:, np.maximum(prev, 0)], -np.inf)
            best = vals.max(-1)
            G[:, S] = e[:, S] + best
            hit = vals >= best[..., None] - tol[:, None, None]
            cnt = np.where(hit & (prev[None] >= 0), cF[:, np.maximum(prev, 0)], 0.0)
            cF[:, S] = cnt.sum(-1)
        res[mode] = sgn * G[:, lat.full]
        if mode == "max":
            GF, CF = G, cF
    # backward max with counts
    GB = np.full((L, lat.N), -np.inf)
    GB[:, lat.full] = 0.0
    cB = np.zeros((L, lat.N))
    cB[:, lat.full] = 1.0
    for k in range(n - 1, -1, -1):
        S = lat.layers[k]
        nxt = lat.add[S]
        vals = np.where(nxt[None] >= 0, et[:, np.maximum(nxt, 0)] + GB[:, np.maximum(nxt, 0)], -np.inf)
        best = vals.max(-1)
        GB[:, S] = best
        hit = vals >= best[..., None] - tol[:, None, None]
        cB[:, S] = np.where(hit & (nxt[None] >= 0), cB[:, np.maximum(nxt, 0)], 0.0).sum(-1)
    dmax = res["max"]
    addc = np.maximum(lat.add, 0)
    through = GF[:, :, None] + et[:, addc] + GB[:, addc]
    opt = (lat.add[None] >= 0) & (through >= dmax[:, None, None] - tol[:, None, None])
    ways = np.where(opt, CF[:, :, None] * cB[:, addc], 0.0)          # (L, N, n)
    total = CF[:, lat.full]
    Pinf = np.zeros((L, n, n))
    for k in range(n):
        Pinf[:, :, k] = ways[:, lat.layers[k], :].sum(1)
    Pinf /= total[:, None, None]
    pairinf = np.einsum("sa,lsb->lab", lat.mem.astype(float), ways) / total[:, None, None]
    # lexicographically smallest optimal ordering, built forward along optimal transitions
    argmax_perm = np.zeros((L, n), dtype=int)
    for l in range(L):
        S = 0
        for k in range(n):
            for j in range(n):
                if not (S >> j) & 1 and opt[l, S, j]:
                    argmax_perm[l, k] = j
                    S |= (1 << j)
                    break
    return {"dmax": dmax, "dmin": res["min"], "n_opt": total, "argmax_perm": argmax_perm,
            "Pinf": Pinf, "pairinf": pairinf}


# ------------------------------------------------------------
# Score scale and inverse temperature
# ------------------------------------------------------------

def null_moments(u, lat=None):
    """
    Exact moments of deg under a uniformly random ordering, which give the null reference level.
    m_x(j,k) = E0[deg 1[sigma(k)=j]] from subset sums, and E0[deg^2] from nested pairs (O(3^n)).
    """
    u = np.atleast_2d(np.asarray(u, float))
    L = u.shape[0]
    n = int(round(math.log2(u.shape[1])))
    lat = lat or lattice(n)
    eta = eta_from_u(u, lat)
    m = np.zeros((L, n, n))
    for l in range(1, n):
        S = lat.layers[l]
        w = 1.0 / comb(n, l)
        memS = lat.mem[S].astype(float)                       # (m, n)
        e = eta[:, S] * w                                     # (L, m)
        in_part = np.einsum("lm,mj->lj", e, memS) / l          # attribute in S, positions k <= l
        out_part = np.einsum("lm,mj->lj", e, 1 - memS) / (n - l)
        m[:, :, :l] += in_part[:, :, None]
        m[:, :, l:] += out_part[:, :, None]
    # second moment
    E2 = np.zeros(L)
    for k in range(1, n):
        S = lat.layers[k]
        E2 += (eta[:, S] ** 2).mean(1)
    # nested pairs S subset T, 1<=|S|<|T|<=n-1, uniform over nested pairs of given sizes
    for (s, t), arr in _nested_pairs(n).items():
        E2 += 2.0 * (eta[:, arr[:, 0]] * eta[:, arr[:, 1]]).mean(1)
    return m, E2


_NESTED = {}


def _nested_pairs(n):
    if n not in _NESTED:
        lat = lattice(n)
        acc = {}
        for T in range(lat.N):
            t = lat.size[T]
            if t < 2 or t > n - 1:
                continue
            sub = (T - 1) & T
            while sub > 0:
                acc.setdefault((int(lat.size[sub]), int(t)), []).append((sub, T))
                sub = (sub - 1) & T
        _NESTED[n] = {k: np.array(v) for k, v in acc.items()}
    return _NESTED[n]


def score_scale_s(E2_train):
    """Median training-lesion standard deviation of deg under uniform rankings, with the stated fallbacks."""
    sd = np.sqrt(np.maximum(E2_train, 0.0))
    med = float(np.median(sd))
    if med > 0:
        return med, "median"
    pos = sd[sd > 0]
    if pos.size > 0:
        return float(np.median(pos)), "median_of_positive_scales_FLAG"
    return 1.0, "unit_logit_scale_FLAG_uninformative"


# ------------------------------------------------------------
# Population geometry on the permutohedron
# ------------------------------------------------------------

def borda_consensus(ybar):
    """Vertex nearest to ybar, sorting ybar decreasing; exact ties broken by attribute index and flagged."""
    ybar = np.asarray(ybar, float)
    order = np.lexsort((np.arange(len(ybar)), -ybar))
    srt = ybar[order]
    tie = bool(np.any(np.isclose(np.diff(srt), 0.0, atol=1e-12)))
    return order, tie


def population_geometry(P, weights=None):
    """Explanation points, ambiguity, between-lesion spread, certainty and the Borda consensus."""
    L, n, _ = P.shape
    w = np.full(L, 1.0 / L) if weights is None else np.asarray(weights, float) / np.sum(weights)
    rho = weyl_rho(n)
    r2 = rho_norm2(n)
    y = P @ rho
    A = 1.0 - (y ** 2).sum(1) / r2
    ybar = (w[:, None] * y).sum(0)
    kappa = float(np.sqrt((ybar ** 2).sum() / r2))
    Abar = float((w * A).sum())
    Bsp = float((w * ((y - ybar) ** 2).sum(1)).sum() / r2)
    order, tie = borda_consensus(ybar)
    ystar = vertex(order)
    return {"y": y, "A": A, "ybar": ybar, "kappa": kappa, "Abar": Abar, "B": Bsp,
            "decomp_residual": (1 - kappa ** 2) - (Abar + Bsp), "sigma_star": order,
            "borda_tie": tie, "ystar": ystar, "weights": w}


# ------------------------------------------------------------
# Lift of a point of the permutohedron to a symmetric matrix with the prescribed spectrum
# ------------------------------------------------------------

def schur_horn_lift(y, lam, tol=1e-12):
    """
    Construct O in SO(n) with diag(O diag(lam) O^T) = y (y majorized by lam), through
    T-transforms (Marshall-Olkin 2.B.1) realized as Givens rotations, then a permutation.
    """
    y = np.asarray(y, float)
    lam = np.asarray(lam, float)
    n = len(y)
    X = np.diag(np.sort(lam)[::-1])
    O = np.eye(n)
    target = np.sort(y)[::-1]
    for _ in range(4 * n):
        d = np.diag(X).copy()
        ordr = np.argsort(-d, kind="stable")
        ds = d[ordr]
        diff = ds - target
        if np.max(np.abs(diff)) < tol:
            break
        jpos = np.where(diff > tol)[0]
        if len(jpos) == 0:
            break
        j = jpos.max()
        kc = np.where((np.arange(n) > j) & (diff < -tol))[0]
        if len(kc) == 0:
            break
        k = kc.min()
        delta = min(ds[j] - target[j], target[k] - ds[k])
        a, b = ordr[j], ordr[k]
        goal = X[a, a] - delta

        def f(t):
            c, s = np.cos(t), np.sin(t)
            return c * c * X[a, a] + s * s * X[b, b] + 2 * c * s * X[a, b] - goal
        lo_t, hi_t = 0.0, np.pi / 2
        flo = f(lo_t)
        for _ in range(200):
            mid = 0.5 * (lo_t + hi_t)
            if np.sign(f(mid)) == np.sign(flo):
                lo_t, flo = mid, f(mid)
            else:
                hi_t = mid
        t = 0.5 * (lo_t + hi_t)
        G = np.eye(n)
        c, s = np.cos(t), np.sin(t)
        G[a, a] = c; G[b, b] = c; G[a, b] = -s; G[b, a] = s
        # new row a of X' = G^T X G has (a,a) entry c^2 Xaa + s^2 Xbb + 2cs Xab
        X = G.T @ X @ G
        O = G.T @ O
    d = np.diag(X)
    # permutation matching the actual coordinates of y
    pd = np.argsort(-d, kind="stable")
    py = np.argsort(-y, kind="stable")
    Q = np.zeros((n, n))
    Q[py, pd] = 1.0
    X = Q @ X @ Q.T
    O = Q @ O
    # O maps diag(sorted lam) -> X with X = O diag(sorted lam) O^T
    if np.linalg.det(O) < 0:
        D = np.eye(n)
        D[0, 0] = -1.0
        O = D @ O
        X = D @ X @ D
    return O, X


def orthostochastic(O):
    return O ** 2


# ------------------------------------------------------------
# Pairwise majorities, tournaments and the Kemeny order
# ------------------------------------------------------------

def pooled_pairs(pair, weights=None):
    L = pair.shape[0]
    w = np.full(L, 1.0 / L) if weights is None else np.asarray(weights, float) / np.sum(weights)
    return np.einsum("l,lab->ab", w, pair)


def three_cycles(M, tol=1e-9):
    """Directed 3-cycles of the strict majority tournament; shares within tol of 1/2 are ties, not majorities."""
    n = M.shape[0]
    cyc = []
    h = 0.5 + tol
    for a, b, c in itertools.combinations(range(n), 3):
        if M[a, b] > h and M[b, c] > h and M[c, a] > h:
            cyc.append((a, b, c))
        if M[a, c] > h and M[c, b] > h and M[b, a] > h:
            cyc.append((a, c, b))
    return cyc


def kemeny_dp(M):
    """Exact Kemeny consensus by subset DP: cost sums M[b,a] over pairs ordered a before b. Returns order, cost, number of optima."""
    n = M.shape[0]
    lat = lattice(n)
    K = np.full(lat.N, np.inf)
    K[0] = 0.0
    cnt = np.zeros(lat.N)
    cnt[0] = 1
    for k in range(1, n + 1):
        for S in lat.layers[k]:
            best = np.inf
            c = 0.0
            for j in range(n):
                if (S >> j) & 1:
                    Sp = S ^ (1 << j)
                    others = [i for i in range(n) if (Sp >> i) & 1]
                    cost = K[Sp] + sum(M[j, i] for i in others)
                    if cost < best - 1e-12:
                        best, c = cost, cnt[Sp]
                    elif abs(cost - best) <= 1e-12:
                        c += cnt[Sp]
            K[S], cnt[S] = best, c
    # backtrack (lowest index tie-break)
    S = lat.full
    order = []
    for _ in range(n):
        for j in range(n):
            if (S >> j) & 1:
                Sp = S ^ (1 << j)
                others = [i for i in range(n) if (Sp >> i) & 1]
                if abs(K[Sp] + sum(M[j, i] for i in others) - K[S]) <= 1e-9:
                    order.append(j)
                    S = Sp
                    break
    return np.array(order[::-1]), float(K[lat.full]), int(cnt[lat.full])


def kendall_distance(p1, p2):
    n = len(p1)
    pos1 = np.empty(n, int); pos1[np.asarray(p1)] = np.arange(n)
    pos2 = np.empty(n, int); pos2[np.asarray(p2)] = np.arange(n)
    d = 0
    for a in range(n):
        for b in range(a + 1, n):
            if (pos1[a] - pos1[b]) * (pos2[a] - pos2[b]) < 0:
                d += 1
    return d


def kendall_tau(p1, p2):
    n = len(p1)
    return 1.0 - 4.0 * kendall_distance(p1, p2) / (n * (n - 1))


def rbo(p1, p2, p=0.9):
    """Extrapolated rank-biased overlap of two complete rankings (Webber et al.)."""
    k = len(p1)
    s1, s2 = set(), set()
    acc = 0.0
    X = 0
    for d in range(1, k + 1):
        s1.add(p1[d - 1]); s2.add(p2[d - 1])
        X = len(s1 & s2)
        acc += (X / d) * p ** d
    return (1 - p) / p * acc + (X / k) * p ** k


# ------------------------------------------------------------
# Exchange coordinates
# ------------------------------------------------------------

def pair_list(n):
    return [(a, b) for a in range(n) for b in range(a + 1, n)]


def exchange_coordinates(P, sigma_star):
    """omega_x(a,b) = (R_x(a,b)+R_x(b,a))/2 with R_x = P_x P_sigma*^T, pairs a<b."""
    n = P.shape[-1]
    Ps = perm_matrix(sigma_star)
    R = P @ Ps.T
    pl = pair_list(n)
    om = np.stack([0.5 * (R[:, a, b] + R[:, b, a]) for a, b in pl], axis=1)
    pos = np.empty(n, int)
    pos[np.asarray(sigma_star)] = np.arange(n)
    Delta = np.array([pos[a] - pos[b] for a, b in pl], float)
    return om, Delta, R


def exchange_report(omega, Delta, weights=None, max_components=5):
    """Pair shares of the expected squared distance to the reference, with the principal subspaces."""
    L, m2 = omega.shape
    w = np.full(L, 1.0 / L) if weights is None else np.asarray(weights, float) / np.sum(weights)
    ombar = (w[:, None] * omega).sum(0)
    num = ombar * Delta ** 2
    den = num.sum()
    shares = num / den if den > 0 else np.full(m2, np.nan)
    C = omega - ombar
    Sigma = (w[:, None, None] * np.einsum("li,lj->lij", C, C)).sum(0)
    evals, evecs = np.linalg.eigh(Sigma)
    order = np.argsort(evals)[::-1]
    evals, evecs = evals[order], evecs[:, order]
    tr = float(np.trace(Sigma))
    vshare = evals / tr if tr > 0 else np.full(m2, np.nan)
    # sign convention: largest absolute loading positive
    for i in range(m2):
        k = np.argmax(np.abs(evecs[:, i]))
        if evecs[k, i] < 0:
            evecs[:, i] *= -1
    npos = int(np.sum(evals > 1e-12 * max(1.0, tr)))
    dmax = min(max_components, m2, npos)
    return {"ombar": ombar, "pair_shares": shares, "Sigma": Sigma, "evals": evals, "evecs": evecs,
            "var_shares": vshare, "trace": tr, "n_components": dmax}


def so_basis(n, a, b):
    Y = np.zeros((n, n))
    Y[a, b] = 1.0
    Y[b, a] = -1.0
    return Y


def J_sigma_star(v, sigma_star):
    """Reference-oriented isometry of pair coordinates into so(n)."""
    n = len(sigma_star)
    pos = np.empty(n, int)
    pos[np.asarray(sigma_star)] = np.arange(n)
    X = np.zeros((n, n))
    for i, (a, b) in enumerate(pair_list(n)):
        if pos[a] < pos[b]:
            X += v[i] * so_basis(n, a, b)
        else:
            X += v[i] * so_basis(n, b, a)
    return X


def xi_pm(v, sigma_star):
    """Oriented square-root realizations of the nonnegative parts of a loading vector."""
    vp, vm = np.maximum(v, 0), np.maximum(-v, 0)
    return J_sigma_star(np.sqrt(vp), sigma_star), J_sigma_star(np.sqrt(vm), sigma_star)


def W_of_xi(xi):
    return xi ** 2 * (1 - np.eye(xi.shape[0]))


# ------------------------------------------------------------
# Ordered tiers as faces of the permutohedron
# ------------------------------------------------------------

def components(n, edges):
    parent = list(range(n))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    groups = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)
    return list(groups.values())


def lie_closure_dim(n, edges, tol=1e-9):
    """Dimension of the Lie algebra generated by the rotation generators of the graph edges."""
    if not edges:
        return 0
    basis = []

    def add(M):
        v = M.ravel()
        if basis:
            B = np.array(basis)
            r = v - B.T @ np.linalg.lstsq(B.T, v, rcond=None)[0]
        else:
            r = v
        if np.linalg.norm(r) > tol:
            basis.append(v.copy())
            return True
        return False
    mats = []
    for a, b in edges:
        Y = so_basis(n, a, b)
        if add(Y):
            mats.append(Y)
    changed = True
    while changed:
        changed = False
        cur = list(mats)
        for X in cur:
            for Y in cur:
                Z = X @ Y - Y @ X
                if add(Z):
                    mats.append(Z)
                    changed = True
    return len(basis)


def tier_face(ombar, sigma_star, tau):
    """Exchange graph, its components, the closure into runs of consecutive positions and the tier face."""
    n = len(sigma_star)
    pl = pair_list(n)
    edges = [pl[i] for i in range(len(pl)) if ombar[i] > tau]
    comps = components(n, edges)
    pos = np.empty(n, int)
    pos[np.asarray(sigma_star)] = np.arange(n)
    link = np.zeros(n - 1, bool)                          # link[k]: positions k and k+1 (0-based) are joined
    for B in comps:
        if len(B) < 2:
            continue
        lo, hi = min(pos[B]), max(pos[B])
        link[lo:hi] = True
    tiers = []
    cur = [sigma_star[0]]
    for k in range(n - 1):
        if link[k]:
            cur.append(sigma_star[k + 1])
        else:
            tiers.append(cur)
            cur = [sigma_star[k + 1]]
    tiers.append(cur)
    consecutive = all((max(pos[B]) - min(pos[B]) + 1) == len(B) for B in comps)
    return {"edges": edges, "components": comps, "tiers": tiers, "q": len(tiers),
            "dim_face": n - len(tiers), "dim_Q": n - len(comps), "Q_equals_face": consecutive}


def crossing_mass(M, tiers):
    X = 0.0
    npairs = 0
    for i in range(len(tiers)):
        for j in range(i + 1, len(tiers)):
            for a in tiers[i]:
                for b in tiers[j]:
                    X += M[b, a]
                    npairs += 1
    share = X / npairs if npairs > 0 else np.nan
    return X, npairs, share


def tier_labels(tiers, n):
    lab = np.zeros(n, int)
    for i, T in enumerate(tiers):
        lab[list(T)] = i
    return lab


# ------------------------------------------------------------
# Operational confidence rule
# ------------------------------------------------------------

def operational_certificates(ybar, M, ombar, tau, N, n, delta=0.05, extra_y=0.0, extra_p=0.0):
    m2 = n * (n - 1) // 2
    r_y = (n - 1) * math.sqrt(math.log(6 * n / delta) / (2 * N)) + extra_y
    r_p = math.sqrt(math.log(6 * m2 / delta) / (2 * N)) + extra_p
    srt = np.sort(ybar)[::-1]
    gaps = -np.diff(srt)
    order_cert = bool(np.all(gaps > 2 * r_y))
    pl = pair_list(n)
    wall_cert = np.array([abs(M[a, b] - 0.5) > r_p for a, b in pl])
    edge_cert = np.abs(ombar - tau) > r_p
    graph_cert = bool(order_cert and np.all(edge_cert))
    return {"r_y": r_y, "r_p": r_p, "order_certified": order_cert, "min_adjacent_gap": float(gaps.min()),
            "wall_certified": wall_cert, "edge_certified": edge_cert if order_cert else np.zeros_like(edge_cert),
            "graph_and_face_certified": graph_cert}


def pca_certificate(evals, N, n, delta=0.05, d_max=5):
    m2 = n * (n - 1) // 2
    eps = 3 * m2 * math.sqrt(math.log(4 * m2 ** 2 / delta) / (2 * N))
    rows = []
    for d in range(1, min(d_max, m2 - 1) + 1):
        g = evals[d - 1] - evals[d]
        lower = g - 2 * eps
        rows.append({"d": d, "empirical_gap": g, "eps_Sigma": eps, "gap_lower_bound": lower,
                     "subspace_sin_theta_bound": (2 * eps / lower) if lower > 0 else np.nan,
                     "certified": bool(lower > 0)})
    return rows


def hoeffding_radius(a, b, N, delta):
    return (b - a) * math.sqrt(math.log(2 / delta) / (2 * N))

# ------------------------------------------------------------
# Additions for the case study
# ------------------------------------------------------------

def removal_curves(u, perms, lat):
    """
    MoRF and LeRF curves with their unit-step trapezoid areas, for one ranking per game.
    u (L, 2^n), perms (L, n). MoRF records u(S_k), LeRF records u(C minus S_{n-k}), k = 0..n.
    The degradation score lerf - morf is the SOFI score deg of Lemma 1.
    """
    u = np.atleast_2d(np.asarray(u, float))
    perms = np.atleast_2d(np.asarray(perms, int))
    L, n = perms.shape
    bits = (1 << perms)
    pref = np.concatenate([np.zeros((L, 1), int), np.cumsum(bits, axis=1)], axis=1)     # S_0..S_n
    morf_v = np.take_along_axis(u, pref, 1)
    lerf_v = np.take_along_axis(u, lat.full ^ pref[:, ::-1], 1)
    morf = 0.5 * (morf_v[:, :-1] + morf_v[:, 1:]).sum(1)
    lerf = 0.5 * (lerf_v[:, :-1] + lerf_v[:, 1:]).sum(1)
    return morf, lerf, lerf - morf


def null_curve_area(u, lat):
    """Expected MoRF area (equal to the expected LeRF area) under a uniformly random ordering."""
    u = np.atleast_2d(np.asarray(u, float))
    n = lat.n
    mk = np.stack([u[:, lat.layers[k]].mean(1) for k in range(n + 1)], 1)
    return 0.5 * mk[:, 0] + mk[:, 1:n].sum(1) + 0.5 * mk[:, n]


_SHAPLEY_G = {}


def shapley_values(u, lat):
    """Exact Shapley values of the presence game val(K) = u(C minus K), vectorized over games."""
    n = lat.n
    if n not in _SHAPLEY_G:
        G = np.zeros((lat.N, n))
        fact = [math.factorial(i) for i in range(n + 1)]
        for S in range(lat.N):
            s = int(lat.size[S])
            for j in range(n):
                if not (S >> j) & 1:
                    w = fact[s] * fact[n - s - 1] / fact[n]
                    G[S | (1 << j), j] += w
                    G[S, j] -= w
        _SHAPLEY_G[n] = G
    val = np.atleast_2d(u)[:, lat.comp]
    return val @ _SHAPLEY_G[n]


def additive_fit(u, lat):
    """
    Least-squares fit u(A) = u0 - sum_{j in A} c_j over all 2^n subsets. On the full subset design the
    coefficient c_j equals the average drop of the score when attribute j is removed (its Banzhaf
    removal effect). Returns c (L, n) with R^2 of the fit per game.
    """
    u = np.atleast_2d(np.asarray(u, float))
    X = np.concatenate([np.ones((lat.N, 1)), -lat.mem.astype(float)], 1)
    coef = np.linalg.lstsq(X, u.T, rcond=None)[0]                  # (n+1, L)
    fit = (X @ coef).T
    ss_res = ((u - fit) ** 2).sum(1)
    ss_tot = ((u - u.mean(1, keepdims=True)) ** 2).sum(1)
    r2 = np.where(ss_tot > 0, 1 - ss_res / np.where(ss_tot > 0, ss_tot, 1.0), np.nan)
    return coef[1:].T, r2


def sort_desc(v):
    """Ordering that sorts v in decreasing order, lowest attribute index on exact ties."""
    v = np.asarray(v, float)
    return np.lexsort((np.arange(len(v)), -v))


def positions_of(perms):
    """pos[l, j] = position (0-based) of attribute j in ranking l."""
    perms = np.atleast_2d(perms)
    return np.argsort(perms, axis=1)


def pair_matrix_of_rankings(perms):
    """Share of rankings that place a before b, for a collection of complete rankings."""
    pos = positions_of(perms)
    return (pos[:, :, None] < pos[:, None, :]).astype(float).mean(0)


def order_str(sig, names=None):
    names = names or CFG.CONCEPT_ABBREV
    return " > ".join(names[j] for j in sig)


def transitive_closure(E):
    """Boolean transitive closure of a relation given as an n x n boolean matrix."""
    R = np.array(E, bool)
    n = R.shape[0]
    for k in range(n):
        R = R | (R[:, [k]] & R[[k], :])
    return R


def hasse_edges(R):
    """Covering pairs of a strict partial order R (transitive reduction)."""
    n = R.shape[0]
    return [(a, b) for a in range(n) for b in range(n)
            if R[a, b] and not any(R[a, c] and R[c, b] for c in range(n) if c not in (a, b))]


def order_levels(R):
    """Level of each element: 0 for elements with nothing certified above them, then longest-path depth."""
    n = R.shape[0]
    lev = np.zeros(n, int)
    for _ in range(n):
        for b in range(n):
            above = [a for a in range(n) if R[a, b]]
            if above:
                lev[b] = max(lev[b], 1 + max(lev[a] for a in above))
    return lev

## Removal games and local ranking laws

A float64 mirror of the trained heads evaluates the removal game of each lesion on all 128 subsets of concepts. The local ranking law of a lesion is the Gibbs law of its SOFI scores on a shared scale, so the laws of different lesions are comparable.

In [9]:
# ============================================================
# REMOVAL GAMES AND LOCAL RANKING LAWS
# Game values are computed in float64 from a numpy mirror of the trained heads. The melanoma logit is
# s = logit_1 - logit_0. A population explains the logit of its own class, so the game of a
# non-melanoma lesion is -u for the melanoma game u (zero removal of the predicted concept blocks).
# ============================================================

class NumpyCBM:
    """Float64 mirror of the concept heads and of the diagnosis head."""

    def __init__(self, model=None, concept_info=None, cbm_type=None):
        if model is None:
            return
        self.cbm_type = cbm_type or model.cbm_type
        self.sizes = [concept_info[c]["n_classes"] for c in CFG.CONCEPT_COLUMNS]
        self.offsets = np.concatenate([[0], np.cumsum(self.sizes)]).astype(int)
        self.Wc = [model.concept_heads[c].weight.detach().double().cpu().numpy() for c in CFG.CONCEPT_COLUMNS]
        self.bc = [model.concept_heads[c].bias.detach().double().cpu().numpy() for c in CFG.CONCEPT_COLUMNS]
        cl = model.classifier
        self.W1 = cl[0].weight.detach().double().cpu().numpy()
        self.b1 = cl[0].bias.detach().double().cpu().numpy()
        self.g = cl[1].weight.detach().double().cpu().numpy()
        self.be = cl[1].bias.detach().double().cpu().numpy()
        self.ln_eps = cl[1].eps
        self.W2 = cl[4].weight.detach().double().cpu().numpy()
        self.b2 = cl[4].bias.detach().double().cpu().numpy()

    def concept_probs(self, z):
        out = []
        for W, b in zip(self.Wc, self.bc):
            lg = z @ W.T + b
            e = np.exp(lg - lg.max(1, keepdims=True))
            out.append(e / e.sum(1, keepdims=True))
        return out

    def bottleneck(self, z):
        blocks = []
        for p, k in zip(self.concept_probs(z), self.sizes):
            blocks.append(np.eye(k)[p.argmax(1)] if self.cbm_type == "hard" else p)
        return np.concatenate(blocks, 1)

    def logit(self, v):
        """Melanoma logit s(v) = logit_1 - logit_0 for head inputs v (..., M)."""
        h = v @ self.W1.T + self.b1
        mu = h.mean(-1, keepdims=True)
        var = h.var(-1, keepdims=True)
        h = (h - mu) / np.sqrt(var + self.ln_eps) * self.g + self.be
        h = np.maximum(h, 0.0)
        o = h @ self.W2.T + self.b2
        return o[..., 1] - o[..., 0]


def zero_removal_game(ncbm, V):
    """u_x(A) = s(v with every block of A set to zero), for all 2^n subsets A (melanoma orientation)."""
    lat = lattice(N_ATTR)
    L, M = V.shape
    blockmask = np.zeros((lat.N, M), bool)
    for j in range(N_ATTR):
        blockmask[lat.mem[:, j], ncbm.offsets[j]:ncbm.offsets[j + 1]] = True
    X = np.where(blockmask[None], 0.0, V[:, None, :])
    return ncbm.logit(X)


def predicted_states(ncbm, V):
    """Index of the predicted state of every attribute, read from the one-hot blocks of the bottleneck."""
    return np.stack([V[:, ncbm.offsets[j]:ncbm.offsets[j + 1]].argmax(1) for j in range(N_ATTR)], 1)


def local_bank(u, beta):
    """Every local Gibbs-SOFI object for a batch of games at inverse temperature beta."""
    lat = lattice(N_ATTR)
    dp = gibbs_sofi_dp(u, beta, lat)
    ex = deg_extremes_dp(u, lat)
    return {"P": dp["P"], "pair": dp["pair"], "H": dp["H"], "logZ": dp["logZ"], "Edeg": dp["Edeg"],
            "y": dp["P"] @ weyl_rho(N_ATTR), "h": dp["H"] / math.lgamma(N_ATTR + 1),
            "dmax": ex["dmax"], "dmin": ex["dmin"], "n_opt": ex["n_opt"], "argmax_perm": ex["argmax_perm"],
            "Pinf": ex["Pinf"], "pairinf": ex["pairinf"]}

## Models, populations, tuned hyperparameters and the selected model

For every seed, the correctly classified validation lesions tune the inverse temperature beta0 (split-half agreement over 11 values) as well as the tier threshold tau of each population (bootstrap stability over 12 values). Both choices are frozen in `frozen_choices.json`, while the full tuning curves are printed and saved. The test lesions take no part in any choice.

The model studied in the first and third experiments is the seed with the largest test F1. This choice is internal to the illustration. Its only advantage is a larger number of correctly classified test images to explain. The last table checks the algebraic identities of the paper on the real explanations, where every residual should be at rounding level.

In [ ]:
# ============================================================
# TRAINED MODELS, POPULATIONS AND FROZEN ANALYSIS CHOICES
# The trained models of Part 1 are reloaded from their checkpoints. The training split is used by no
# computation of Part 2. For every seed, the correctly classified validation lesions tune the inverse
# temperature and the tier thresholds. The correctly classified test lesions of one class form one
# population. The tuning curves are cached, so a rerun prints and saves them without recomputing.
# ============================================================
import pickle
import gc

LAT = lattice(N_ATTR)
PERMS = all_perms(N_ATTR)
IU = np.triu_indices(N_ATTR, 1)
PAIRS = pair_list(N_ATTR)
M2 = len(PAIRS)
RHO = weyl_rho(N_ATTR)
R2 = rho_norm2(N_ATTR)
POP_SIGN = {p: (1.0 if c == 1 else -1.0) for p, c in CFG.POPULATIONS.items()}


def apg_population(P, pair, tau, weights=None, sigma_ref=None, with_kemeny=True):
    """Every population quantity of Theorems 3, 4, 6 and 7 for one collection of local ranking laws."""
    n = P.shape[-1]
    geo = population_geometry(P, weights)
    sigma = geo["sigma_star"] if sigma_ref is None else np.asarray(sigma_ref)
    M = pooled_pairs(pair, geo["weights"])
    om, Delta, R = exchange_coordinates(P, sigma)
    exr = exchange_report(om, Delta, geo["weights"], CFG.MAX_PCA_COMPONENTS)
    tf = tier_face(exr["ombar"], sigma, tau if np.isfinite(tau) else -np.inf)
    X, npairs, share = crossing_mass(M, tf["tiers"])
    kem = kemeny_dp(M) if with_kemeny else (np.full(n, -1), np.nan, 0)
    ystar = vertex(sigma)
    dist = 2 * (exr["ombar"] * Delta ** 2).sum()
    return {"geo": geo, "sigma": sigma, "M": M, "omega": om, "Delta": Delta, "R": R, "exch": exr, "tiers": tf,
            "crossing": (X, npairs, share), "kemeny": kem, "cycles": three_cycles(M),
            "pair_allocation_residual": dist - (rho_norm2(n) * (1 - geo["kappa"] ** 2) + ((geo["ybar"] - ystar) ** 2).sum()),
            "N": P.shape[0]}


def _pkl_save(obj, path):
    with open(path, "wb") as fh:
        pickle.dump(obj, fh, protocol=4)


def _pkl_load(path):
    with open(path, "rb") as fh:
        return pickle.load(fh)


def load_trained_model(seed):
    entry = MANIFEST["seeds"][str(seed)]
    ck = torch.load(entry["checkpoint"], map_location=CFG.DEVICE, weights_only=False)
    model = ConceptBottleneckModel(ck["backbone"], CONCEPT_INFO, CFG.CBM_TYPE).to(CFG.DEVICE)
    model.load_state_dict(ck["model_state_dict"])
    model.eval()
    return model, ck


def recorded_predictions(seed, run):
    """Predictions recorded by Part 1, checked row by row against the splits and the float64 mirror."""
    rec = pd.read_csv(MANIFEST["seeds"][str(seed)]["predictions"])
    out = {}
    for split in ("valid", "test"):
        r = rec[rec["split"] == split].sort_values("position").reset_index(drop=True)
        f = run["feats"][split]
        same = len(r) == len(f["derm"]) and np.array_equal(r["derm"].astype(str).str.lower().values,
                                                           pd.Series(f["derm"]).astype(str).str.lower().values)
        if not same:
            raise RuntimeError(f"The {split} split of Part 2 does not match the predictions recorded by Part 1.")
        mirror = (run["ncbm"].logit(run["ncbm"].bottleneck(f["z"])) > 0).astype(int)
        out[split] = {"correct": r["correct"].values.astype(bool), "y_pred": r["y_pred"].values,
                      "mirror_agreement": float((mirror == r["y_pred"].values).mean())}
    return out


def get_run(seed):
    """Reloaded model of one seed with its pooled features (cached per checkpoint fingerprint)."""
    tag = f"{CFG.CBM_TYPE}_{CFG.BACKBONE}_{CFG.FILTER_STRATEGY}_seed{seed}"
    cpath = os.path.join(CFG.CACHE_DIR, f"{tag}_{model_stamp(seed)}.pkl")
    if os.path.exists(cpath):
        say(f"Loading cached features of {tag}")
        return _pkl_load(cpath)
    t0 = time.time()
    model, ck = load_trained_model(seed)
    mets = pd.DataFrame([{"seed": seed, "split": s, **ck["split_results"][s]["lm"],
                          "concept_acc": ck["split_results"][s]["c_acc"], "concept_f1": ck["split_results"][s]["c_f1"],
                          "best_epoch": ck["best_epoch"]} for s in ("train", "valid", "test")])
    feats = extract_features(model, SPLITS, LABEL_ENCODERS, CONCEPT_INFO)
    run = {"tag": tag, "seed": seed, "ncbm": NumpyCBM(model, CONCEPT_INFO), "feats": feats, "metrics": mets}
    with torch.no_grad():
        x = torch.tensor(feats["test"]["z"][:32], dtype=torch.float32, device=CFG.DEVICE)
        lg = model.classifier(model.bottleneck(model.concept_logits_from_features(x)))
        s_t = (lg[:, 1] - lg[:, 0]).double().cpu().numpy()
    run["mirror_max_abs_error"] = float(np.abs(s_t - run["ncbm"].logit(run["ncbm"].bottleneck(feats["test"]["z"][:32]))).max())
    run["recorded"] = recorded_predictions(seed, run)
    run["seconds"] = time.time() - t0
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    _pkl_save(run, cpath)
    return run


def explain_base(run):
    """Melanoma-orientation games under zero removal, bottleneck inputs and full logits, per split."""
    out = {"U": {}, "V": {}, "s_full": {}}
    for s in ("valid", "test"):
        V = run["ncbm"].bottleneck(run["feats"][s]["z"])
        out["V"][s] = V
        out["U"][s] = zero_removal_game(run["ncbm"], V)
        out["s_full"][s] = run["ncbm"].logit(V)
    return out


# ---------------- choices tuned on the validation lesions, then frozen ----------------

def frozen_choices():
    if os.path.exists(CFG.FROZEN_FILE):
        with open(CFG.FROZEN_FILE) as fh:
            return json.load(fh)
    return {}


def seed_key(seed):
    return f"seed{seed}_{model_stamp(seed)}"


def cached(name, fn):
    """Result of fn() stored once in CACHE_DIR under name."""
    path = os.path.join(CFG.CACHE_DIR, f"{name}.pkl")
    if os.path.exists(path):
        return _pkl_load(path)
    out = fn()
    _pkl_save(out, path)
    return out


def save_frozen(**kw):
    d = frozen_choices()
    for k, v in kw.items():
        d.setdefault(k, v)
    with open(CFG.FROZEN_FILE, "w") as fh:
        json.dump(d, fh, indent=1)
    return d


def shared_scale(run, base):
    """Median null standard deviation of the score over every correctly classified validation lesion."""
    ok = run["recorded"]["valid"]["correct"]
    if ok.sum() < 2:
        ok = np.ones_like(ok)
    _, E2 = null_moments(base["U"]["valid"][ok], LAT)
    return score_scale_s(E2)


def _patient_halves(case, seed):
    cases = np.unique(case)
    first = set(np.random.default_rng(seed).permutation(cases)[: len(cases) // 2])
    return np.array([c in first for c in case])


def choose_beta0(run, base, seed, s):
    """
    Dimensionless inverse temperature beta0, chosen on the correctly classified validation lesions of both
    populations. The patients of each population are split in two halves. The pairwise shares M of one
    half are compared with the local pair probabilities p of the other half, both ways, through
    1 - sum (M - p)^2 / sum (1/2 - p)^2. The largest mean agreement wins, ties go to the smaller beta0.
    """
    key = f"beta0_{seed_key(seed)}"
    grid = tuple(CFG.BETA0_GRID)
    df = cached(f"tuning_beta0_{seed_key(seed)}_{abs(hash(grid)) % 10**8}", lambda: _beta0_curve(run, base, seed, s))
    fr = frozen_choices()
    if key in fr:
        beta0 = fr[key]
    elif df["split_half_agreement"].notna().any():
        best = df["split_half_agreement"].round(6).max()
        beta0 = float(df[df["split_half_agreement"].round(6) == best]["beta0"].min())
    else:
        beta0 = 1.0
        say("Too few correctly classified validation patients to choose beta0. The default beta0 = 1 is used.")
    df = df.copy()
    df["chosen"] = np.isclose(df["beta0"], beta0)
    save_frozen(**{key: beta0})
    return beta0, df


def _beta0_curve(run, base, seed, s):
    ok = run["recorded"]["valid"]["correct"]
    u_all = base["U"]["valid"][ok]
    y, case = run["feats"]["valid"]["y"][ok], run["feats"]["valid"]["case"][ok]
    groups = [(POP_SIGN[p] * u_all[y == c], case[y == c]) for p, c in CFG.POPULATIONS.items()]
    rows = []
    for b0 in CFG.BETA0_GRID:
        agree = []
        for u, cs in groups:
            half = _patient_halves(cs, seed)
            if half.sum() < 2 or (~half).sum() < 2:
                continue
            for A, B in ((half, ~half), (~half, half)):
                M = pooled_pairs(gibbs_sofi_dp(u[A], b0 / s, want_chain=False)["pair"])
                p = gibbs_sofi_dp(u[B], b0 / s, want_chain=False)["pair"][:, IU[0], IU[1]]
                q = np.broadcast_to(M[IU], p.shape)
                agree.append(1 - ((q - p) ** 2).sum() / max(((0.5 - p) ** 2).sum(), 1e-300))
        rows.append({"seed": seed, "beta0": b0, "beta_effective": b0 / s,
                     "split_half_agreement": np.mean(agree) if agree else np.nan,
                     "split_half_sd": np.std(agree) if agree else np.nan, "comparisons": len(agree)})
    return pd.DataFrame(rows)


def select_tau(P, pair, rng):
    """Tier threshold chosen on the grid by bootstrap stability (mean ARI to the full-sample tiers)."""
    L = P.shape[0]
    full = {tau: tier_labels(apg_population(P, pair, tau, with_kemeny=False)["tiers"]["tiers"], N_ATTR)
            for tau in CFG.TAU_GRID}
    boots = [rng.integers(0, L, L) for _ in range(CFG.TAU_BOOTSTRAP)]
    rows = []
    for tau in CFG.TAU_GRID:
        aris = [adjusted_rand_score(full[tau], tier_labels(apg_population(P[ii], pair[ii], tau, with_kemeny=False)
                                                           ["tiers"]["tiers"], N_ATTR)) for ii in boots]
        rows.append({"tau": tau, "stability_mean_ARI": float(np.mean(aris)), "stability_sd": float(np.std(aris)),
                     "n_tiers_full_sample": int(full[tau].max() + 1)})
    df = pd.DataFrame(rows)
    df["stability_rounded"] = df["stability_mean_ARI"].round(CFG.TAU_TIE_DECIMALS)
    df["nontrivial"] = (df["n_tiers_full_sample"] > 1) & (df["n_tiers_full_sample"] < N_ATTR)
    cand = df[df["nontrivial"]]
    if len(cand) == 0:
        df["chosen"] = False
        return np.nan, df
    top = cand[cand["stability_rounded"] == cand["stability_rounded"].max()]
    chosen = float(top.sort_values(["n_tiers_full_sample", "tau"], ascending=[False, True]).iloc[0]["tau"])
    df["chosen"] = df["tau"] == chosen
    return chosen, df


def population_data(run, base, pop, beta0, s):
    """Oriented games, local ranking laws, predicted concept states and labels of one population."""
    cls, sign = CFG.POPULATIONS[pop], POP_SIGN[pop]
    data = {}
    for split in ("valid", "test"):
        keep = run["recorded"][split]["correct"] & (run["feats"][split]["y"] == cls)
        u = sign * base["U"][split][keep]
        _, E2 = null_moments(u, LAT) if len(u) else (None, np.zeros(0))
        data[split] = {"u": u, "bank": local_bank(u, beta0 / s) if len(u) else None,
                       "sd0": np.sqrt(np.maximum(E2, 0.0)), "logit": sign * base["s_full"][split][keep],
                       "states": predicted_states(run["ncbm"], base["V"][split][keep]),
                       "true_states": run["feats"][split]["concepts"][keep],
                       "derm": run["feats"][split]["derm"][keep], "case": run["feats"][split]["case"][keep],
                       "N": int(keep.sum())}
    return {"population": pop, "label": CFG.POPULATION_LABEL[pop], "sign": sign, "seed": run["seed"],
            "beta0": beta0, "scale": s, "beta": beta0 / s, "data": data, "tau": np.nan}


def freeze_tau(PD):
    """Tier threshold of a population, chosen on its own validation lesions for its own seed, then frozen."""
    sk = seed_key(PD["seed"])
    key = f"tau_{PD['population']}_{sk}"
    b = PD["data"]["valid"]["bank"]
    name = f"tuning_tau_{PD['population']}_{sk}_{PD['beta0']}_{abs(hash(tuple(CFG.TAU_GRID))) % 10**8}"
    tau, df = cached(name, lambda: select_tau(b["P"], b["pair"], np.random.default_rng(PD["seed"])))
    fr = frozen_choices()
    if key in fr:
        tau = fr[key] if fr[key] is not None else np.nan
    save_frozen(**{key: (None if not np.isfinite(tau) else float(tau))})
    PD["tau"] = tau
    df = df.copy()
    df["chosen"] = np.isclose(df["tau"], tau) if np.isfinite(tau) else False
    df.insert(0, "population", PD["population"])
    df.insert(0, "seed", PD["seed"])
    return df

In [11]:
banner("MODELS, POPULATIONS AND ANALYSIS CHOICES TUNED ON THE VALIDATION IMAGES OF EACH SEED")
CFG.ACTIVE_IMAGES_DIR = resolve_images_dir()
meta, SPLITS, CONCEPT_INFO, LABEL_ENCODERS, INCONSISTENT_KEYS = load_data()
show(pd.DataFrame([{"split": s, "images": len(SPLITS[s]), "melanoma": int((SPLITS[s][CFG.TARGET_COLUMN] == 1).sum()),
                    "non_melanoma": int((SPLITS[s][CFG.TARGET_COLUMN] == 0).sum()),
                    "patients": len(set(patient_ids(SPLITS[s])))} for s in ("train", "valid", "test")]),
     f"Filtered splits ({CFG.FILTER_STRATEGY} filter, {len(INCONSISTENT_KEYS)} inconsistent concept profiles)", "split_summary")
show(pd.DataFrame([{"attribute": c, "n_states": CONCEPT_INFO[c]["n_classes"], "states": "; ".join(CONCEPT_INFO[c]["states"])}
                   for c in CFG.CONCEPT_COLUMNS]), "Concept state sets", "concept_states")

RUNS = {seed: get_run(seed) for seed in CFG.SEEDS}
show(pd.concat([r["metrics"] for r in RUNS.values()], ignore_index=True),
     "Diagnosis and concept performance per seed and split (recorded by Part 1)", "model_performance")

BASE, PDS = {}, {}
beta_rows, tau_rows, size_rows = [], [], []
for seed in CFG.SEEDS:
    run = RUNS[seed]
    BASE[seed] = explain_base(run)
    s, conv = shared_scale(run, BASE[seed])
    beta0, sel = choose_beta0(run, BASE[seed], seed, s)
    beta_rows.append(sel)
    for pop in CFG.POPULATIONS:
        PD = population_data(run, BASE[seed], pop, beta0, s)
        nv, nt = PD["data"]["valid"]["N"], PD["data"]["test"]["N"]
        ok = min(nv, nt) >= CFG.MIN_POPULATION_LESIONS
        size_rows.append({"seed": seed, "population": pop, "valid_lesions": nv, "test_lesions": nt,
                          "test_patients": len(set(PD["data"]["test"]["case"])), "analyzed": ok,
                          "score_scale_s": s, "scale_convention": conv, "beta0": beta0, "beta": beta0 / s,
                          "mirror_agreement_test": run["recorded"]["test"]["mirror_agreement"],
                          "mirror_max_abs_error": run["mirror_max_abs_error"]})
        if not ok:
            say(f"seed {seed}, {pop}: {nv} validation and {nt} test lesions, below {CFG.MIN_POPULATION_LESIONS}. Not analyzed.")
            continue
        tau_rows.append(freeze_tau(PD))
        size_rows[-1]["tau"] = PD["tau"]
        PDS[pop, seed] = PD
show(pd.concat(beta_rows), "Tuning curve of beta0 on the validation images of each seed (split-half agreement, then frozen)",
     "tuning_beta0")
TUNING_BETA = pd.concat(beta_rows, ignore_index=True)
TUNING_TAU = pd.concat(tau_rows, ignore_index=True) if tau_rows else pd.DataFrame()
if tau_rows:
    show(TUNING_TAU, "Tuning curve of the tier threshold per population on the validation images of each seed "
         "(bootstrap stability, then frozen)", "tuning_tau")
show(pd.DataFrame(size_rows), "Correctly classified lesions per population, scale, temperature and threshold", "population_sizes")
if not PDS:
    raise RuntimeError("No population reaches MIN_POPULATION_LESIONS in both validation and test for any seed.")

# ---------------- model studied in depth ----------------
# The seed with the largest test F1 is studied in the first and third experiments. The choice is internal to
# the illustration. Its only purpose is to explain the largest number of correctly classified test images.
_perf = pd.concat([r["metrics"] for r in RUNS.values()], ignore_index=True)
_crit = _perf[_perf["split"] == CFG.SELECTION_SPLIT].set_index("seed")[CFG.SELECTION_METRIC]
_eligible = [s_ for s_ in CFG.SEEDS if all((p_, s_) in PDS for p_ in CFG.POPULATIONS)]
if not _eligible:
    raise RuntimeError("No seed has both populations analyzed.")
SEL = int(_crit.loc[_eligible].idxmax())
_rows = []
for s_ in CFG.SEEDS:
    r_ = {"seed": s_, f"{CFG.SELECTION_SPLIT}_{CFG.SELECTION_METRIC}": float(_crit.loc[s_]), "selected": s_ == SEL}
    for p_ in CFG.POPULATIONS:
        r_[f"test_images_{p_}"] = PDS[p_, s_]["data"]["test"]["N"] if (p_, s_) in PDS else 0
    _rows.append(r_)
show(pd.DataFrame(_rows), f"Model studied in depth, the seed with the largest {CFG.SELECTION_SPLIT} {CFG.SELECTION_METRIC}", "model_selection")
say(f"The selected model is seed {SEL}. The first and third experiments study this model. The second compares all seeds.")

# numerical residuals of the exact identities on the fitted games (all should be close to zero)
ver = []
for (pop, seed), PD in PDS.items():
    d = PD["data"]["test"]
    b = d["bank"]
    dpd = deg_from_eta_perm(eta_from_u(d["u"], LAT), PERMS, LAT)
    popg = apg_population(b["P"], b["pair"], 0.1, with_kemeny=False)
    lift = schur_horn_lift(popg["geo"]["ybar"], RHO)[0]
    _, _, degs = removal_curves(d["u"], b["argmax_perm"], LAT)
    ver.append({"seed": seed, "population": pop,
                "Lemma1_uniform_mean_of_score": float(np.abs(dpd.mean(1)).max()),
                "Lemma1_curve_areas_vs_prefix_sum": float(np.abs(degs - b["dmax"]).max()),
                "Thm3_decomposition_residual": popg["geo"]["decomp_residual"],
                "Thm3_schur_horn_lift_residual": float(np.abs(np.diag(lift @ np.diag(RHO) @ lift.T) - popg["geo"]["ybar"]).max()),
                "Thm6_pair_allocation_residual": popg["pair_allocation_residual"]})
show(pd.DataFrame(ver), "Residuals of the exact identities on the fitted games", "identity_checks")


MODELS, POPULATIONS AND ANALYSIS CHOICES TUNED ON THE VALIDATION IMAGES OF EACH SEED
Images copied to local disk in 37 s

--- Filtered splits (asymmetric filter, 50 inconsistent concept profiles) (3 rows)
split  images  melanoma  non_melanoma  patients
train     346        90           256       346
valid     171        61           110       171
 test     324       101           223       324

--- Concept state sets (7 rows)
            attribute  n_states                                                                                  states
      pigment_network         3                                                               absent; atypical; typical
              streaks         3                                                              absent; irregular; regular
         pigmentation         5      absent; diffuse irregular; diffuse regular; localized irregular; localized regular
regression_structures         4                                           absent; blue ar

,seed,population,Lemma1_uniform_mean_of_score,Lemma1_curve_areas_vs_prefix_sum,Thm3_decomposition_residual,Thm3_schur_horn_lift_residual,Thm6_pair_allocation_residual
0,42,melanoma,4.038e-15,7.105e-15,-2.220e-16,8.882e-16,7.105e-15
1,42,non_melanoma,5.296e-15,3.553e-15,0.0000,1.332e-15,0.0000
2,64,melanoma,3.797e-15,5.329e-15,0.0000,1.776e-15,1.776e-14
3,64,non_melanoma,4.302e-15,3.553e-15,3.331e-16,2.220e-15,-7.105e-15
4,128,melanoma,5.793e-15,2.665e-15,3.331e-16,1.110e-15,-2.132e-14
5,128,non_melanoma,4.548e-15,3.553e-15,0.0000,1.776e-15,-3.553e-15
6,256,melanoma,4.568e-15,3.553e-15,0.0000,3.997e-15,-1.066e-14
7,256,non_melanoma,1.594e-15,3.553e-15,1.110e-16,6.661e-16,1.421e-14
8,512,melanoma,3.361e-15,3.553e-15,3.331e-16,6.661e-16,-1.066e-14
9,512,non_melanoma,4.418e-15,3.553e-15,0.0000,6.661e-16,0.0000


# First experiment. Every geometric and algebraic object for the selected model

The case study runs on the correctly classified test lesions of the selected model, once per population. The console output follows the objects of the paper in order, and each block opens with a short note on how to read it.

1. Lemma 1 describes the local explanation itself. It reports how far each SOFI optimum sits above a random ordering, how often the optimum is unique, how well an additive game fits, plus the removal effect of every concept.
2. Theorem 3 places the explanations in the moment polytope. It separates the ambiguity inside each image from the variation between images, then gives the Borda consensus with the slot matrix of positions.
3. Theorem 4 reads the root walls. Each pair of concepts receives the share of explanations that places one before the other, together with majority cycles, the Kemeny order and the images needed to settle open pairs.
4. Theorem 6 uses exchange coordinates. It states which pairs of concepts carry the distance of the explanations from the consensus, plus the principal patterns of exchange and the concept states that drive them.
5. Theorem 7 builds ordered tiers as faces of the permutohedron. It reports the share of explanations that cross a tier boundary. For melanoma it also compares the explanations with the major and minor criteria of the seven-point checklist.
6. Proposition 8 gives simultaneous finite-sample certificates for every statement above.
7. Proposition 9 assembles the certified statements into a partial order of concepts with its levels.

Each block ends with direct answers to the three questions of the introduction. The contrast between the two populations of the model follows. The case study of the other seeds is computed in the same cell, although it is printed in the second experiment.

In [12]:
# ============================================================
# CASE STUDY FUNCTIONS
# One block per mathematical object of the paper, each computed on the test lesions of one population:
#   L1  Lemma 1       SOFI score in Weyl coordinates, null level, removal effects and uniqueness
#   T3  Theorem 3     explanation points, ambiguity, between-lesion spread, Borda consensus
#   T4  Theorem 4     root walls, pairwise shares, majority tournament, 3-cycles, Kemeny order
#   T6  Theorem 6     exchange coordinates, pair shares of the distance, principal exchange subspaces
#   T7  Theorem 7     exchange graph, Lie closure, tier face, crossing shares, external checklist tiers
#   P8  Proposition 8 simultaneous certificates, radii and the images needed to resolve open statements
#   P9  Proposition 9 certified partial order, its closure, covering pairs and levels
# ============================================================
VERTS = np.stack([vertex(p) for p in PERMS])                 # (n!, n) vertices sigma . rho


def _boot_mean_ci(x, rng, B=None):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan, np.nan
    B = B or CFG.N_BOOT
    bs = np.array([x[rng.integers(0, len(x), len(x))].mean() for _ in range(B)])
    return float(x.mean()), float(np.median(x)), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


def checklist_tiers():
    idx = {c: i for i, c in enumerate(CFG.CONCEPT_COLUMNS)}
    return [[idx[c] for c in CFG.CHECKLIST_MAJOR], [idx[c] for c in CFG.CHECKLIST_MINOR]]


def boundary_crossings(M, tiers):
    """Crossing share of every boundary between consecutive tiers (Theorem 7 iii applied per boundary)."""
    rows = []
    for i in range(len(tiers) - 1):
        upper = [a for T in tiers[:i + 1] for a in T]
        lower = [b for T in tiers[i + 1:] for b in T]
        mass = sum(M[b, a] for a in upper for b in lower)
        rows.append({"boundary": f"after tier {i + 1}", "pairs": len(upper) * len(lower),
                     "crossing_mass": mass, "crossing_share": mass / (len(upper) * len(lower))})
    return rows


def bootstrap_population(P, pair, tau, sigma_fixed, tiers_fixed, rng, B):
    """Case bootstrap of the population summaries at the fixed consensus and fixed tiers."""
    L = P.shape[0]
    chk = checklist_tiers()
    out = {k: [] for k in ("ybar", "M", "ombar", "kappa", "Abar", "B", "checklist", "face", "shares")}
    for _ in range(B):
        ii = rng.integers(0, L, L)
        pp = apg_population(P[ii], pair[ii], tau, sigma_ref=sigma_fixed, with_kemeny=False)
        out["ybar"].append(pp["geo"]["ybar"]); out["M"].append(pp["M"]); out["ombar"].append(pp["exch"]["ombar"])
        out["kappa"].append(pp["geo"]["kappa"]); out["Abar"].append(pp["geo"]["Abar"]); out["B"].append(pp["geo"]["B"])
        out["checklist"].append(crossing_mass(pp["M"], chk)[2])
        out["face"].append(crossing_mass(pp["M"], tiers_fixed)[2])
        out["shares"].append(pp["exch"]["pair_shares"])
    return {k: np.array(v) for k, v in out.items()}


def _ci(a, axis=0):
    return np.nanpercentile(a, 2.5, axis=axis), np.nanpercentile(a, 97.5, axis=axis)


def state_name(j, v):
    return CONCEPT_INFO[CFG.CONCEPT_COLUMNS[j]]["states"][int(v)]


def case_study(PD):
    """Every table of the case study for one population of one seed, plus the facts used by the report."""
    d = PD["data"]["test"]
    b, u, L, n = d["bank"], d["u"], d["N"], N_ATTR
    seed, popn = PD["seed"], PD["population"]
    tau = PD["tau"]
    has_tiers = bool(np.isfinite(tau))
    base = {"seed": seed, "population": popn}
    rng = np.random.default_rng(1000 * seed + (1 if popn == "melanoma" else 2))
    T, F = {}, {"seed": seed, "population": popn, "N": L, "tau": tau, "beta0": PD["beta0"], "beta": PD["beta"],
                "patients": len(set(d["case"]))}

    pop = apg_population(b["P"], b["pair"], tau)
    geo, sigma, M = pop["geo"], pop["sigma"], pop["M"]
    ex_ = pop["exch"]
    pos_star = np.empty(n, int)
    pos_star[sigma] = np.arange(n)
    boot = bootstrap_population(b["P"], b["pair"], tau, sigma, pop["tiers"]["tiers"], rng, CFG.N_BOOT)
    cert = operational_certificates(geo["ybar"], M, ex_["ombar"], tau if has_tiers else np.nan, L, n, CFG.DELTA)

    # ================= L1, Lemma 1 =================
    sd0 = np.where(d["sd0"] > 0, d["sd0"], np.nan)
    c_eff, r2 = additive_fit(u, LAT)
    phi = shapley_values(u, LAT)
    degs_all = deg_from_eta_perm(eta_from_u(u, LAT), PERMS, LAT)                  # (L, n!)
    lin_all = 2 * c_eff @ VERTS.T
    corr = np.array([np.corrcoef(degs_all[l], lin_all[l])[0, 1] if degs_all[l].std() > 0 else np.nan for l in range(L)])
    eff_order = np.stack([sort_desc(c_eff[l]) for l in range(L)])
    single = b["argmax_perm"]
    same = np.array([np.array_equal(eff_order[l], single[l]) for l in range(L)], float)
    kt_eff = np.array([kendall_tau(eff_order[l], single[l]) for l in range(L)])
    zmax = b["dmax"] / sd0
    n_eff_orders = len({tuple(r) for r in eff_order})
    rows = []
    for name, x in (("best SOFI score in null standard deviations (0 = random ordering)", zmax),
                    ("share of images with a unique SOFI optimum", (b["n_opt"] == 1).astype(float)),
                    ("R2 of the additive fit of the removal game", r2),
                    ("correlation of the SOFI score with its Weyl-linear form over all orderings", corr),
                    ("share of images whose SOFI optimum sorts the removal effects", same),
                    ("Kendall tau between the SOFI optimum and the removal-effect order", kt_eff)):
        m_, med, lo, hi = _boot_mean_ci(x, rng)
        rows.append(dict(base, statistic=name, mean=m_, median=med, boot_lo=lo, boot_hi=hi))
    rows.append(dict(base, statistic="distinct removal-effect orders among the images (Theorem 4 iii)",
                     mean=float(n_eff_orders), median=np.nan, boot_lo=np.nan, boot_hi=np.nan))
    T["L1_summary"] = pd.DataFrame(rows)
    pos_single = positions_of(single)
    rows = []
    for j in range(n):
        rows.append(dict(base, concept=NAME[j], mean_removal_effect_logit=float(np.mean(c_eff[:, j])),
                         share_supporting_effect=float((c_eff[:, j] > 0).mean()),
                         mean_shapley_logit=float(np.mean(phi[:, j])),
                         share_supporting_shapley=float((phi[:, j] > 0).mean()),
                         mean_position_single_sofi=float(pos_single[:, j].mean() + 1),
                         share_first_single_sofi=float((pos_single[:, j] == 0).mean()),
                         share_last_single_sofi=float((pos_single[:, j] == n - 1).mean())))
    T["L1_concepts"] = pd.DataFrame(rows)
    F["L1"] = {"zmax_mean": float(np.nanmean(zmax)), "unique_share": float((b["n_opt"] == 1).mean()),
               "r2_mean": float(np.nanmean(r2)), "r2_median": float(np.nanmedian(r2)),
               "corr_mean": float(np.nanmean(corr)), "same_share": float(same.mean()),
               "n_eff_orders": n_eff_orders,
               "support_share": {NAME[j]: float((c_eff[:, j] > 0).mean()) for j in range(n)},
               "effect_logit": {NAME[j]: float(np.mean(c_eff[:, j])) for j in range(n)}}

    # ================= T3, Theorem 3 =================
    y, A = geo["y"], geo["A"]
    mass_single = np.exp(PD["beta"] * b["dmax"] - b["logZ"])
    kd_single = np.array([sum(b["pair"][l][c_, a_] if pos_single[l, a_] < pos_single[l, c_] else b["pair"][l][a_, c_]
                              for a_, c_ in PAIRS) for l in range(L)])
    rho_corr = lambda x, z: sps.spearmanr(x, z) if np.std(x) > 0 and np.std(z) > 0 else (np.nan, np.nan)
    rs_logit, p_logit = rho_corr(A, d["logit"])
    rs_z, p_z = rho_corr(A, zmax)
    rows = []
    for name, x in (("ambiguity A_x of the local law", A), ("normalized entropy of the local law", b["h"]),
                    ("probability of the single SOFI ranking under its own law", mass_single),
                    ("expected Kendall distance between the single ranking and the law (of 21 pairs)", kd_single)):
        m_, med, lo, hi = _boot_mean_ci(x, rng)
        rows.append(dict(base, statistic=name, mean=m_, median=med, boot_lo=lo, boot_hi=hi))
    rows.append(dict(base, statistic="Spearman correlation of A_x with the logit of the explained class",
                     mean=float(rs_logit), median=np.nan, boot_lo=np.nan, boot_hi=float(p_logit)))
    rows.append(dict(base, statistic="Spearman correlation of A_x with the best SOFI score (sd units)",
                     mean=float(rs_z), median=np.nan, boot_lo=np.nan, boot_hi=float(p_z)))
    T["T3_local"] = pd.DataFrame(rows).rename(columns={"boot_hi": "boot_hi_or_p"})
    tot = 1 - geo["kappa"] ** 2
    Aj = (np.einsum("ljk,k->lj", b["P"], RHO ** 2) - y ** 2).mean(0) / R2          # within share per concept
    Bj = ((y - geo["ybar"]) ** 2).mean(0) / R2                                    # between share per concept
    Pbar = b["P"].mean(0)
    ylo, yhi = _ci(boot["ybar"])
    rows = []
    for j in sigma:
        states = d["states"][:, j]
        sst = ((y[:, j] - y[:, j].mean()) ** 2).sum()
        ssb = sum((states == v).sum() * (y[states == v, j].mean() - y[:, j].mean()) ** 2 for v in np.unique(states))
        rows.append(dict(base, borda_rank=int(pos_star[j]) + 1, concept=NAME[j], ybar=geo["ybar"][j],
                         mean_position=(n + 1) / 2 - geo["ybar"][j], mean_position_lo=(n + 1) / 2 - yhi[j],
                         mean_position_hi=(n + 1) / 2 - ylo[j], P_first=Pbar[j, 0], P_top3=Pbar[j, :3].sum(),
                         P_last=Pbar[j, -1], spread_share=(Aj[j] + Bj[j]) / tot if tot > 0 else np.nan,
                         within_part=Aj[j] / (Aj[j] + Bj[j]) if Aj[j] + Bj[j] > 0 else np.nan,
                         between_explained_by_state=ssb / sst if sst > 0 else np.nan))
    T["T3_consensus"] = pd.DataFrame(rows)
    klo, khi = _ci(boot["kappa"]); alo, ahi = _ci(boot["Abar"]); blo, bhi = _ci(boot["B"])
    T["T3_decomposition"] = pd.DataFrame([dict(
        base, N=L, kappa=geo["kappa"], kappa_lo=klo, kappa_hi=khi, one_minus_kappa2=tot,
        Abar=geo["Abar"], Abar_lo=alo, Abar_hi=ahi, B=geo["B"], B_lo=blo, B_hi=bhi,
        within_share=geo["Abar"] / tot if tot > 0 else np.nan, between_share=geo["B"] / tot if tot > 0 else np.nan,
        identity_residual=geo["decomp_residual"], borda_order=order_str(sigma), borda_exact_tie=geo["borda_tie"])])
    T["T3_positions"] = pd.DataFrame([dict(base, concept=NAME[j], **{f"slot{k + 1}_{AB[sigma[k]]}": 100 * Pbar[j, k]
                                                                    for k in range(n)}) for j in sigma])
    rows = []
    for j in range(n):
        for v in np.unique(d["states"][:, j]):
            m = d["states"][:, j] == v
            rows.append(dict(base, concept=NAME[j], predicted_state=state_name(j, v), images=int(m.sum()),
                             share=float(m.mean()), mean_position=float(((n + 1) / 2 - y[m, j]).mean()),
                             P_first=float(b["P"][m, j, 0].mean()), P_last=float(b["P"][m, j, -1].mean()),
                             mean_removal_effect_logit=float(np.mean(c_eff[m, j])),
                             share_supporting=float((c_eff[m, j] > 0).mean())))
    T["T3_states"] = pd.DataFrame(rows)
    F["T3"] = {"kappa": geo["kappa"], "kappa_ci": (klo, khi), "Abar": geo["Abar"], "B": geo["B"],
               "within_share": geo["Abar"] / tot if tot > 0 else np.nan,
               "between_share": geo["B"] / tot if tot > 0 else np.nan,
               "order": [NAME[j] for j in sigma], "mean_pos": {NAME[j]: (n + 1) / 2 - geo["ybar"][j] for j in range(n)},
               "P_first": {NAME[j]: Pbar[j, 0] for j in range(n)}, "P_last": {NAME[j]: Pbar[j, -1] for j in range(n)},
               "within_part": {NAME[j]: Aj[j] / (Aj[j] + Bj[j]) if Aj[j] + Bj[j] > 0 else np.nan for j in range(n)},
               "spread_share": {NAME[j]: (Aj[j] + Bj[j]) / tot if tot > 0 else np.nan for j in range(n)},
               "state_expl": {r["concept"]: r["between_explained_by_state"] for r in T["T3_consensus"].to_dict("records")},
               "mass_single_mean": float(mass_single.mean()), "kd_single_mean": float(kd_single.mean()),
               "A_mean": float(A.mean()), "A_logit_rho": float(rs_logit), "A_logit_p": float(p_logit),
               "states": T["T3_states"]}

    # ================= T4, Theorem 4 =================
    Mlo, Mhi = _ci(boot["M"])
    img_pref = (b["pair"] > 0.5)
    rows = []
    for i, (a_, c_) in enumerate(PAIRS):
        w, l_ = (a_, c_) if M[a_, c_] > 0.5 else (c_, a_)
        margin = abs(M[a_, c_] - 0.5)
        rows.append(dict(base, pair=f"{AB[a_]}-{AB[c_]}", winner=NAME[w], loser=NAME[l_], share_winner_first=M[w, l_],
                         boot_lo=Mlo[w, l_], boot_hi=Mhi[w, l_], margin=margin,
                         images_preferring_winner=float(img_pref[:, w, l_].mean()),
                         images_preferring_loser=float(img_pref[:, l_, w].mean()),
                         certified=bool(cert["wall_certified"][i]),
                         images_needed_to_certify=int(math.ceil(math.log(6 * M2 / CFG.DELTA) / (2 * margin ** 2)))
                         if margin > 0 else np.nan))
    T["T4_walls"] = pd.DataFrame(rows).sort_values("margin", ascending=False)
    wins = np.array([sum((M[a_, c_] > 0.5) + 0.5 * np.isclose(M[a_, c_], 0.5) for c_ in range(n) if c_ != a_)
                     for a_ in range(n)])
    kem, kem_cost, kem_cnt = pop["kemeny"]
    borda_vs_major = sum(1 for a_, c_ in PAIRS if (pos_star[a_] < pos_star[c_]) != (M[a_, c_] > 0.5))
    exp_kd = sum(M[c_, a_] if pos_star[a_] < pos_star[c_] else M[a_, c_] for a_, c_ in PAIRS)
    T["T4_tournament"] = pd.DataFrame([dict(
        base, majority_3cycles=len(pop["cycles"]),
        cycles="; ".join(" -> ".join(AB[j] for j in cy) for cy in pop["cycles"]) or "none",
        copeland_wins=", ".join(f"{AB[j]}={wins[j]:.1f}" for j in np.argsort(-wins, kind="stable")),
        kemeny_order=order_str(kem), kemeny_n_optima=kem_cnt, kemeny_equals_borda=bool(np.array_equal(kem, sigma)),
        pairs_where_borda_contradicts_majority=borda_vs_major,
        expected_kendall_distance_to_consensus=exp_kd,
        pairs_with_image_level_disagreement_over_25pct=int((T["T4_walls"]["images_preferring_loser"] > 0.25).sum()))])
    F["T4"] = {"cycles": len(pop["cycles"]), "kemeny_equals_borda": bool(np.array_equal(kem, sigma)),
               "exp_kd": exp_kd, "walls": T["T4_walls"], "borda_vs_major": borda_vs_major,
               "kemeny": [NAME[j] for j in kem]}

    # ================= T6, Theorem 6 =================
    olo, ohi = _ci(boot["ombar"])
    slo, shi = _ci(boot["shares"])
    rows = []
    for i, (a_, c_) in enumerate(PAIRS):
        rows.append(dict(base, pair=f"{AB[a_]}-{AB[c_]}", concept_a=NAME[a_], concept_b=NAME[c_],
                         omega_bar=ex_["ombar"][i], omega_lo=olo[i], omega_hi=ohi[i],
                         consensus_gap=int(abs(pop["Delta"][i])), distance_share=ex_["pair_shares"][i],
                         share_lo=slo[i], share_hi=shi[i], edge_at_tau=bool(has_tiers and ex_["ombar"][i] > tau),
                         edge_certified=bool(cert["edge_certified"][i]) if has_tiers else False))
    T["T6_pairs"] = pd.DataFrame(rows).sort_values("distance_share", ascending=False)
    pc_cert = pca_certificate(ex_["evals"], L, n, CFG.DELTA)
    scores = (pop["omega"] - ex_["ombar"]) @ ex_["evecs"]
    rows, prof = [], []
    for k in range(ex_["n_components"]):
        v = ex_["evecs"][:, k]
        top = np.argsort(-np.abs(v))[:4]
        xp, xm = xi_pm(v, sigma)
        W = W_of_xi(xp) + W_of_xi(xm)
        rs_a = rho_corr(scores[:, k], A)
        rows.append(dict(base, component=k + 1, eigenvalue=ex_["evals"][k], variance_share=ex_["var_shares"][k],
                         top_loadings="; ".join(f"{AB[PAIRS[t][0]]}-{AB[PAIRS[t][1]]} {v[t]:+.2f}" for t in top),
                         gap_certified=pc_cert[k]["certified"] if k < len(pc_cert) else np.nan,
                         subspace_bound=pc_cert[k]["subspace_sin_theta_bound"] if k < len(pc_cert) else np.nan,
                         spearman_score_vs_ambiguity=float(rs_a[0]),
                         xi_realization_residual=float(max(abs(W[a_, c_] - abs(v[i])) for i, (a_, c_) in enumerate(PAIRS)))))
        if k < CFG.PCA_COMPONENTS_TO_PROFILE:
            tests = []
            for j in range(n):
                for vs in np.unique(d["states"][:, j]):
                    m = d["states"][:, j] == vs
                    if CFG.MIN_STATE_IMAGES <= m.sum() <= L - CFG.MIN_STATE_IMAGES:
                        p_ = sps.mannwhitneyu(scores[m, k], scores[~m, k], alternative="two-sided").pvalue
                        tests.append(dict(base, component=k + 1, concept=NAME[j], predicted_state=state_name(j, vs),
                                          images_in_state=int(m.sum()),
                                          mean_score_in_state=float(scores[m, k].mean()),
                                          mean_score_elsewhere=float(scores[~m, k].mean()), mannwhitney_p=p_))
            if tests:
                tdf = pd.DataFrame(tests)
                tdf["holm_p"] = holm(tdf["mannwhitney_p"].values)
                prof.append(tdf.sort_values("mannwhitney_p").head(6))
    T["T6_loadings"] = pd.DataFrame([dict(base, pair=f"{AB[a_]}-{AB[c_]}", concept_a=NAME[a_], concept_b=NAME[c_],
                                          **{f"pc{k + 1}": float(ex_["evecs"][i, k]) for k in range(ex_["n_components"])})
                                     for i, (a_, c_) in enumerate(PAIRS)])
    T["T6_pca"] = pd.DataFrame(rows) if rows else pd.DataFrame([dict(base, component=0, note="no exchange variance")])
    T["T6_pca_states"] = pd.concat(prof, ignore_index=True) if prof else pd.DataFrame([dict(base, note="no state group large enough")])
    top_pairs = T["T6_pairs"].head(3)
    F["T6"] = {"pairs": T["T6_pairs"], "pca": T["T6_pca"], "pca_states": T["T6_pca_states"],
               "residual": pop["pair_allocation_residual"]}

    # ================= T7, Theorem 7 =================
    tf = pop["tiers"]
    X, npairs, share = pop["crossing"]
    flo, fhi = _ci(boot["face"])
    chk = checklist_tiers()
    Xc, npc, sc = crossing_mass(M, chk)
    clo, chi = _ci(boot["checklist"])
    T["T7_face"] = pd.DataFrame([dict(
        base, tau=tau, edges=len(tf["edges"]) if has_tiers else 0,
        components=" | ".join(",".join(AB[j] for j in B_) for B_ in tf["components"]) if has_tiers else "not selected",
        lie_algebra_dim=lie_closure_dim(n, tf["edges"]) if has_tiers else np.nan,
        lie_algebra_dim_expected=sum(len(B_) * (len(B_) - 1) // 2 for B_ in tf["components"]) if has_tiers else np.nan,
        tiers=" | ".join(",".join(AB[j] for j in T_) for T_ in tf["tiers"]), n_tiers=tf["q"],
        face_dimension=tf["dim_face"], block_orbit_dimension=tf["dim_Q"], block_orbit_equals_face=tf["Q_equals_face"],
        crossing_mass=X, crossing_pairs=npairs, crossing_share=share, crossing_share_lo=flo, crossing_share_hi=fhi,
        checklist_crossing_share=sc, checklist_lo=clo, checklist_hi=chi)])
    T["T7_boundaries"] = pd.DataFrame([dict(base, tiering="selected face", **r) for r in boundary_crossings(M, tf["tiers"])]
                                      + [dict(base, tiering="seven-point checklist, major before minor", **r)
                                         for r in boundary_crossings(M, chk)])
    rows = []
    for tt in CFG.TAU_GRID:
        pp = apg_population(b["P"], b["pair"], tt, with_kemeny=False)
        rows.append(dict(base, tau=tt, frozen=bool(has_tiers and np.isclose(tt, tau)),
                         edges=int((pp["exch"]["ombar"] > tt).sum()), n_tiers=pp["tiers"]["q"],
                         tiers=" | ".join(",".join(AB[j] for j in T_) for T_ in pp["tiers"]["tiers"]),
                         crossing_share=pp["crossing"][2]))
    T["T7_hierarchy"] = pd.DataFrame(rows)
    rows = []
    for a_ in chk[0]:
        for c_ in chk[1]:
            rows.append(dict(base, major=NAME[a_], minor=NAME[c_], share_minor_above_major=M[c_, a_]))
    T["T7_checklist_pairs"] = pd.DataFrame(rows).sort_values("share_minor_above_major", ascending=False)
    F["T7"] = {"has_tiers": has_tiers, "tiers": [[NAME[j] for j in T_] for T_ in tf["tiers"]], "n_tiers": tf["q"],
               "crossing_share": share, "crossing_ci": (flo, fhi), "checklist_share": sc, "checklist_ci": (clo, chi),
               "boundaries": T["T7_boundaries"], "Q_equals_face": tf["Q_equals_face"],
               "checklist_pairs": T["T7_checklist_pairs"], "hierarchy": T["T7_hierarchy"]}

    # ================= P8, Proposition 8 =================
    gap = cert["min_adjacent_gap"]
    n_order = int(math.ceil(2 * (n - 1) ** 2 * math.log(6 * n / CFG.DELTA) / gap ** 2)) if gap > 0 else np.nan
    T["P8_certificates"] = pd.DataFrame([dict(
        base, N=L, delta=CFG.DELTA, r_y=cert["r_y"], r_p=cert["r_p"], order_certified=cert["order_certified"],
        min_adjacent_gap=gap, images_needed_for_whole_order=n_order,
        walls_certified=int(cert["wall_certified"].sum()), walls_total=M2,
        edges_certified=int(cert["edge_certified"].sum()) if has_tiers else 0,
        graph_and_face_certified=cert["graph_and_face_certified"] if has_tiers else False,
        pca_gaps_certified=int(sum(bool(r["certified"]) for r in pc_cert)),
        hoeffding_radius_Abar_B_kappa2=hoeffding_radius(0, 1, L, CFG.DELTA / 3))])
    F["P8"] = {"r_y": cert["r_y"], "r_p": cert["r_p"], "order_certified": cert["order_certified"], "gap": gap,
               "n_order": n_order, "walls_certified": int(cert["wall_certified"].sum())}

    # ================= P9, Proposition 9 =================
    E0 = (M - 0.5) > cert["r_p"]
    np.fill_diagonal(E0, False)
    R = transitive_closure(E0)
    acyclic = not np.any(np.diag(R))
    hasse = hasse_edges(R) if acyclic else []
    lev = order_levels(R) if acyclic else np.zeros(n, int)
    levels = {int(k): [NAME[j] for j in range(n) if lev[j] == k] for k in sorted(set(lev))}
    comparable = int(R.sum())
    T["P9_partial_order"] = pd.DataFrame([dict(
        base, certified_pairs_direct=int(E0.sum()), certified_pairs_after_closure=comparable,
        added_by_closure=comparable - int(E0.sum()), share_of_pairs_ordered=comparable / M2,
        strict_partial_order=acyclic, no_majority_3cycle=len(pop["cycles"]) == 0,
        covering_pairs="; ".join(f"{AB[a_]} > {AB[c_]}" for a_, c_ in hasse) or "none",
        levels=" | ".join(",".join(AB[CFG.CONCEPT_NAME.index(x)] for x in v) for v in levels.values()))])
    T["P9_edges"] = pd.DataFrame([dict(base, above=NAME[a_], below=NAME[c_], share=M[a_, c_], direct=bool(E0[a_, c_]),
                                       covering=(a_, c_) in hasse) for a_ in range(n) for c_ in range(n) if R[a_, c_]]
                                 or [dict(base, note="no certified pair")])
    F["P9"] = {"direct": int(E0.sum()), "closure": comparable, "hasse": [(NAME[a_], NAME[c_]) for a_, c_ in hasse],
               "levels": levels, "acyclic": acyclic}

    # image-level table (csv, console only when PRINT_IMAGE_LEVEL)
    T["image_level"] = pd.DataFrame(dict(
        base, derm=d["derm"], patient=d["case"], logit=d["logit"], ambiguity=A, normalized_entropy=b["h"],
        best_score_sd=zmax, n_optima=b["n_opt"], additive_R2=r2, single_sofi=[order_str(p) for p in single],
        prob_single=mass_single, kendall_single_vs_law=kd_single,
        **{f"pos_{AB[j]}": (n + 1) / 2 - y[:, j] for j in range(n)},
        **{f"state_{AB[j]}": [state_name(j, v) for v in d["states"][:, j]] for j in range(n)},
        **{f"pc{k + 1}": scores[:, k] for k in range(min(2, scores.shape[1]))}))

    # ================= answers to the three questions of the introduction =================
    wt = T["T4_walls"]
    contested = wt.sort_values("margin").iloc[0]
    tp = T["T6_pairs"].iloc[0]
    T["answers"] = pd.DataFrame([
        dict(base, question="Q1 ambiguity within images or variation between images",
             answer=f"within {100 * F['T3']['within_share']:.0f}%, between {100 * F['T3']['between_share']:.0f}% of the spread | "
                    f"kappa {geo['kappa']:.2f} | single SOFI ranking carries {100 * mass_single.mean():.2f}% of its own law"),
        dict(base, question="Q2 pairwise preferences, cycles, sources of disagreement",
             answer=f"{int(cert['wall_certified'].sum())} of {M2} pairs certified | {len(pop['cycles'])} majority 3-cycles | "
                    f"most contested {contested['winner']} vs {contested['loser']} ({100 * contested['share_winner_first']:.0f}%) | "
                    f"largest distance share {tp['concept_a']}-{tp['concept_b']} ({100 * tp['distance_share']:.0f}%)"),
        dict(base, question="Q3 ordered tiers and crossings",
             answer=(f"{tf['q']} tiers {' | '.join(','.join(AB[j] for j in T_) for T_ in tf['tiers'])}, crossing share "
                     f"{100 * share:.0f}%" if has_tiers and tf['q'] > 1 else "no nontrivial tier structure")
                    + f" | checklist major-before-minor reversed in {100 * sc:.0f}% of cross pairs | "
                      f"certified partial order covers {comparable} of {M2} pairs")])
    return {"tables": T, "facts": F, "pop": pop}


HOW_TO_READ = {
    "L1_summary": "Lemma 1. A random ordering scores zero on average, so the best score in null standard deviations says "
                  "how far the SOFI explanation stands above chance. The additive fit tells whether the ranking reduces to "
                  "sorting the individual removal effects of the concepts.",
    "L1_concepts": "Lemma 1. Effects are in logit units of the explained class. A positive removal effect means that hiding the concept lowers the score of the explained class, "
                   "so the concept supports the decision. Concepts with negative effects sit at the end of a SOFI ranking "
                   "as counter-evidence, not as irrelevant features.",
    "T3_local": "Theorem 3 (ii). Ambiguity is zero when one ranking takes all the probability of an image. The probability "
                "of the single SOFI ranking says how much of the explanation law one reported ranking represents.",
    "T3_consensus": "Theorem 3 (iii)-(iv). Borda consensus with mean positions. The spread share splits 1 - kappa^2 over "
                    "concepts exactly. The within part is the fraction of that concept's spread that comes from uncertainty "
                    "inside single images. The last column is the share of the between-image variation of the concept's "
                    "position explained by its predicted state.",
    "T3_decomposition": "Theorem 3 (iii). 1 - kappa^2 = Abar + B, the exact split of the lack of agreement into ambiguity "
                        "within images plus variation between images.",
    "T3_positions": "Soft ranking averaged over images (percent). Column k is the consensus slot k, labeled with the concept "
                    "that holds it in the consensus. The same table gives the exchange matrix R of Theorem 6 averaged "
                    "over images, since R_x(a,b) is the probability that a occupies the consensus slot of b.",
    "T3_states": "Theorem 3 applied to images grouped by the predicted state of each concept. The ranking concerns the "
                 "attribute at the state the model predicted, so this table says which state drives each position.",
    "T4_walls": "Theorem 4. Share of explanations that rank the winner first, with the share of images whose own law "
                "prefers each direction. Certified pairs pass the simultaneous rule of Proposition 8.",
    "T4_tournament": "Theorem 4 (ii). Without 3-cycles the majorities fit one order, the Kemeny order. The expected "
                     "Kendall distance counts how many of the 21 pairwise orders of the consensus an explanation reverses.",
    "T6_pairs": "Theorem 6 (iv). Exact shares of the expected squared distance to the consensus carried by each pair, "
                "weighting how often two concepts trade places by how far apart the consensus puts them.",
    "T6_pca": "Theorem 6. Principal exchange subspaces, the patterns of trading places that recur across images. A "
              "certified gap means the pattern is stable at this sample size (Proposition 8 ii).",
    "T6_loadings": "Theorem 6. Full loadings of every principal exchange pattern on the 21 pairs, for plotting.",
    "T6_pca_states": "Predicted concept states of the images that carry each principal exchange pattern most strongly "
                     "(Mann-Whitney tests, Holm corrected within a component).",
    "T7_face": "Theorem 7. Lie closure of the exchange graph gives concept groups, whose consecutive closure gives ordered "
               "tiers that form a face of the permutohedron. The crossing share is the fraction of cross-tier orders "
               "the explanations reverse.",
    "T7_boundaries": "Theorem 7 (iii) at each boundary, for the selected tiers plus the external tiering of the "
                     "seven-point checklist (major criteria before minor criteria). A share near 50% means the "
                     "explanations ignore the boundary.",
    "T7_hierarchy": "Tiers along the threshold grid. Larger thresholds keep only frequent exchanges, so the tiers refine.",
    "T7_checklist_pairs": "Share of explanations that place a minor checklist criterion above a major one, pair by pair.",
    "P8_certificates": "Proposition 8. Simultaneous radii at level 1 - delta, with the number of images that would be "
                       "needed to certify the whole consensus order at the observed smallest gap.",
    "P9_partial_order": "Proposition 9. Pairs with a certified majority direction, closed under transitivity. The covering "
                        "pairs draw the Hasse diagram. Level 0 holds concepts with nothing certified above them.",
    "P9_edges": "Every certified order of the partial order, with its pooled share.",
    "answers": "Short answers to the three questions of the introduction.",
}


def print_case_study(R):
    T = R["tables"]
    F = R["facts"]
    banner(f"CASE STUDY | {CFG.POPULATION_LABEL[F['population']]} | seed {F['seed']} | {F['N']} test images "
           f"from {F['patients']} patients | beta0 {F['beta0']} | tau {F['tau']}", "#")
    for key in ("L1_summary", "L1_concepts", "T3_local", "T3_decomposition", "T3_consensus", "T3_positions", "T3_states",
                "T4_walls", "T4_tournament", "T6_pairs", "T6_pca", "T6_loadings", "T6_pca_states", "T7_face", "T7_boundaries",
                "T7_hierarchy", "T7_checklist_pairs", "P8_certificates", "P9_partial_order", "P9_edges", "answers"):
        say("")
        say(f"[{key}] {HOW_TO_READ[key]}")
        show(T[key], key, key=f"case_{key}")
    show(T["image_level"], "image-level quantities", key="case_image_level", echo=CFG.PRINT_IMAGE_LEVEL)


def holm(p):
    p = np.asarray(p, float)
    out = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    pv = p[ok]
    m = len(pv)
    order = np.argsort(pv)
    adj = np.empty(m)
    run = 0.0
    for r, i in enumerate(order):
        run = max(run, min(1.0, (m - r) * pv[i]))
        adj[i] = run
    out[ok] = adj
    return out


def t_interval(x, alpha=0.05):
    x = np.asarray(x, float)
    x = x[~np.isnan(x)]
    if len(x) < 2:
        return (float(x.mean()) if len(x) else np.nan, np.nan, np.nan)
    m, s = x.mean(), x.std(ddof=1)
    h = sps.t.ppf(1 - alpha / 2, len(x) - 1) * s / np.sqrt(len(x))
    return m, m - h, m + h


# ---------------- contrast between the two populations of one model ----------------
CONTRAST_STATS = [f"mean position of {NAME[j]}" for j in range(N_ATTR)] + [
    "agreement kappa", "ambiguity within images Abar", "variation between images B",
    "checklist crossing share"]


def compare_two_populations(bA, bB, seed):
    """Melanoma minus non-melanoma: permutation test on population labels, case bootstrap, Holm."""
    Y = np.concatenate([bA["y"], bB["y"]])
    pair = np.concatenate([bA["pair"], bB["pair"]])
    lab = np.r_[np.ones(len(bA["y"]), bool), np.zeros(len(bB["y"]), bool)]
    chk = checklist_tiers()
    cross = [(a_, c_) for a_ in chk[0] for c_ in chk[1]]

    def stats(idx):
        yb = Y[idx].mean(0)
        Mm = pair[idx].mean(0)
        return np.r_[(N_ATTR + 1) / 2 - yb, math.sqrt((yb ** 2).sum() / R2), (1 - (Y[idx] ** 2).sum(1) / R2).mean(),
                     ((Y[idx] - yb) ** 2).sum(1).mean() / R2, np.mean([Mm[c_, a_] for a_, c_ in cross])]

    def diff(lb):
        return stats(np.where(lb)[0]) - stats(np.where(~lb)[0])

    obs = diff(lab)
    rng = np.random.default_rng(seed + 11)
    perm = np.array([diff(rng.permutation(lab)) for _ in range(CFG.N_PERM)])
    p = ((np.abs(perm) >= np.abs(obs) - 1e-15).sum(0) + 1) / (CFG.N_PERM + 1)
    ia, ib = np.where(lab)[0], np.where(~lab)[0]
    boots = np.array([stats(rng.choice(ia, len(ia))) - stats(rng.choice(ib, len(ib))) for _ in range(CFG.N_BOOT)])
    df = pd.DataFrame({"seed": seed, "statistic": CONTRAST_STATS, "melanoma": stats(ia), "non_melanoma": stats(ib),
                       "difference": obs, "boot_lo": np.percentile(boots, 2.5, 0), "boot_hi": np.percentile(boots, 97.5, 0),
                       "permutation_p": p})
    df["holm_p"] = holm(df["permutation_p"].values)
    return df

In [13]:
banner(f"FIRST EXPERIMENT | EVERY GEOMETRIC AND ALGEBRAIC OBJECT FOR THE SELECTED MODEL (SEED {SEL})")
# The case study is computed for every seed here, since the second experiment reuses it.
# Only the selected model is printed in this experiment. The other seeds are printed in the second one.
CS = {}
for (pop, seed), PD in PDS.items():
    t0 = time.time()
    CS[pop, seed] = case_study(PD)
    say(f"case study of {pop}, seed {seed}, computed in {time.time() - t0:.1f} s")
for pop in CFG.POPULATIONS:
    if (pop, SEL) in CS:
        print_case_study(CS[pop, SEL])

CONTRAST_ALL = {}
for seed in CFG.SEEDS:
    if all((p, seed) in PDS for p in CFG.POPULATIONS):
        bA, bB = PDS["melanoma", seed]["data"]["test"]["bank"], PDS["non_melanoma", seed]["data"]["test"]["bank"]
        CONTRAST_ALL[seed] = compare_two_populations(bA, bB, seed)
banner(f"MELANOMA VERSUS NON-MELANOMA EXPLANATIONS OF THE SELECTED MODEL (SEED {SEL})")
say("[contrast] Each population explains its own class. Differences are melanoma minus non-melanoma, with case-bootstrap")
say("intervals, permutation p-values on population labels and Holm adjustment over the statistics.")
show(CONTRAST_ALL[SEL], "contrast of the selected model", "contrast")


FIRST EXPERIMENT | EVERY GEOMETRIC AND ALGEBRAIC OBJECT FOR THE SELECTED MODEL (SEED 64)
case study of melanoma, seed 42, computed in 2.0 s
case study of non_melanoma, seed 42, computed in 1.8 s
case study of melanoma, seed 64, computed in 1.4 s
case study of non_melanoma, seed 64, computed in 1.8 s
case study of melanoma, seed 128, computed in 1.4 s
case study of non_melanoma, seed 128, computed in 1.8 s
case study of melanoma, seed 256, computed in 1.4 s
case study of non_melanoma, seed 256, computed in 2.3 s
case study of melanoma, seed 512, computed in 1.9 s
case study of non_melanoma, seed 512, computed in 2.0 s

####################################################################################################
CASE STUDY | melanoma | seed 64 | 68 test images from 68 patients | beta0 3.0 | tau 0.2
####################################################################################################

[L1_summary] Lemma 1. A random ordering scores zero on average, so the best score 

,seed,statistic,melanoma,non_melanoma,difference,boot_lo,boot_hi,permutation_p,holm_p
0,64,mean position of pigment network,3.6158,5.1414,-1.5257,-1.8022,-1.2436,4.998e-04,0.0055
1,64,mean position of streaks,3.5346,3.2372,0.2975,-0.0244,0.6552,0.0600,0.0600
2,64,mean position of pigmentation,5.2272,2.9160,2.3112,2.0784,2.5757,4.998e-04,0.0055
3,64,mean position of regression structures,5.2028,4.5086,0.6942,0.4421,0.9223,4.998e-04,0.0055
4,64,mean position of dots and globules,1.7253,3.5605,-1.8352,-2.0281,-1.6124,4.998e-04,0.0055
5,64,mean position of blue-whitish veil,3.9390,3.3135,0.6255,0.1029,1.1112,4.998e-04,0.0055
6,64,mean position of vascular structures,4.7553,5.3228,-0.5675,-0.7821,-0.3011,4.998e-04,0.0055
7,64,agreement kappa,0.5690,0.4525,0.1165,0.0699,0.1654,4.998e-04,0.0055
8,64,ambiguity within images Abar,0.3291,0.5806,-0.2515,-0.2720,-0.2310,4.998e-04,0.0055
9,64,variation between images B,0.3471,0.2146,0.1325,0.0749,0.1814,4.998e-04,0.0055


## Hyperparameter sensitivity of the selected model

The frozen choices come from the validation images. The tables below describe how the test results of the selected model move along the grid of beta0, tau and the certificate level delta, one choice at a time with the others fixed.

In [ ]:
# ============================================================
# HYPERPARAMETER SENSITIVITY OF THE SELECTED MODEL
# The choices are frozen on the validation images. The tables below only describe how the test-image
# results of the selected model would move along each grid, one choice at a time, the others fixed.
#   beta0  inverse temperature of the local ranking laws (the single SOFI ranking does not depend on it)
#   tau    tier threshold (the tier hierarchy along the grid is the case_T7_hierarchy table)
#   delta  level of the simultaneous certificates of Proposition 8
# ============================================================

def _expected_kendall_single(single, pair):
    pos = positions_of(single)
    out = np.zeros(len(single))
    for a_, c_ in PAIRS:
        a_first = pos[:, a_] < pos[:, c_]
        out += np.where(a_first, pair[:, c_, a_], pair[:, a_, c_])
    return out


def _closure_pairs(M, r_p):
    E0 = (M - 0.5) > r_p
    np.fill_diagonal(E0, False)
    return int(transitive_closure(E0).sum())


def beta_sensitivity(seed):
    rows = []
    for pop in CFG.POPULATIONS:
        if (pop, seed) not in PDS:
            continue
        PD = PDS[pop, seed]
        d = PD["data"]["test"]
        u, L, s, tau = d["u"], d["N"], PD["scale"], PD["tau"]
        sigma_ref = CS[pop, seed]["pop"]["sigma"]
        chk = checklist_tiers()
        for b0 in CFG.BETA0_GRID:
            bk = local_bank(u, b0 / s)
            pp = apg_population(bk["P"], bk["pair"], tau)
            geo, M = pp["geo"], pp["M"]
            tot = 1 - geo["kappa"] ** 2
            cert = operational_certificates(geo["ybar"], M, pp["exch"]["ombar"], tau if np.isfinite(tau) else np.nan,
                                            L, N_ATTR, CFG.DELTA)
            _, _, degs = removal_curves(u, np.tile(pp["sigma"], (L, 1)), LAT)
            dmax = np.where(bk["dmax"] > 0, bk["dmax"], np.nan)
            sd0 = np.where(d["sd0"] > 0, d["sd0"], np.nan)
            rows.append({"seed": seed, "population": pop, "beta0": b0, "beta": b0 / s,
                         "frozen": bool(np.isclose(b0, PD["beta0"])),
                         "kappa": geo["kappa"], "Abar": geo["Abar"], "B": geo["B"],
                         "within_share": geo["Abar"] / tot if tot > 0 else np.nan,
                         "between_share": geo["B"] / tot if tot > 0 else np.nan,
                         "mean_normalized_entropy": float(bk["h"].mean()),
                         "mean_prob_single_sofi": float(np.exp(b0 / s * bk["dmax"] - bk["logZ"]).mean()),
                         "mean_expected_kendall_single_vs_law": float(_expected_kendall_single(bk["argmax_perm"], bk["pair"]).mean()),
                         "consensus": order_str(pp["sigma"]),
                         "kendall_tau_to_frozen_consensus": kendall_tau(pp["sigma"], sigma_ref),
                         "majority_3cycles": len(pp["cycles"]),
                         "walls_certified": int(cert["wall_certified"].sum()),
                         "partial_order_pairs": _closure_pairs(M, cert["r_p"]),
                         "n_tiers_at_frozen_tau": pp["tiers"]["q"] if np.isfinite(tau) else np.nan,
                         "tiers_at_frozen_tau": " | ".join(",".join(AB[j] for j in T_) for T_ in pp["tiers"]["tiers"])
                         if np.isfinite(tau) else "not selected",
                         "tier_crossing_share": pp["crossing"][2] if np.isfinite(tau) else np.nan,
                         "checklist_crossing_share": crossing_mass(M, chk)[2],
                         "top_distance_pair": "-".join(AB[j] for j in PAIRS[int(np.argmax(pp["exch"]["pair_shares"]))]),
                         "pc1_variance_share": float(pp["exch"]["var_shares"][0]) if len(pp["exch"]["var_shares"]) else np.nan,
                         "consensus_degradation_sd": float(np.nanmean(degs / sd0)),
                         "consensus_retained_share": float(np.nanmean(degs / dmax))})
    return pd.DataFrame(rows)


def delta_sensitivity(seed):
    rows = []
    for pop in CFG.POPULATIONS:
        if (pop, seed) not in CS:
            continue
        pp, L = CS[pop, seed]["pop"], CS[pop, seed]["facts"]["N"]
        tau = PDS[pop, seed]["tau"]
        for dl in CFG.DELTA_GRID:
            cert = operational_certificates(pp["geo"]["ybar"], pp["M"], pp["exch"]["ombar"],
                                            tau if np.isfinite(tau) else np.nan, L, N_ATTR, dl)
            gap = cert["min_adjacent_gap"]
            rows.append({"seed": seed, "population": pop, "delta": dl, "frozen": bool(np.isclose(dl, CFG.DELTA)),
                         "r_y": cert["r_y"], "r_p": cert["r_p"], "order_certified": cert["order_certified"],
                         "images_needed_for_whole_order": int(math.ceil(2 * (N_ATTR - 1) ** 2 * math.log(6 * N_ATTR / dl) / gap ** 2))
                         if gap > 0 else np.nan,
                         "walls_certified": int(cert["wall_certified"].sum()),
                         "edges_certified": int(cert["edge_certified"].sum()) if np.isfinite(tau) else 0,
                         "partial_order_pairs": _closure_pairs(pp["M"], cert["r_p"]),
                         "pca_gaps_certified": int(sum(r["certified"] for r in pca_certificate(pp["exch"]["evals"], L, N_ATTR, dl)))})
    return pd.DataFrame(rows)


banner(f"HYPERPARAMETER SENSITIVITY OF THE SELECTED MODEL (SEED {SEL}, TEST IMAGES)")
say("[sensitivity_beta0] One row per value of beta0, with tau and delta frozen. The single SOFI ranking of an image does")
say("not depend on beta0, whereas its law, the ambiguity, the consensus and every certificate built on the law do.")
SENS_BETA = beta_sensitivity(SEL)
show(SENS_BETA, "sensitivity to beta0", "sensitivity_beta0")
say("")
say("[sensitivity_beta0, validation] The tuning curve that fixed beta0 for the selected model.")
show(TUNING_BETA[TUNING_BETA["seed"] == SEL], "validation tuning curve of beta0, selected model")
say("")
say("[sensitivity_tau] The tiers along the whole threshold grid are in the case_T7_hierarchy table above. The table below")
say("repeats the validation tuning curve of the selected model, which fixed the frozen threshold.")
if len(TUNING_TAU):
    show(TUNING_TAU[TUNING_TAU["seed"] == SEL], "validation tuning curve of tau, selected model")
say("")
say("[sensitivity_delta] Certificates of Proposition 8 at three levels, with beta0 and tau frozen.")
SENS_DELTA = delta_sensitivity(SEL)
show(SENS_DELTA, "sensitivity to delta", "sensitivity_delta")

## Detailed reports of the selected model

The first experiment closes with one report per population in plain clinical language. The reports are printed to the console and saved as `report_melanoma.txt` and `report_non_melanoma.txt`.

In [ ]:
# ============================================================
# DETAILED REPORTS OF THE SELECTED MODEL
# One report per population, written in plain clinical language from the numbers of the first experiment.
# ============================================================
# ---------------- text helpers (plain sentences, no colons or semicolons) ----------------
SHORT = {"borda_single": "Borda count", "kemeny_single": "Kemeny consensus", "copeland_single": "Copeland count",
         "most_frequent_single": "most frequent ranking", "mean_shapley": "mean Shapley value",
         "apg_borda_law": "APG Borda consensus", "apg_kemeny_law": "APG Kemeny order",
         "best_fixed_ranking": "best fixed ranking", "local_sofi": "image-specific SOFI ranking"}


def pc(x, d=0):
    return "not available" if x is None or not np.isfinite(x) else f"{100 * x:.{d}f}%"


def num(x, d=2):
    return "not available" if x is None or not np.isfinite(x) else f"{x:.{d}f}"


def big(x):
    return "not available" if x is None or not np.isfinite(x) else f"{x:,.0f}"


def span(vals, fmt):
    """Mean over seeds with the range in parentheses, meant to close a sentence."""
    v = [x for x in vals if x is not None and np.isfinite(x)]
    if not v:
        return "not available"
    if len(v) == 1 or np.isclose(min(v), max(v)):
        return fmt(v[0]) + ("" if len(v) == 1 else " at every seed")
    return f"{fmt(float(np.mean(v)))} on average ({fmt(min(v))} to {fmt(max(v))} across seeds)"


def rng_(vals, fmt):
    v = [x for x in vals if x is not None and np.isfinite(x)]
    if not v:
        return "not available"
    return fmt(v[0]) if np.isclose(min(v), max(v)) else f"from {fmt(min(v))} to {fmt(max(v))}"


def listing(items):
    items = list(items)
    if not items:
        return "none"
    if len(items) == 1:
        return items[0]
    return ", ".join(items[:-1]) + " as well as " + items[-1]


def chunks(items, k=3):
    items = list(items)
    return [items[i:i + k] for i in range(0, len(items), k)]


def seeds_phrase(k, S):
    return "every seed" if k == S else f"{k} of {S} seeds"


def tiers_txt(tiers):
    return ", then ".join("{" + ", ".join(t) + "}" for t in tiers)


def population_report(pop, seed):
    """Detailed plain-language report of one population for the model of one seed."""
    lab = CFG.POPULATION_LABEL[pop]
    if (pop, seed) not in CS:
        return [f"The model of seed {seed} has too few correctly classified {lab} images for a report."]
    F = CS[pop, seed]["facts"]
    PD = PDS[pop, seed]
    out = []
    add = out.append

    add(f"DETAILED REPORT FOR THE {lab.upper()} POPULATION OF THE MODEL TRAINED WITH SEED {seed}")
    add("")
    add("1. What this report covers")
    add(f"The population holds the {F['N']} correctly classified {lab} images of the test split, from {F['patients']} patients.")
    add(f"Each explanation ranks the seven dermoscopic concepts by how much the model's score for {lab} relies on them.")
    add("Hiding a concept means replacing its predicted state by a neutral zero input inside the model.")
    add("Every number describes this trained model under that rule, not the biology of the lesions.")
    add(f"This model has the largest test F1 among the trained seeds, which gives the largest number of images to explain.")

    add("")
    add("2. Which concepts the model relies on first")
    add(f"The consensus order is {', then '.join(F['T3']['order'])}.")
    pf = pd.Series(F["T3"]["P_first"])
    add(f"Under the explanation laws, {pf.idxmax()} holds the first slot in {pc(pf.max())} of explanations, the largest share of any concept.")
    lc = CS[pop, seed]["tables"]["L1_concepts"].set_index("concept")
    add(f"Among the single SOFI rankings, {lc['share_first_single_sofi'].idxmax()} comes first for {pc(lc['share_first_single_sofi'].max())} of images.")
    add(f"By contrast, {pf.idxmin()} opens only {pc(pf.min(), 1)} of explanations under the laws.")
    sup = pd.Series(F["L1"]["support_share"])
    against = sup[sup < 0.5].sort_values()
    if len(against):
        for c, v in against.items():
            add(f"Hiding {c} raises the {lab} score in {pc(1 - v)} of images.")
        add("A late position of such a concept marks evidence against the decision rather than irrelevance.")
    else:
        add(f"Hiding any single concept lowers the {lab} score in most images, so every concept tends to support the decision.")
    st = pd.Series(F["T3"]["state_expl"]).sort_values(ascending=False)
    sd_all = F["T3"]["states"]
    for concept in list(st.index[:3]):
        if not np.isfinite(st[concept]) or st[concept] < 0.2:
            continue
        sd = sd_all[(sd_all["concept"] == concept) & (sd_all["images"] >= CFG.MIN_STATE_IMAGES)]
        g = sd.set_index("predicted_state")["mean_position"].sort_values()
        if len(g) >= 2:
            add(f"The predicted state of {concept} explains {pc(st[concept])} of the image-to-image variation of its position.")
            add(f"The concept sits near position {num(g.iloc[0], 1)} when the model predicts '{g.index[0]}', "
                f"whereas it sits near position {num(g.iloc[-1], 1)} when the model predicts '{g.index[-1]}'.")

    add("")
    add("3. Question 1. Does one average order tell the whole story?")
    add(f"Agreement between explanations is kappa = {num(F['T3']['kappa'])} (95% bootstrap interval "
        f"{num(F['T3']['kappa_ci'][0])} to {num(F['T3']['kappa_ci'][1])}).")
    add("On this scale, one would mean the same ranking for every image.")
    add(f"The share of the remaining disagreement due to ambiguity inside single images is {pc(F['T3']['within_share'])}.")
    add("Differences between images account for the rest.")
    if F["T3"]["within_share"] > 0.5:
        add("Most of the disagreement therefore lives inside single images, which admit several almost equally faithful rankings.")
    else:
        add("Most of the disagreement therefore comes from differences between images, which calls for reading subgroups of images separately.")
    add(f"The single SOFI ranking of an image carries a probability of {pc(F['T3']['mass_single_mean'], 2)} on average under that image's own explanation law.")
    add(f"A ranking drawn from the law reverses {num(F['T3']['kd_single_mean'], 1)} of the 21 pairwise orders of the single ranking on average.")
    wp = pd.Series(F["T3"]["within_part"])
    add(f"The position of {wp.idxmax()} is the least settled inside single images, since {pc(wp.max())} of its spread arises within images.")
    add(f"The position of {wp.idxmin()} depends most on the image, with the largest between-image part of its spread ({pc(1 - wp.min())}).")
    add(f"The best SOFI ranking of an image stands {num(F['L1']['zmax_mean'], 1)} null standard deviations above a random ordering on average.")
    add("Lemma 1 sets the expected score of a random ordering to zero, which makes this scale comparable across images.")
    add(f"The best ranking is unique for {pc(F['L1']['unique_share'])} of images.")
    add(f"The removal game has a median additive R2 of {num(F['L1']['r2_median'])}.")
    add(f"Even so, the SOFI ranking equals the order of the individual removal effects for only {pc(F['L1']['same_share'])} of images.")
    add("Small interactions between concepts are enough to reorder concepts whose individual effects are close.")
    if np.isfinite(F["T3"]["A_logit_p"]) and F["T3"]["A_logit_p"] < 0.05:
        rho_ = F["T3"]["A_logit_rho"]
        add(f"Explanations are more ambiguous for images with a {'lower' if rho_ < 0 else 'higher'} {lab} score "
            f"(Spearman correlation {num(rho_)}, p = {F['T3']['A_logit_p']:.2g}).")
        if rho_ > 0:
            add("The most confident decisions are thus the hardest to summarize with one ranking.")
    else:
        add(f"The ambiguity of an explanation shows no significant association with the {lab} score of the image.")
    add(f"These ambiguity figures hold at the inverse temperature beta0 = {PD['beta0']} chosen on the validation images.")
    if "SENS_BETA" in globals() and len(SENS_BETA):
        sb = SENS_BETA[SENS_BETA["population"] == pop]
        add(f"Over the grid from beta0 = {CFG.BETA0_GRID[0]} to {CFG.BETA0_GRID[-1]}, the within-image share moves "
            f"from {pc(sb['within_share'].min())} to {pc(sb['within_share'].max())}, whereas kappa moves from "
            f"{num(sb['kappa'].min())} to {num(sb['kappa'].max())}.")
        add(f"The consensus keeps a Kendall tau of at least {num(sb['kendall_tau_to_frozen_consensus'].min())} with the frozen one over that grid.")

    add("")
    add("4. Question 2, first part. Do the pairwise preferences fit one order?")
    w = F["T4"]["walls"]
    wc = w[w["certified"]]
    add(f"A majority direction is certified at the {pc(1 - CFG.DELTA)} level for {len(wc)} of the 21 concept pairs.")
    if len(wc):
        items = [f"{r.winner} before {r.loser} ({pc(r.share_winner_first)} of explanations)" for r in wc.itertuples()]
        for gi, grp in enumerate(chunks(items, 3)):
            add(("They include " if gi == 0 else "They also include ") + listing(grp) + ".")
    if F["T4"]["cycles"] == 0:
        add("No majority 3-cycle appears, so the pairwise majorities fit one order.")
        add(f"The Kemeny order of Theorem 4 {'equals' if F['T4']['kemeny_equals_borda'] else 'differs from'} the Borda consensus.")
    else:
        add(f"The majorities form {F['T4']['cycles']} 3-cycles, so no single ranking agrees with every pairwise majority.")
    r = w.sort_values("margin").iloc[0]
    add(f"The most contested pair is {r.winner} against {r.loser}, with {pc(r.share_winner_first)} of explanations for the first.")
    if len(wc):
        r = wc.sort_values("images_preferring_loser", ascending=False).iloc[0]
        add("Even among certified pairs, some images prefer the opposite order.")
        add(f"The largest such minority concerns {r.winner} before {r.loser}, reversed by the own law of {pc(r.images_preferring_loser)} of images.")
    wu = w[~w["certified"]]
    if len(wu):
        r = wu.sort_values("margin", ascending=False).iloc[0]
        add(f"The strongest uncertified pair, {r.winner} before {r.loser}, would need {big(r.images_needed_to_certify)} images to be certified.")

    add("")
    add("5. Question 2, second part. Which exchanges of concepts produce the disagreement?")
    tp = F["T6"]["pairs"].head(3)
    add(f"The exchange of {tp.iloc[0].concept_a} with {tp.iloc[0].concept_b} carries the largest share of the distance to the consensus ({pc(tp.iloc[0].distance_share)}).")
    add(f"The next contributions come from {listing([f'{x.concept_a} with {x.concept_b} ({pc(x.distance_share)})' for x in tp.iloc[1:].itertuples()])}.")
    pca = F["T6"]["pca"]
    if "variance_share" in pca:
        p1 = pca.iloc[0]
        add(f"The leading pattern of exchanges carries {pc(p1['variance_share'])} of the exchange variance.")
        add(f"Its largest loadings are {p1['top_loadings'].replace(';', ',')}, with a {'certified' if bool(p1['gap_certified']) else 'non-certified'} eigengap.")
        prof = F["T6"]["pca_states"]
        if "holm_p" in prof:
            hit = prof[(prof["component"] == 1) & (prof["holm_p"] < 0.05)]
            if len(hit):
                r = hit.sort_values("holm_p").iloc[0]
                add(f"Images with a predicted '{r.predicted_state}' state of {r.concept} carry this pattern "
                    f"{'more' if r.mean_score_in_state > r.mean_score_elsewhere else 'less'} than other images (Holm-adjusted p = {r.holm_p:.2g}).")
            else:
                add("No predicted concept state separates the images carrying this pattern after Holm correction.")

    add("")
    add("6. Question 3. Can the concepts be presented as ordered tiers?")
    f7 = F["T7"]
    if f7["has_tiers"] and f7["n_tiers"] > 1:
        add(f"At the threshold tau = {PD['tau']} chosen on the validation images, the tiers are {tiers_txt(f7['tiers'])}.")
        add(f"Explanations reverse {pc(f7['crossing_share'])} of the cross-tier orders "
            f"(95% bootstrap interval {pc(f7['crossing_ci'][0])} to {pc(f7['crossing_ci'][1])}).")
        add("Concepts inside one tier have no reliable order among themselves.")
    else:
        add("The validation images supported no nontrivial tier structure at any threshold of the grid, so the report presents no tiers.")
    lv = F["P9"]["levels"]
    if F["P9"]["closure"] > 0:
        add(f"The certified partial order of Proposition 9 groups the concepts into the levels {tiers_txt(lv.values())}.")
        add("These levels carry a finite-sample guarantee, whereas the tiers above carry only a stability check.")
    if pop == "melanoma":
        add("The seven-point checklist places its major criteria above its minor criteria as evidence for melanoma.")
        add(f"The explanations reverse that split in {pc(f7['checklist_share'])} of cross comparisons "
            f"(95% bootstrap interval {pc(f7['checklist_ci'][0])} to {pc(f7['checklist_ci'][1])}).")
        add("A share of 50% would mean that the reliance of the model ignores the checklist split.")
        cp = f7["checklist_pairs"].set_index(["minor", "major"])["share_minor_above_major"]
        add(f"The strongest reversal places {cp.idxmax()[0]} above {cp.idxmax()[1]} in {pc(cp.max())} of explanations.")
        add(f"The closest agreement places {cp.idxmin()[1]} above {cp.idxmin()[0]} in {pc(1 - cp.min())} of explanations.")
    else:
        add("The checklist weighs its criteria as evidence for melanoma, so it gives no reference order for a non-melanoma decision.")

    add("")
    add("7. How much support these statements have")
    if F["P8"]["order_certified"]:
        add("The whole consensus order is certified.")
    else:
        add("The whole consensus order is not certified.")
        add(f"At the observed smallest gap between neighboring concepts, certifying it would need {big(F['P8']['n_order'])} images.")
    add(f"The certified partial order orders {F['P9']['closure']} of the 21 pairs, {F['P9']['closure'] - F['P9']['direct']} of them through transitivity.")
    add(f"Every certified statement holds simultaneously with probability at least {pc(1 - CFG.DELTA)} under the sampling assumptions of Proposition 8.")
    if "SENS_DELTA" in globals() and len(SENS_DELTA):
        sd_ = SENS_DELTA[SENS_DELTA["population"] == pop].set_index("delta")["partial_order_pairs"]
        add(f"At the levels {listing([pc(1 - x) for x in sd_.index])}, the partial order covers {listing([str(int(v)) for v in sd_.values])} pairs, respectively.")

    add("")
    add("8. How these explanations differ from those of the other class")
    ct = CONTRAST_ALL.get(seed)
    if ct is not None:
        sgn = 1 if pop == "melanoma" else -1
        sig = ct[ct["holm_p"] < 0.05]
        add(f"After Holm correction, {len(sig)} of the {len(ct)} contrast statistics differ between the two classes.")
        for r in sig.itertuples():
            v = sgn * r.difference
            if r.statistic.startswith("mean position"):
                add(f"{r.statistic.replace('mean position of ', '').capitalize()} sits {abs(r.difference):.1f} positions "
                    f"{'earlier' if v < 0 else 'later'} in {lab} explanations.")
            elif r.statistic == "checklist crossing share":
                continue
            else:
                add(f"The {r.statistic} is {'higher' if v > 0 else 'lower'} for {lab} by {abs(r.difference):.2f}.")

    add("")
    add("9. What this report does not say")
    add(f"The statements describe how one trained model scores {lab} under one computational removal rule.")
    add("A model trained with another seed gives other explanations, which the second experiment documents.")
    add("They are not claims about the biology of the lesions, the accuracy of a diagnosis, or benefits for patients.")
    add("Agreement measures the similarity of explanations, not the confidence of the model in its diagnosis.")
    return out

In [16]:
REPORTS = {}
for pop in CFG.POPULATIONS:
    lines = population_report(pop, SEL)
    REPORTS[pop] = lines
    banner(f"DETAILED REPORT | {CFG.POPULATION_LABEL[pop].upper()} | SELECTED MODEL (SEED {SEL})", "*")
    for line in lines:
        say(line)
    with open(os.path.join(CFG.OUTPUT_DIR, f"report_{pop}.txt"), "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines) + "\n")


****************************************************************************************************
DETAILED REPORT | MELANOMA | SELECTED MODEL (SEED 64)
****************************************************************************************************
DETAILED REPORT FOR THE MELANOMA POPULATION OF THE MODEL TRAINED WITH SEED 64

1. What this report covers
The population holds the 68 correctly classified melanoma images of the test split, from 68 patients.
Each explanation ranks the seven dermoscopic concepts by how much the model's score for melanoma relies on them.
Hiding a concept means replacing its predicted state by a neutral zero input inside the model.
Every number describes this trained model under that rule, not the biology of the lesions.
This model has the largest test F1 among the trained seeds, which gives the largest number of images to explain.

2. Which concepts the model relies on first
The consensus order is dots and globules, then streaks, then pigment network, 

# Second experiment. The same analysis for every trained model

Different seeds give different models, whose explanations differ. This is expected, so the experiment documents the variation rather than testing for agreement. Each seed uses its own beta0 and tau, tuned on its own validation images. The full case study of every other seed is printed first, followed by tables that compare the seeds, namely performance and choices, key quantities, agreement between pairs of models, pairwise orders, positions, state effects, exchange patterns and the population contrast. A short summary per population is saved as `seed_variation_*.txt`.

In [17]:
# ============================================================
# SECOND EXPERIMENT. THE SAME ANALYSIS FOR EVERY TRAINED MODEL
# Different seeds give different models, whose explanations differ. The tables below document how every
# structure of the first experiment varies with retraining, and which statements persist across seeds.
# Each seed uses its own beta0 and tau, tuned on its own validation images.
# ============================================================

def _certified_relation(R):
    """Certified partial order of Proposition 9 as a boolean matrix (after transitive closure)."""
    M, L = R["pop"]["M"], R["facts"]["N"]
    r_p = R["facts"]["P8"]["r_p"]
    E0 = (M - 0.5) > r_p
    np.fill_diagonal(E0, False)
    return transitive_closure(E0)


def seeds_overview():
    perf = pd.concat([r["metrics"] for r in RUNS.values()], ignore_index=True)
    rows = []
    for seed in CFG.SEEDS:
        te = perf[(perf["seed"] == seed) & (perf["split"] == "test")].iloc[0]
        va = perf[(perf["seed"] == seed) & (perf["split"] == "valid")].iloc[0]
        r = {"seed": seed, "selected": seed == SEL, "test_f1": te["label_f1"], "valid_f1": va["label_f1"],
             "test_accuracy": te["label_acc"], "test_auroc": te["auroc"], "test_melanoma_sensitivity": te["mel_sensitivity"],
             "test_concept_accuracy": te["concept_acc"], "best_epoch": te["best_epoch"],
             "beta0": TUNING_BETA[(TUNING_BETA["seed"] == seed) & TUNING_BETA["chosen"]]["beta0"].iloc[0]}
        for pop in CFG.POPULATIONS:
            r[f"images_{pop}"] = PDS[pop, seed]["data"]["test"]["N"] if (pop, seed) in PDS else 0
            r[f"tau_{pop}"] = PDS[pop, seed]["tau"] if (pop, seed) in PDS else np.nan
        rows.append(r)
    return pd.DataFrame(rows)


def synthesis_per_seed():
    rows = []
    for (pop, seed), R in sorted(CS.items(), key=lambda kv: (kv[0][0], CFG.SEEDS.index(kv[0][1]))):
        F_ = R["facts"]
        t6 = F_["T6"]["pairs"].iloc[0]
        pca = F_["T6"]["pca"]
        rows.append({"population": pop, "seed": seed, "selected": seed == SEL, "images": F_["N"],
                     "beta0": F_["beta0"], "tau": F_["tau"],
                     "borda_order": order_str([NAME.index(c) for c in F_["T3"]["order"]]),
                     "kemeny_order": order_str([NAME.index(c) for c in F_["T4"]["kemeny"]]),
                     "kappa": F_["T3"]["kappa"], "Abar": F_["T3"]["Abar"], "B": F_["T3"]["B"],
                     "within_share": F_["T3"]["within_share"], "between_share": F_["T3"]["between_share"],
                     "mean_prob_single_sofi": F_["T3"]["mass_single_mean"], "mean_kendall_single_vs_law": F_["T3"]["kd_single_mean"],
                     "spearman_ambiguity_vs_logit": F_["T3"]["A_logit_rho"], "spearman_p": F_["T3"]["A_logit_p"],
                     "best_score_sd": F_["L1"]["zmax_mean"], "additive_R2_median": F_["L1"]["r2_median"],
                     "share_sofi_equals_effect_order": F_["L1"]["same_share"],
                     "walls_certified": F_["P8"]["walls_certified"], "majority_3cycles": F_["T4"]["cycles"],
                     "kemeny_equals_borda": F_["T4"]["kemeny_equals_borda"],
                     "n_tiers": F_["T7"]["n_tiers"] if F_["T7"]["has_tiers"] else 1,
                     "tiers": " | ".join(",".join(AB[NAME.index(c)] for c in t) for t in F_["T7"]["tiers"]) if F_["T7"]["has_tiers"] else "none",
                     "tier_crossing_share": F_["T7"]["crossing_share"] if F_["T7"]["has_tiers"] else np.nan,
                     "checklist_crossing_share": F_["T7"]["checklist_share"],
                     "top_distance_pair": t6["pair"], "top_distance_share": t6["distance_share"],
                     "pc1_variance_share": float(pca["variance_share"].iloc[0]) if "variance_share" in pca else np.nan,
                     "pc1_top_loadings": pca["top_loadings"].iloc[0] if "top_loadings" in pca else "",
                     "partial_order_pairs": F_["P9"]["closure"],
                     "partial_order_levels": " | ".join(",".join(AB[NAME.index(c)] for c in v) for v in F_["P9"]["levels"].values()),
                     "order_certified": F_["P8"]["order_certified"], "images_needed_for_whole_order": F_["P8"]["n_order"]})
    return pd.DataFrame(rows)


def seed_pair_agreement():
    rows = []
    for pop in CFG.POPULATIONS:
        ss = [s for s in CFG.SEEDS if (pop, s) in CS]
        for i in range(len(ss)):
            for j in range(i + 1, len(ss)):
                A_, B_ = CS[pop, ss[i]], CS[pop, ss[j]]
                RA, RB = _certified_relation(A_), _certified_relation(B_)
                union = int((RA | RB).sum())
                tA, tB = A_["facts"]["T7"], B_["facts"]["T7"]
                both_tiers = tA["has_tiers"] and tB["has_tiers"]
                rows.append({"population": pop, "seed_a": ss[i], "seed_b": ss[j],
                             "kendall_tau_consensus": kendall_tau(A_["pop"]["sigma"], B_["pop"]["sigma"]),
                             "same_leading_concept": bool(A_["pop"]["sigma"][0] == B_["pop"]["sigma"][0]),
                             "wall_direction_agreement": float(np.mean([(A_["pop"]["M"][a, b] > 0.5) == (B_["pop"]["M"][a, b] > 0.5)
                                                                        for a, b in PAIRS])),
                             "certified_pairs_a": int(RA.sum()), "certified_pairs_b": int(RB.sum()),
                             "certified_in_both_same_direction": int((RA & RB).sum()),
                             "certified_in_opposite_directions": int((RA & RB.T).sum()),
                             "certified_relation_jaccard": (RA & RB).sum() / union if union else np.nan,
                             "tier_ARI": adjusted_rand_score(tier_labels(A_["pop"]["tiers"]["tiers"], N_ATTR),
                                                             tier_labels(B_["pop"]["tiers"]["tiers"], N_ATTR)) if both_tiers else np.nan,
                             "pca1_abs_cosine": float(abs(A_["pop"]["exch"]["evecs"][:, 0] @ B_["pop"]["exch"]["evecs"][:, 0])),
                             "distance_share_correlation": float(np.corrcoef(A_["pop"]["exch"]["pair_shares"],
                                                                             B_["pop"]["exch"]["pair_shares"])[0, 1])})
    return pd.DataFrame(rows)


def walls_across_seeds():
    rows = []
    for pop in CFG.POPULATIONS:
        ss = [s for s in CFG.SEEDS if (pop, s) in CS]
        if not ss:
            continue
        Ms = np.stack([CS[pop, s]["pop"]["M"] for s in ss])
        Rs = np.stack([_certified_relation(CS[pop, s]) for s in ss])
        for a, b in PAIRS:
            if Ms[:, a, b].mean() < 0.5:
                a, b = b, a
            rows.append({"population": pop, "pair": f"{AB[a]} before {AB[b]}", "first": NAME[a], "second": NAME[b],
                         "seeds": len(ss), "mean_share": Ms[:, a, b].mean(), "min_share": Ms[:, a, b].min(),
                         "max_share": Ms[:, a, b].max(), "share_selected_model": Ms[ss.index(SEL), a, b] if SEL in ss else np.nan,
                         "seeds_majority_this_direction": int((Ms[:, a, b] > 0.5).sum()),
                         "seeds_certified_this_direction": int(Rs[:, a, b].sum()),
                         "seeds_certified_opposite": int(Rs[:, b, a].sum()),
                         **{f"share_seed{s}": Ms[k, a, b] for k, s in enumerate(ss)}})
    return pd.DataFrame(rows).sort_values(["population", "seeds_certified_this_direction", "mean_share"], ascending=[True, False, False])


def positions_across_seeds():
    rows = []
    for pop in CFG.POPULATIONS:
        ss = [s for s in CFG.SEEDS if (pop, s) in CS]
        for j in range(N_ATTR):
            mp = np.array([CS[pop, s]["facts"]["T3"]["mean_pos"][NAME[j]] for s in ss])
            rk = np.array([int(np.where(CS[pop, s]["pop"]["sigma"] == j)[0][0]) + 1 for s in ss])
            rows.append({"population": pop, "concept": NAME[j], "mean_position_over_seeds": mp.mean(),
                         "sd_over_seeds": mp.std(ddof=1) if len(mp) > 1 else np.nan, "min": mp.min(), "max": mp.max(),
                         "consensus_rank_min": rk.min(), "consensus_rank_max": rk.max(),
                         "seeds_first_in_consensus": int((rk == 1).sum()), "seeds_last_in_consensus": int((rk == N_ATTR).sum()),
                         **{f"position_seed{s}": mp[k] for k, s in enumerate(ss)},
                         **{f"rank_seed{s}": rk[k] for k, s in enumerate(ss)}})
    return pd.DataFrame(rows).sort_values(["population", "mean_position_over_seeds"])


def states_across_seeds():
    rows = []
    for pop in CFG.POPULATIONS:
        ss = [s for s in CFG.SEEDS if (pop, s) in CS]
        st = pd.concat([CS[pop, s]["facts"]["T3"]["states"] for s in ss], ignore_index=True)
        st = st[st["images"] >= CFG.MIN_STATE_IMAGES]
        eta = {s: CS[pop, s]["facts"]["T3"]["state_expl"] for s in ss}
        for (c, v), g in st.groupby(["concept", "predicted_state"]):
            e = np.array([eta[s][c] for s in ss])
            rows.append({"population": pop, "concept": c, "predicted_state": v, "seeds_with_state": len(g),
                         "mean_position": g["mean_position"].mean(), "min_position": g["mean_position"].min(),
                         "max_position": g["mean_position"].max(), "mean_share_supporting": g["share_supporting"].mean(),
                         "state_explained_share_mean": np.nanmean(e), "state_explained_share_min": np.nanmin(e),
                         **{f"position_seed{s}": (g[g["seed"] == s]["mean_position"].iloc[0] if (g["seed"] == s).any() else np.nan)
                            for s in ss}})
    return pd.DataFrame(rows).sort_values(["population", "state_explained_share_mean", "concept", "mean_position"],
                                          ascending=[True, False, True, True])


def exchange_across_seeds():
    rows = []
    for (pop, seed), R in CS.items():
        pca = R["facts"]["T6"]["pca"]
        prof = R["facts"]["T6"]["pca_states"]
        hit = prof[(prof["component"] == 1) & (prof["holm_p"] < 0.05)] if "holm_p" in prof else pd.DataFrame()
        top = hit.sort_values("holm_p").iloc[0] if len(hit) else None
        tp = R["facts"]["T6"]["pairs"].head(3)
        rows.append({"population": pop, "seed": seed, "selected": seed == SEL,
                     "top3_distance_pairs": "; ".join(f"{r.pair} {100 * r.distance_share:.1f}%" for r in tp.itertuples()),
                     "pc1_variance_share": float(pca["variance_share"].iloc[0]) if "variance_share" in pca else np.nan,
                     "pc1_top_loadings": pca["top_loadings"].iloc[0] if "top_loadings" in pca else "",
                     "pc1_gap_certified": bool(pca["gap_certified"].iloc[0]) if "gap_certified" in pca else False,
                     "pc1_top_state": f"{top.concept} = {top.predicted_state}" if top is not None else "none",
                     "pc1_top_state_direction": ("higher" if top.mean_score_in_state > top.mean_score_elsewhere else "lower")
                     if top is not None else "",
                     "pc1_top_state_holm_p": top.holm_p if top is not None else np.nan})
    return pd.DataFrame(rows).sort_values(["population", "seed"])


def contrast_across_seeds():
    df = pd.concat(CONTRAST_ALL.values(), ignore_index=True)
    rows = []
    for st_, g in df.groupby("statistic", sort=False):
        sig = g["holm_p"] < 0.05
        rows.append({"statistic": st_, "seeds": len(g), "mean_difference": g["difference"].mean(),
                     "min_difference": g["difference"].min(), "max_difference": g["difference"].max(),
                     "seeds_significant_melanoma_higher": int((sig & (g["difference"] > 0)).sum()),
                     "seeds_significant_melanoma_lower": int((sig & (g["difference"] < 0)).sum()),
                     "seeds_not_significant": int((~sig).sum()),
                     "same_sign_and_significant_at_every_seed": bool(sig.all() and (np.sign(g["difference"]).nunique() == 1)),
                     **{f"difference_seed{s}": v for s, v in zip(g["seed"], g["difference"])}})
    return pd.DataFrame(rows)


def seed_variation_report(pop):
    """Plain-language summary of what changes and what persists across the trained models."""
    lab = CFG.POPULATION_LABEL[pop]
    ss = [s for s in CFG.SEEDS if (pop, s) in CS]
    S = len(ss)
    out = []
    add = out.append
    add(f"HOW THE {lab.upper()} EXPLANATIONS CHANGE ACROSS {S} TRAINED MODELS")
    add("")
    add("Every seed trains a different model, whose explanations differ. This summary describes that variation.")
    add(f"Each model uses its own analysis choices, tuned on its own validation images. The populations hold "
        f"{listing([str(CS[pop, s]['facts']['N']) for s in ss])} images for seeds {listing([str(s) for s in ss])}, respectively.")
    sp = SEEDS_PAIRS[SEEDS_PAIRS["population"] == pop]
    if len(sp):
        add(f"The Kendall tau between the consensus orders of two models ranges from {num(sp['kendall_tau_consensus'].min())} "
            f"to {num(sp['kendall_tau_consensus'].max())}, with a mean of {num(sp['kendall_tau_consensus'].mean())}.")
        add(f"Two models share the leading concept of the consensus in {pc(sp['same_leading_concept'].mean())} of the {len(sp)} model pairs.")
        add(f"On average, two models agree on the majority direction of {pc(sp['wall_direction_agreement'].mean())} of the 21 concept pairs.")
        opp = int(sp["certified_in_opposite_directions"].sum())
        add(f"Certified orders of two models contradict each other in {opp} cases over all model pairs.")
    ps = SEEDS_POS[SEEDS_POS["population"] == pop]
    first = ps.sort_values("seeds_first_in_consensus", ascending=False).iloc[0]
    add(f"{first.concept.capitalize()} leads the consensus for {int(first.seeds_first_in_consensus)} of {S} models.")
    stable_last = ps[ps["seeds_last_in_consensus"] == S]
    if len(stable_last):
        add(f"{listing(list(stable_last['concept'])).capitalize()} closes the consensus for every model.")
    rng_pos = ps.sort_values("sd_over_seeds")
    add(f"The mean position of {rng_pos.iloc[0].concept} varies least across models, from {num(rng_pos.iloc[0]['min'], 1)} to {num(rng_pos.iloc[0]['max'], 1)}.")
    add(f"The mean position of {rng_pos.iloc[-1].concept} varies most, from {num(rng_pos.iloc[-1]['min'], 1)} to {num(rng_pos.iloc[-1]['max'], 1)}.")
    wa = SEEDS_WALLS[(SEEDS_WALLS["population"] == pop) & (SEEDS_WALLS["seeds_certified_this_direction"] == S)]
    if len(wa):
        items = [f"{r.first} before {r.second} ({pc(r.min_share)} to {pc(r.max_share)} of explanations)" for r in wa.itertuples()]
        add(f"{len(wa)} pairwise orders are certified for every model.")
        for gi, grp in enumerate(chunks(items, 3)):
            add(("They are " if gi == 0 else "They also include ") + listing(grp) + ".")
    else:
        add("No pairwise order is certified for every model.")
    st = SEEDS_STATES[(SEEDS_STATES["population"] == pop) & (SEEDS_STATES["seeds_with_state"] == S)]
    top = st.groupby("concept")["state_explained_share_min"].first().sort_values(ascending=False)
    for c in list(top.index[:2]):
        if top[c] < 0.2:
            continue
        g = st[st["concept"] == c].sort_values("mean_position")
        if len(g) < 2:
            continue
        add(f"For every model, the predicted state of {c} explains at least {pc(top[c])} of the between-image variation of its position.")
        add(f"Across models, the state '{g.iloc[0].predicted_state}' places it between positions {num(g.iloc[0].min_position, 1)} "
            f"and {num(g.iloc[0].max_position, 1)}, whereas '{g.iloc[-1].predicted_state}' places it between "
            f"{num(g.iloc[-1].min_position, 1)} and {num(g.iloc[-1].max_position, 1)}.")
    sy = SEEDS_SYN[SEEDS_SYN["population"] == pop]
    add(f"The share of disagreement due to ambiguity inside images ranges from {pc(sy['within_share'].min())} to {pc(sy['within_share'].max())} across models.")
    add(f"The certified partial order covers from {int(sy['partial_order_pairs'].min())} to {int(sy['partial_order_pairs'].max())} of the 21 pairs.")
    nt = sy["n_tiers"]
    add(f"Nontrivial tiers appear for {int((nt > 1).sum())} of {S} models.")
    ex = SEEDS_EXCH[SEEDS_EXCH["population"] == pop]
    tops = ex["pc1_top_state"][ex["pc1_top_state"] != "none"].str.split(" = ").str[0].value_counts()
    if len(tops):
        add(f"The leading exchange pattern is tied to the predicted state of {tops.index[0]} for {int(tops.iloc[0])} of {S} models.")
    add(f"The selected model of the first experiment is seed {SEL}. Its statements belong to that model, not to every model.")
    return out


banner("SECOND EXPERIMENT | THE SAME ANALYSIS FOR EVERY TRAINED MODEL")
say("Different seeds give different models, whose explanations differ. The full case study of every other seed follows,")
say("then the tables that compare the seeds. Each seed uses its own beta0 and tau tuned on its own validation images.")
for seed in CFG.SEEDS:
    if seed == SEL:
        continue
    for pop in CFG.POPULATIONS:
        if (pop, seed) in CS:
            print_case_study(CS[pop, seed])
    if seed in CONTRAST_ALL:
        banner(f"MELANOMA VERSUS NON-MELANOMA EXPLANATIONS, SEED {seed}")
        show(CONTRAST_ALL[seed], f"contrast of seed {seed}", "contrast")

banner("COMPARISON OF THE TRAINED MODELS")
SEEDS_OVERVIEW = show(seeds_overview(), "Performance and analysis choices of every model", "seeds_overview")
SEEDS_SYN = show(synthesis_per_seed(), "Key quantities per population and model", "seeds_synthesis")
say("[seeds_pair_agreement] Agreement between the reports of two models. Certified orders come from Proposition 9.")
say("The PCA cosine compares the leading exchange patterns. The last column correlates the pair shares of Theorem 6.")
SEEDS_PAIRS = show(seed_pair_agreement(), "Agreement between pairs of models", "seeds_pair_agreement")
say("[seeds_walls] Each pair is oriented by its mean share over models. The counts say how many models share the direction.")
SEEDS_WALLS = show(walls_across_seeds(), "Pairwise orders across models", "seeds_walls")
SEEDS_POS = show(positions_across_seeds(), "Mean positions and consensus ranks across models", "seeds_positions")
SEEDS_STATES = show(states_across_seeds(), "Positions by predicted concept state across models", "seeds_states")
SEEDS_EXCH = show(exchange_across_seeds(), "Leading exchanges across models", "seeds_exchanges")
say("[seeds_contrast] Melanoma minus non-melanoma per model. A difference is stable only if it keeps its sign at every seed.")
SEEDS_CONTRAST = show(contrast_across_seeds(), "Population contrast across models", "seeds_contrast")
for pop in CFG.POPULATIONS:
    lines = seed_variation_report(pop)
    banner(f"SUMMARY OF THE VARIATION ACROSS MODELS | {CFG.POPULATION_LABEL[pop].upper()}", "*")
    for line in lines:
        say(line)
    with open(os.path.join(CFG.OUTPUT_DIR, f"seed_variation_{pop}.txt"), "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines) + "\n")


SECOND EXPERIMENT | THE SAME ANALYSIS FOR EVERY TRAINED MODEL
Different seeds give different models, whose explanations differ. The full case study of every other seed follows,
then the tables that compare the seeds. Each seed uses its own beta0 and tau tuned on its own validation images.

####################################################################################################
CASE STUDY | melanoma | seed 42 | 67 test images from 67 patients | beta0 3.0 | tau 0.2
####################################################################################################

[L1_summary] Lemma 1. A random ordering scores zero on average, so the best score in null standard deviations says how far the SOFI explanation stands above chance. The additive fit tells whether the ranking reduces to sorting the individual removal effects of the concepts.

--- L1_summary (7 rows)
 seed population                                                                  statistic    mean  median  boot_lo 

# Third experiment. One global ranking per population versus the local SOFI rankings

Eight rules build one global ranking per population of the selected model. Five aggregate the single SOFI rankings of the images (Borda, Kemeny, Copeland, most frequent ranking, mean Shapley value), two summarize the full ranking laws (Borda and Kemeny of the population law), and the last is the best fixed ranking found by dynamic programming over all orderings. Each rule is fitted on the test images themselves, which is the most favorable setting for a single ranking, as well as on the validation images of the same class.

The concepts of every test image are then removed through zero removal, the operator of the SOFI game, in the order of the global ranking. The same metrics follow from the image's own SOFI order. MoRF measures sparsity (lower is better), LeRF measures faithfulness (higher is better), whereas the degradation score LeRF minus MoRF is the SOFI score of Lemma 1. Scores are centered at the random-ordering expectation and scaled by its standard deviation for each image. Equal global and local scores would mean that one ranking explains every image of the population.

Paired differences between global and local scores carry bootstrap intervals for all three metrics. MoRF and LeRF receive two-sided Wilcoxon signed-rank tests with rank-biserial correlations and Holm adjustment. The local order maximizes the degradation score on each image, so its difference is reported as an effect size without a test. The experiment is repeated for every seed, with Student intervals over seeds, and a short summary per population is saved as `global_vs_local_*.txt`.

In [18]:
# ============================================================
# THIRD EXPERIMENT FUNCTIONS. ONE GLOBAL RANKING PER POPULATION VERSUS THE LOCAL SOFI RANKINGS
# Several aggregation rules build one ranking per population. The concepts of every image are then removed
# (zero removal, the operator of the SOFI game) in the order of that ranking, which gives the MoRF area
# (sparsity), the LeRF area (faithfulness) and the degradation score LeRF - MoRF on the image's own game.
# The same metrics follow from the image's own SOFI order. Equal scores would mean that one ranking can
# explain every image of the population. The degradation score is the SOFI score, so the local order
# maximizes it on each image and the paired degradation difference is an effect size, never negative.
# MoRF and LeRF alone are not maximized by SOFI, so they receive paired tests.
# Scores are centered at the random-ordering expectation and expressed in null standard deviations.
# ============================================================
GLOBAL_METHODS = {
    "borda_single": "Borda count of the single SOFI rankings (mean position)",
    "kemeny_single": "Kemeny consensus of the single SOFI rankings",
    "copeland_single": "Copeland count of pairwise wins of the single SOFI rankings",
    "most_frequent_single": "most frequent single SOFI ranking",
    "mean_shapley": "concepts sorted by mean standardized Shapley value",
    "apg_borda_law": "Borda consensus of the Gibbs ranking laws (APG consensus, Theorem 3 iv)",
    "apg_kemeny_law": "Kemeny order of the pooled pairwise shares (Theorem 4 ii)",
    "best_fixed_ranking": "ranking with the largest mean standardized degradation over the fitting images",
}


def global_rankings(dfit):
    """One ranking per aggregation rule, built from the images of dfit (one population, one split)."""
    b, u = dfit["bank"], dfit["u"]
    n = N_ATTR
    single = b["argmax_perm"]
    pos = positions_of(single)
    mean_pos = pos.mean(0)
    borda = np.lexsort((np.arange(n), mean_pos))
    Ms = pair_matrix_of_rankings(single)
    wins = np.array([sum((Ms[a, c] > 0.5) + 0.5 * np.isclose(Ms[a, c], 0.5) for c in range(n) if c != a) for a in range(n)])
    rank_b = np.empty(n)
    rank_b[borda] = np.arange(n)
    copeland = np.lexsort((np.arange(n), rank_b, -wins))
    tuples, counts = np.unique(single, axis=0, return_counts=True)
    best = np.where(counts == counts.max())[0]
    mode = tuples[min(best, key=lambda i: (kendall_distance(tuples[i], borda), tuple(tuples[i])))]
    sd0 = np.where(dfit["sd0"] > 0, dfit["sd0"], 1.0)
    phi = shapley_values(u, LAT) / sd0[:, None]
    ustd = (u / sd0[:, None]).mean(0, keepdims=True)
    return {"borda_single": borda, "kemeny_single": kemeny_dp(Ms)[0], "copeland_single": copeland,
            "most_frequent_single": mode, "mean_shapley": sort_desc(phi.mean(0)),
            "apg_borda_law": population_geometry(b["P"])["sigma_star"],
            "apg_kemeny_law": kemeny_dp(pooled_pairs(b["pair"]))[0],
            "best_fixed_ranking": deg_extremes_dp(ustd, LAT)["argmax_perm"][0]}


def evaluate_rankings(d, rankings):
    """Image-level MoRF, LeRF and degradation for the local SOFI ranking and for every global ranking."""
    u, b = d["u"], d["bank"]
    L = d["N"]
    sd0 = np.where(d["sd0"] > 0, d["sd0"], np.nan)
    e0 = null_curve_area(u, LAT)
    degs_all = deg_from_eta_perm(eta_from_u(u, LAT), PERMS, LAT)
    local_pos = positions_of(b["argmax_perm"])
    rows = []
    for method, perm in [("local_sofi", None)] + list(rankings.items()):
        perms = b["argmax_perm"] if perm is None else np.tile(np.asarray(perm), (L, 1))
        morf, lerf, deg = removal_curves(u, perms, LAT)
        tol = 1e-9 * (1 + np.abs(degs_all).max(1))
        pct = ((degs_all < deg[:, None] - tol[:, None]).sum(1) + 0.5 * (np.abs(degs_all - deg[:, None]) <= tol[:, None]).sum(1)) / degs_all.shape[1]
        kd = np.array([kendall_distance(perms[l], b["argmax_perm"][l]) for l in range(L)])
        rows.append(pd.DataFrame({"method": method, "image": np.arange(L), "derm": d["derm"],
                                  "ranking": order_str(perms[0]) if perm is not None else [order_str(p) for p in perms],
                                  "morf_auc": morf, "lerf_auc": lerf, "degradation": deg,
                                  "morf_centered_sd": (morf - e0) / sd0, "lerf_centered_sd": (lerf - e0) / sd0,
                                  "degradation_sd": deg / sd0, "retained_share": deg / np.where(b["dmax"] > 0, b["dmax"], np.nan),
                                  "percentile_among_orderings": pct, "kendall_to_local": kd}))
    return pd.concat(rows, ignore_index=True)


def removal_curve_points(u, perms, lat):
    """Game values along the MoRF and LeRF paths, k = 0..n concepts removed."""
    u = np.atleast_2d(np.asarray(u, float))
    perms = np.atleast_2d(np.asarray(perms, int))
    bits = (1 << perms)
    pref = np.concatenate([np.zeros((perms.shape[0], 1), int), np.cumsum(bits, axis=1)], axis=1)
    return np.take_along_axis(u, pref, 1), np.take_along_axis(u, lat.full ^ pref[:, ::-1], 1)


def mean_curves(d, rankings, meta):
    """Mean centered MoRF and LeRF curves per method, with bootstrap intervals over images."""
    u, b, L = d["u"], d["bank"], d["N"]
    sd0 = np.where(d["sd0"] > 0, d["sd0"], np.nan)
    layer_mean = np.stack([u[:, LAT.layers[k]].mean(1) for k in range(N_ATTR + 1)], 1)
    rng = np.random.default_rng(meta["seed"] + 71)
    boots = [rng.integers(0, L, L) for _ in range(CFG.N_BOOT)]
    rows = []
    for method, perm in [("local_sofi", None)] + list(rankings.items()):
        perms = b["argmax_perm"] if perm is None else np.tile(np.asarray(perm), (L, 1))
        mv, lv = removal_curve_points(u, perms, LAT)
        for curve, v in (("MoRF", mv), ("LeRF", lv)):
            c = (v - layer_mean) / sd0[:, None]
            bm = np.array([np.nanmean(c[ii], 0) for ii in boots])
            for k in range(N_ATTR + 1):
                rows.append(dict(meta, method=method, curve=curve, concepts_removed=k,
                                 mean_centered_sd=float(np.nanmean(c[:, k])),
                                 boot_lo=float(np.percentile(bm[:, k], 2.5)), boot_hi=float(np.percentile(bm[:, k], 97.5)),
                                 mean_raw_logit=float(v[:, k].mean())))
    return pd.DataFrame(rows)


def _wilcoxon(x, y=None, alternative="two-sided"):
    """Wilcoxon signed-rank p-value with the matched-pairs rank-biserial correlation (NaN when degenerate)."""
    dlt = np.asarray(x, float) - (0.0 if y is None else np.asarray(y, float))
    dlt = dlt[np.isfinite(dlt)]
    nz = dlt[dlt != 0]
    if len(nz) < 1:
        return np.nan, np.nan
    r = sps.rankdata(np.abs(nz))
    rb = (r[nz > 0].sum() - r[nz < 0].sum()) / r.sum()
    try:
        p = sps.wilcoxon(dlt, alternative=alternative).pvalue
    except ValueError:
        p = np.nan
    return float(p), float(rb)


def _paired(x, rng):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan, np.nan
    bs = np.array([x[rng.integers(0, len(x), len(x))].mean() for _ in range(CFG.N_BOOT)])
    return float(x.mean()), float(np.median(x)), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


def global_ranking_stats(ev, seed, pop, fit_on, rng):
    """Per method, the metrics of the global ranking and the paired differences with the local SOFI order."""
    loc = ev[ev["method"] == "local_sofi"].set_index("image")
    rows, diffs = [], []
    for method, g in ev.groupby("method", sort=False):
        g = g.set_index("image").loc[loc.index]
        r = {"seed": seed, "population": pop, "fit_on": fit_on, "method": method,
             "ranking": g["ranking"].iloc[0] if method != "local_sofi" else "image-specific", "images": len(g)}
        for col in ("morf_centered_sd", "lerf_centered_sd", "degradation_sd"):
            m_, med, lo, hi = _paired(g[col].values, rng)
            r.update({f"{col}_mean": m_, f"{col}_median": med, f"{col}_lo": lo, f"{col}_hi": hi})
        r.update({"share_degradation_nonpositive": float((g["degradation"].values <= 0).mean()),
                  "mean_percentile_among_orderings": float(g["percentile_among_orderings"].mean()),
                  "mean_retained_share": float(np.nanmean(g["retained_share"])),
                  "median_retained_share": float(np.nanmedian(g["retained_share"])),
                  "share_retained_below_half": float(np.nanmean(g["retained_share"] < 0.5)),
                  "mean_kendall_to_local": float(g["kendall_to_local"].mean()),
                  "share_same_order_as_local": float((g["kendall_to_local"] == 0).mean())})
        if method != "local_sofi":
            r["p_better_than_random"], r["rank_biserial_vs_random"] = _wilcoxon(g["degradation_sd"].values, None, "greater")
            tol = 1e-9 * (1 + np.abs(loc["degradation"].values))
            for col, better in (("morf_centered_sd", "lower"), ("lerf_centered_sd", "higher"), ("degradation_sd", "higher")):
                dlt = g[col].values - loc[col].values
                m_, med, lo, hi = _paired(dlt, rng)
                row = {"seed": seed, "population": pop, "fit_on": fit_on, "method": method, "metric": col.replace("_centered_sd", "").replace("_sd", ""),
                       "better_when": better, "global_mean": float(np.nanmean(g[col])), "local_mean": float(np.nanmean(loc[col])),
                       "difference_global_minus_local": m_, "difference_median": med, "boot_lo": lo, "boot_hi": hi,
                       "share_global_better": float(np.mean(dlt < -tol if better == "lower" else dlt > tol)),
                       "share_equal": float(np.mean(np.abs(dlt) <= tol)),
                       "share_local_better": float(np.mean(dlt > tol if better == "lower" else dlt < -tol))}
                if col != "degradation_sd":
                    row["wilcoxon_p"], row["rank_biserial"] = _wilcoxon(g[col].values, loc[col].values, "two-sided")
                else:
                    row["wilcoxon_p"], row["rank_biserial"] = np.nan, np.nan
                    row["note"] = "local order maximizes this score, the difference is an effect size"
                diffs.append(row)
        rows.append(r)
    df = pd.DataFrame(rows)
    glob = df["method"] != "local_sofi"
    df.loc[glob, "holm_better_than_random"] = holm(df.loc[glob, "p_better_than_random"].values)
    dd = pd.DataFrame(diffs)
    tested = dd["wilcoxon_p"].notna()
    dd.loc[tested, "holm_p"] = holm(dd.loc[tested, "wilcoxon_p"].values)
    wide = ev[ev["method"] != "local_sofi"].pivot(index="image", columns="method", values="degradation_sd").dropna()
    extra = {"seed": seed, "population": pop, "fit_on": fit_on, "images": len(wide),
             "share_no_method_positive": float((wide.max(1) <= 0).mean()) if len(wide) else np.nan,
             "share_best_method_below_half_local": float(
                 (wide.max(1).values < 0.5 * loc.loc[wide.index, "degradation_sd"].values).mean()) if len(wide) else np.nan,
             "distinct_global_rankings": int(ev[ev["method"] != "local_sofi"].groupby("method")["ranking"].first().nunique())}
    if wide.shape[1] >= 3 and len(wide) >= 3 and wide.std().sum() > 0:
        st = sps.friedmanchisquare(*[wide[c].values for c in wide.columns])
        extra.update(friedman_chi2=float(st.statistic), friedman_p=float(st.pvalue),
                     kendall_W=float(st.statistic / (len(wide) * (wide.shape[1] - 1))))
    return df, dd, extra

In [19]:
banner(f"THIRD EXPERIMENT | ONE GLOBAL RANKING PER POPULATION VERSUS THE LOCAL SOFI RANKINGS (SEED {SEL})")
say("Every image loses its concepts through zero removal in the order of a global ranking, then in the order of its own SOFI")
say("ranking. MoRF and LeRF areas are centered at their random-ordering expectation and expressed in null standard deviations")
say("of the image. Lower MoRF is better (sparsity), higher LeRF is better (faithfulness). Degradation = LeRF - MoRF is the SOFI")
say("score of Lemma 1, zero for a random ordering and maximal for the local SOFI order of the image.")
say("Global rankings fitted on the test images use the most favorable setting for one ranking. Those fitted on the validation")
say("images of the same class are applied to test images they have never seen.")
E3_IMG, E3_ROWS, E3_DIFF, E3_EXTRA_ROWS, RANK_ROWS, CURVES = [], [], [], [], [], []
for (pop, seed), PD in PDS.items():
    for fit_on in CFG.EXP3_FIT_SOURCES:
        dfit = PD["data"][fit_on]
        if dfit["N"] < CFG.MIN_POPULATION_LESIONS:
            continue
        rk = global_rankings(dfit)
        for mth, perm in rk.items():
            RANK_ROWS.append({"population": pop, "seed": seed, "selected": seed == SEL, "fit_on": fit_on, "method": mth,
                              "description": GLOBAL_METHODS[mth], "ranking": order_str(perm)})
        ev = evaluate_rankings(PD["data"]["test"], rk)
        ev.insert(0, "fit_on", fit_on); ev.insert(0, "seed", seed); ev.insert(0, "population", pop)
        E3_IMG.append(ev)
        st_, dd, extra = global_ranking_stats(ev, seed, pop, fit_on, np.random.default_rng(seed + 29))
        E3_ROWS.append(st_); E3_DIFF.append(dd); E3_EXTRA_ROWS.append(extra)
        CURVES.append(mean_curves(PD["data"]["test"], rk, {"seed": seed, "population": pop, "fit_on": fit_on}))
E3_STATS = pd.concat(E3_ROWS, ignore_index=True)
E3_DIFFS = pd.concat(E3_DIFF, ignore_index=True)
E3_EXTRA = pd.DataFrame(E3_EXTRA_ROWS)
E3_IMAGES = pd.concat(E3_IMG, ignore_index=True)
E3_CURVES = pd.concat(CURVES, ignore_index=True)
RANKS = pd.DataFrame(RANK_ROWS)

sel = lambda df: df[df["seed"] == SEL]
show(sel(RANKS), "Global ranking produced by every aggregation rule, selected model", "exp3_rankings")
for fit_on in CFG.EXP3_FIT_SOURCES:
    for pop in CFG.POPULATIONS:
        sub = E3_STATS[(E3_STATS["seed"] == SEL) & (E3_STATS["fit_on"] == fit_on) & (E3_STATS["population"] == pop)]
        if len(sub):
            show(sub, f"Metrics of every ranking, built on {fit_on}, applied to the {pop} test images", "exp3_metrics")
say("")
say("[exp3_paired] Paired differences global minus local per image, with bootstrap intervals over images. MoRF and LeRF")
say("receive two-sided Wilcoxon signed-rank tests with rank-biserial correlations, Holm-adjusted over rules and both metrics")
say("within one population and fitting source. The degradation difference is never positive by construction, so it is an")
say("effect size without a test. share_equal counts images for which the global ranking reproduces the local score.")
for fit_on in CFG.EXP3_FIT_SOURCES:
    for pop in CFG.POPULATIONS:
        sub = E3_DIFFS[(E3_DIFFS["seed"] == SEL) & (E3_DIFFS["fit_on"] == fit_on) & (E3_DIFFS["population"] == pop)]
        if len(sub):
            show(sub, f"Global minus local, rankings built on {fit_on}, {pop} test images", "exp3_paired")
show(sel(E3_EXTRA), "Images no rule explains better than chance, with the Friedman test across rules", "exp3_extra")
say("[exp3_curves] Mean centered MoRF and LeRF curves after k removed concepts, for plotting (csv only).")
show(E3_CURVES, "mean removal curves", "exp3_curves", echo=False)
show(E3_IMAGES, "image-level curves and scores", "exp3_image_level", echo=CFG.PRINT_IMAGE_LEVEL)

banner("THIRD EXPERIMENT REPEATED FOR EVERY TRAINED MODEL (PART OF THE MULTI-MODEL ANALYSIS)")
for seed in CFG.SEEDS:
    if seed == SEL:
        continue
    show(RANKS[RANKS["seed"] == seed], f"Global rankings, seed {seed}", "exp3_rankings")
    for fit_on in CFG.EXP3_FIT_SOURCES:
        for pop in CFG.POPULATIONS:
            sub = E3_STATS[(E3_STATS["seed"] == seed) & (E3_STATS["fit_on"] == fit_on) & (E3_STATS["population"] == pop)]
            if len(sub):
                show(sub, f"Metrics, seed {seed}, built on {fit_on}, {pop}", "exp3_metrics")
            sub = E3_DIFFS[(E3_DIFFS["seed"] == seed) & (E3_DIFFS["fit_on"] == fit_on) & (E3_DIFFS["population"] == pop)]
            if len(sub):
                show(sub, f"Global minus local, seed {seed}, built on {fit_on}, {pop}", "exp3_paired")
    show(E3_EXTRA[E3_EXTRA["seed"] == seed], f"Extra statistics, seed {seed}", "exp3_extra")
rows = []
for (pop, fit_on, mth, metric), g in E3_DIFFS.groupby(["population", "fit_on", "method", "metric"], sort=False):
    m_, lo, hi = t_interval(g["difference_global_minus_local"].values)
    rows.append({"population": pop, "fit_on": fit_on, "method": mth, "metric": metric, "seeds": len(g),
                 "mean_difference_over_seeds": m_, "t_lo": lo, "t_hi": hi,
                 "seeds_holm_significant": int((g["holm_p"] < 0.05).sum()) if "holm_p" in g else np.nan,
                 "mean_share_global_better": g["share_global_better"].mean(), "mean_share_equal": g["share_equal"].mean()})
show(pd.DataFrame(rows), "Paired differences averaged over models, Student intervals over seeds", "exp3_over_seeds")
rows = []
for (pop, fit_on, mth), g in E3_STATS[E3_STATS["method"] != "local_sofi"].groupby(["population", "fit_on", "method"], sort=False):
    for col in ("degradation_sd_mean", "mean_retained_share", "share_retained_below_half", "share_degradation_nonpositive",
                "mean_percentile_among_orderings"):
        m_, lo, hi = t_interval(g[col].values)
        rows.append({"population": pop, "fit_on": fit_on, "method": mth, "statistic": col, "seeds": len(g),
                     "mean_over_seeds": m_, "t_lo": lo, "t_hi": hi, "min_over_seeds": g[col].min(), "max_over_seeds": g[col].max()})
show(pd.DataFrame(rows), "Global-ranking metrics averaged over models, Student intervals over seeds", "exp3_metrics_over_seeds")


def global_report(pop):
    lab = CFG.POPULATION_LABEL[pop]
    out = []
    add = out.append
    st_ = E3_STATS[(E3_STATS["seed"] == SEL) & (E3_STATS["population"] == pop) & (E3_STATS["fit_on"] == "test")]
    dd = E3_DIFFS[(E3_DIFFS["seed"] == SEL) & (E3_DIFFS["population"] == pop) & (E3_DIFFS["fit_on"] == "test")]
    if not len(st_):
        return [f"No global ranking was evaluated for the {lab} population."]
    loc = st_[st_["method"] == "local_sofi"].iloc[0]
    gl = st_[st_["method"] != "local_sofi"].set_index("method")
    add(f"CAN ONE RANKING EXPLAIN EVERY {lab.upper()} IMAGE? SELECTED MODEL (SEED {SEL})")
    add("")
    add(f"Eight rules built one ranking each from the {int(loc.images)} {lab} test images themselves, the most favorable setting for one ranking.")
    add(f"The rules produced {int(E3_EXTRA[(E3_EXTRA['seed'] == SEL) & (E3_EXTRA['population'] == pop) & (E3_EXTRA['fit_on'] == 'test')]['distinct_global_rankings'].iloc[0])} distinct rankings.")
    add("Every image then lost its concepts in that order, whereas the reference removes them in the image's own SOFI order.")
    add("Scores are in null standard deviations of each image, where zero is the level of a random ordering.")
    add(f"The local SOFI order reaches a mean degradation score of {num(loc.degradation_sd_mean)}.")
    add(f"The global rankings reach from {num(gl['degradation_sd_mean'].min())} for the {SHORT[gl['degradation_sd_mean'].idxmin()]} "
        f"to {num(gl['degradation_sd_mean'].max())} for the {SHORT[gl['degradation_sd_mean'].idxmax()]}.")
    bf = gl.loc["best_fixed_ranking"]
    add(f"The best fixed ranking maximizes the mean score over these very images, yet it keeps {pc(bf.mean_retained_share)} of the local score on average.")
    add(f"It keeps less than half of the local score for {pc(bf.share_retained_below_half)} of images.")
    add(f"It reproduces the local score exactly for {pc(dd[(dd['method'] == 'best_fixed_ranking') & (dd['metric'] == 'degradation')]['share_equal'].iloc[0])} of images.")
    for metric, word in (("morf", "MoRF area"), ("lerf", "LeRF area")):
        d_ = dd[dd["metric"] == metric]
        sig = d_[d_["holm_p"] < 0.05]
        add(f"The centered {word} of the local order is {num(d_['local_mean'].iloc[0])}, against values from "
            f"{num(d_['global_mean'].min())} to {num(d_['global_mean'].max())} for the global rankings.")
        add(f"The paired difference is significant after Holm correction for {len(sig)} of {len(d_)} rules, "
            f"with rank-biserial correlations from {num(d_['rank_biserial'].min())} to {num(d_['rank_biserial'].max())}.")
        add(f"A global ranking does better than the local order on this curve for {pc(d_['share_global_better'].min())} "
            f"to {pc(d_['share_global_better'].max())} of images, depending on the rule.")
    add(f"A global ranking still beats a random ordering for most images, since the share of images with a zero or negative "
        f"score is {rng_(list(gl['share_degradation_nonpositive']), pc)} across rules.")
    ex = E3_EXTRA[(E3_EXTRA["seed"] == SEL) & (E3_EXTRA["population"] == pop) & (E3_EXTRA["fit_on"] == "test")].iloc[0]
    if np.isfinite(ex.get("friedman_p", np.nan)):
        add(f"The Friedman test across rules gives p = {ex['friedman_p']:.2g}, with a Kendall's W of {num(ex['kendall_W'])} on a scale where zero means no difference between rules.")
    va = E3_STATS[(E3_STATS["seed"] == SEL) & (E3_STATS["population"] == pop) & (E3_STATS["fit_on"] == "valid") & (E3_STATS["method"] != "local_sofi")]
    if len(va):
        add(f"Rankings built on the validation images keep {pc(va['mean_retained_share'].min())} to {pc(va['mean_retained_share'].max())} of the local score on the test images.")
    add("A single ranking is one vertex of the permutohedron, whereas the explanations of the images spread across the polytope.")
    add("The difference with the local orders shows that no single ranking explains every image of the population.")
    return out


for pop in CFG.POPULATIONS:
    lines = global_report(pop)
    banner(f"SUMMARY OF THE THIRD EXPERIMENT | {CFG.POPULATION_LABEL[pop].upper()}", "*")
    for line in lines:
        say(line)
    with open(os.path.join(CFG.OUTPUT_DIR, f"global_vs_local_{pop}.txt"), "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines) + "\n")


THIRD EXPERIMENT | ONE GLOBAL RANKING PER POPULATION VERSUS THE LOCAL SOFI RANKINGS (SEED 64)
Every image loses its concepts through zero removal in the order of a global ranking, then in the order of its own SOFI
ranking. MoRF and LeRF areas are centered at their random-ordering expectation and expressed in null standard deviations
of the image. Lower MoRF is better (sparsity), higher LeRF is better (faithfulness). Degradation = LeRF - MoRF is the SOFI
score of Lemma 1, zero for a random ordering and maximal for the local SOFI order of the image.
Global rankings fitted on the test images use the most favorable setting for one ranking. Those fitted on the validation
images of the same class are applied to test images they have never seen.

--- Global ranking produced by every aggregation rule, selected model (32 rows)
  population  seed  selected fit_on               method                                                                    description                            rankin

## CSV files

One csv file per table type, each holding every population and seed. They are written to the `csv` subfolder of the output folder, next to `console_log.txt` and the text summaries (`report_*.txt`, `seed_variation_*.txt`, `global_vs_local_*.txt`). The scripts in `visualization/` build the figures and the table of the manuscript from this `csv` folder once it is copied to `results/`.

In [20]:
banner("CSV FILES")
for key, dfs in TABLES.items():
    path = os.path.join(CFG.CSV_DIR, f"{key}.csv")
    pd.concat(dfs, ignore_index=True).to_csv(path, index=False)
    say(f"{path}  ({sum(len(d) for d in dfs)} rows)")
say(f"Console log in {_LOG_PATH}.")
say(f"Text summaries in {CFG.OUTPUT_DIR}, namely report_*.txt, seed_variation_*.txt and global_vs_local_*.txt.")


CSV FILES
/content/drive/MyDrive/Notebooks/Derm7pt/APG_outputs/casestudy_hard_cbm_efficientnet_b4_asymmetric/csv/split_summary.csv  (3 rows)
/content/drive/MyDrive/Notebooks/Derm7pt/APG_outputs/casestudy_hard_cbm_efficientnet_b4_asymmetric/csv/concept_states.csv  (7 rows)
/content/drive/MyDrive/Notebooks/Derm7pt/APG_outputs/casestudy_hard_cbm_efficientnet_b4_asymmetric/csv/model_performance.csv  (15 rows)
/content/drive/MyDrive/Notebooks/Derm7pt/APG_outputs/casestudy_hard_cbm_efficientnet_b4_asymmetric/csv/tuning_beta0.csv  (55 rows)
/content/drive/MyDrive/Notebooks/Derm7pt/APG_outputs/casestudy_hard_cbm_efficientnet_b4_asymmetric/csv/tuning_tau.csv  (120 rows)
/content/drive/MyDrive/Notebooks/Derm7pt/APG_outputs/casestudy_hard_cbm_efficientnet_b4_asymmetric/csv/population_sizes.csv  (10 rows)
/content/drive/MyDrive/Notebooks/Derm7pt/APG_outputs/casestudy_hard_cbm_efficientnet_b4_asymmetric/csv/model_selection.csv  (5 rows)
/content/drive/MyDrive/Notebooks/Derm7pt/APG_outputs/casestud